# [1.5.4] Superposition의 Toy Model & Sparse Autoencoder (연습 문제)

> **ARENA [Streamlit Page](https://arena-chapter1-transformer-interp.streamlit.app/34_[1.5.4]_Toy_Models_of_Superposition_&_SAEs)**
>
> **Colab: [exercises](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part54_toy_models_of_superposition_and_saes/1.5.4_Toy_Models_of_Superposition_&_SAEs_exercises.ipynb?t=20260329) | [solutions](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part54_toy_models_of_superposition_and_saes/1.5.4_Toy_Models_of_Superposition_&_SAEs_solutions.ipynb?t=20260329)**

문제나 버그가 있다면 [Slack group](https://join.slack.com/t/arena-uk/shared_invite/zt-3afdmdhye-Mdb3Sv~ss_V_mEaXEbkABA)의 `#errata` 채널로 보내주시고, 본 장의 학습 자료에 관한 질문은 전용 채널에서 해주시기 바랍니다.

마크다운 헤더 셀 왼쪽에 있는 화살표 기호를 클릭하면 각 섹션을 접어서 헤더만 보이게 할 수 있습니다.

다른 모든 장으로 가는 링크: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/headers/header-13-1.png" width="350">

# 소개

Superposition은 transformer가 어떻게 작동하는지 이해하는 데 있어 매우 중요한 개념입니다. Neel Nanda의 용어 사전에서 정의한 내용은 다음과 같습니다:

> Superposition은 모델이 $n$ 차원의 activation 공간에 $n$개보다 더 많은 feature를 표현하는 상태를 말합니다. 즉, feature는 여전히 방향(direction)에 대응하지만, **해석 가능한 방향의 집합이 차원의 수보다 더 많습니다**.

왜 이런 일이 일어날 것이라고 예상해야 할까요? 일반적으로 세상에는 모델이 가진 자유도(dimension of freedom)보다 훨씬 더 많은 feature가 존재하며, 따라서 모델 내의 feature와 값 사이에 일대일 매핑을 가질 수 없습니다. 하지만 모델은 어떻게든 이러한 feature들을 표현해야 합니다. 따라서 모델은 (feature 간의 노이즈와 간섭이 추가되는 대가를 치르더라도) 더 적은 차원에 여러 feature를 밀어 넣는 기술을 고안해 냅니다.

이 연습 문제들은 실제 언어 모델에서의 SAE 실용적 적용보다는 superposition의 이론을 깊이 있게 탐구하도록 특별히 설계되었습니다. 이 이론이 현재 SAE 관련 연구의 많은 부분을 뒷받침하고 있지만, 개념 발견(concept discovery)과 circuit 발견을 위해 SAE를 사용하는 방법을 다루는 섹션 1.3.3 및 1.4.2의 더 실용적인 내용을 학습하는 데 있어 이 장을 깊게 공부하는 것이 절대적으로 필수적인 것은 아닙니다. 이 섹션에서 가장 유용한 부분만 빠르게 학습하고 싶다면, 섹션 1️⃣과 5️⃣를 강력히 추천하며, 시간이 있다면 섹션 2️⃣를 짧게 살펴보시기 바랍니다. 나머지는 건너뛰셔도 좋습니다.

## 읽기 자료

* [200 COP in MI: Exploring Polysemanticity and Superposition](https://www.alignmentforum.org/posts/o6ptPu7arZrqRCxyz/200-cop-in-mi-exploring-polysemanticity-and-superposition), <b>15분</b>
    * "Tips" 섹션을 포함하여 그 전까지의 포스트를 읽어주세요 (일부 내용은 여기의 다른 자료들을 읽은 후에 더 잘 이해될 수 있습니다).
* Neel Nanda의 [Dynalist notes on superposition](https://dynalist.io/d/n2ZWtnoYHrU1s4vnFSAQ519J#z=3br1psLRIjQCOv2T4RN3V6F2), <b>10분</b>
    * 내용이 길지 않으므로 빠르게 훑어보시고, 이번 실습을 진행하는 동안 참고 자료로 활용하시기 바랍니다.
* Anthropic의 [Toy Models of Superposition](https://transformer-circuits.pub/2022/toy_model/index.html), <b>20분</b>
    * "Summary: A Hierarchy of Feature Properties" 섹션을 포함하여 그 전까지 읽어주세요.
    * 처음 몇 개의 섹션("Key Results", "Definitions and Motivation", "Empirical Phenomena")이 특히 중요합니다.
    * 실습을 진행하면서 이 논문의 다른 부분들도 함께 살펴볼 예정입니다.

## 콘텐츠 및 학습 목표

### 1️⃣ Superposition의 Toy 모델: Nonprivileged Basis에서의 Superposition

이 섹션에서는 Anthropic의 superposition toy 모델의 기초를 다룹니다. 예제를 통해 superposition이 무엇인지, 왜 발생하는지, 그리고 왜 신경망을 해석하는 데 문제가 되는지 학습합니다.

> ##### 학습 목표
>
> - superposition의 개념과 이것이 모델이 더 많은 feature 세트를 표현하는 데 어떻게 도움이 되는지 이해합니다.
> - superposition과 polysemanticity의 차이점을 이해합니다.
> - sparsity가 superposition에 어떻게 기여하는지 학습합니다.
> - feature importance curve의 개념을 이해합니다.
> - feature correlation이 superposition의 성격과 정도를 어떻게 변화시키는지 학습합니다.

### 2️⃣ Superposition의 Toy 모델: Privileged Basis에서의 Superposition

privileged basis에서의 superposition은 nonprivileged basis와는 조금 다릅니다. 첫째로, 모델은 basis 방향과 더 일치시키려는 경향이 있습니다 (비록 모든 feature를 표현하기 위해 어느 정도의 misalignment가 필요한 경우가 많지만 말입니다). 둘째로, 많은 privileged basis가 privileged인 이유는 단순히 representation을 저장하는 것이 아니라 입력값에 대해 일종의 computation을 수행하기 때문입니다. 따라서 모델이 어떻게 이러한 computation을 수행할 수 있는지 이해하는 것이 중요합니다.

> ##### 학습 목표
>
> - neuron superposition과 bottleneck superposition (또는 computational superposition과 representational superposition)의 차이점을 이해합니다.
> - 예제 `f(x) = abs(x)`를 통해 모델이 superposition 상태에서 어떻게 computation을 수행할 수 있는지 학습합니다.

### 3️⃣ Feature Geometry

이 섹션에서는 superposition의 geometry를 조금 더 깊게 살펴봅니다. 이전 세 섹션의 연습 문제만큼 필수적이지는 않지만, 시간이 있거나 superposition에 대해 더 깊이 파고들고 싶다면 흥미롭고 수행할 가치가 있는 내용입니다. 여기서는 **dimensionality** (모델이 단일 feature에 얼마나 많은 capacity를 할당하고 있는지에 대한 척도)를 다루며, 이 지표를 점점 더 복잡한 geometric structure의 형성과 연결 짓습니다.

> ##### 학습 목표
>
> - 기본적으로 특정 feature에 차원의 어느 정도 비율이 할당되었는지를 측정하는 *dimensionality*에 대해 학습합니다.
> - superposition 뒤에 숨겨진 geometric intuition을 이해하고, 이것이 더 큰 모델에서의 일반적인 superposition 개념과 어떻게 연결되는지 이해합니다.

### 4️⃣ Superposition & Deep Double Descent

Deep Double Descent는 딥러닝에서 오랫동안 관찰된 현상으로, 모델의 크기, 데이터 크기 또는 학습 시간이 증가함에 따라 태스크에 대한 모델 성능이 처음에는 향상되었다가, 정체기에 접어든 후, 다시 향상되기 시작하는 현상입니다. 정체기는 고전 통계학의 예측(즉, 모델 overfitting)과 일치하지만, 이후 loss가 더 감소하는 것은 그렇지 않습니다. Deep double descent에 관한 Anthropic의 논문은 이 현상을 superposition의 개념과 연결하며, 기본적으로 두 가지 scaling 단계가 각각 memorizing solution과 generalizing solution을 나타낸다고 주장합니다. 첫 번째 단계는 *datapoints*를 superposition으로 표현하는 것이 특징이며, 두 번째 단계는 *features*를 superposition으로 표현하는 것이 특징입니다.

이 섹션에서는 이 논문의 핵심 결과들을 재현하는 과정을 안내합니다. 다른 자료들에 비해 가이드가 적은 편이므로, 오늘의 핵심 경로에 있는 연습 문제라기보다는 선택적인 확장 학습으로 취급하시기를 권장합니다.

> ##### 학습 목표
> 
> - deep double descent 현상을 이해하고 특징을 정의합니다.
> - double descent의 서로 다른 단계들을 superposition의 개념과 연결 짓습니다.
> - 가이드가 적은 환경에서 논문의 결과를 재현하는 연습을 합니다.

### 5️⃣ Toy 모델에서의 Sparse Autoencoders

마지막 섹션에서는 sparse autoencoder에 대해 배우고, 이것이 superposition 문제를 해결하는 데 어떻게 도움이 될 수 있는지 학습합니다. 이전 섹션들의 toy 모델 설정에서 sparse autoencoder를 학습시키며, neuron resampling 및 다양한 architecture (예: [DeepMind's paper](https://deepmind.google/research/publications/88147/)의 Gated architecture)와 같은 기법들을 구현합니다.

> ##### 학습 목표
>
> - sparse autoencoder에 대해 배우고, superposition으로 표현된 feature들을 어떻게 disentangle 하는 데 사용할 수 있는지 학습합니다.
> - 이전 섹션의 toy 모델들로 직접 SAE를 학습시키고, feature reconstruction 과정을 시각화합니다.
> - 중요한 SAE 학습 전략 (예: resampling)과 architecture 변형 (예: Gated, Jump ReLU)을 이해합니다.

### ☆ 보너스

평소와 마찬가지로 추천 보너스 자료 및 논문 재현 섹션으로 마무리합니다.

## 질문들

위의 내용을 읽은 후 스스로 답할 수 있어야 하는 질문 세트(일부 간략한 답변 포함)입니다. 읽는 동안 답을 찾지 못했다면 Neel의 Dynalist 노트에서 검색해 보시기 바랍니다.

**privileged basis**란 무엇입니까? 왜 neuron activation은 기본적으로 privileged될 것이라고 기대해야 합니까? 왜 residual stream은 privileged되지 않을 것이라고 기대해야 합니까?

<details>
<summary>답변</summary>

privileged basis란 (해당 basis 상에서 수행되는 계산 구조로 인해) **표준 basis 방향이 의미를 갖는** basis를 말합니다. 이것이 반드시 해당 basis가 interpretable하다는 것을 의미하지는 않습니다.

**Neurons**

neuron activation이 privileged한 이유는 **적용되는 elementwise nonlinear function** 때문입니다. ReLU는 표준 basis에서 쉽게 설명됩니다. 예를 들어 2D에서는 다음과 같습니다:

$$\begin{bmatrix} x \\ y \end{bmatrix} \to \begin{bmatrix} \max(x, 0) \\ \max(y, 0) \end{bmatrix}$$

하지만 basis를 $x' = (x+y)/\sqrt{2}$, $y' = (x-y)/\sqrt{2}$로 재정의한다면, 이 새로운 basis에서 ReLU를 설명하는 것은 매우 복잡해집니다. 더 중요한 점은, 이제 컴포넌트 $x'$과 $y'$ 사이에 간섭이 발생한다는 것입니다. 즉, ReLU가 더 이상 그것들에 독립적으로 작용하지 않습니다.

$$\begin{bmatrix} x' \\ y' \end{bmatrix} \to \frac{1}{\sqrt{2}} \begin{bmatrix} \max(x, 0) + \max(y, 0) \\ \max(x, 0) - \max(y, 0) \end{bmatrix} = \frac{1}{2} \begin{bmatrix} \max(x'+y', 0) + \max(x'-y', 0) \\ \max(x'+y', 0) - \max(x'-y', 0) \end{bmatrix}$$

**Residual stream**

residual stream은 그것으로부터 읽고 쓰는 모든 작업이 linear map을 사용하기 때문에 privileged하지 않습니다. 사고 실험으로, 모든 쓰는 행렬(즉, MLP layer의 $W_{out}$와 attention layer의 $W_O$)을 $W \to W R$로 바꾸고, 모든 읽는 행렬(즉, MLP layer의 $W_{in}$와 attention layer의 $W_Q$, $W_K$, $W_V$)을 $W \to W R^{-1}$로 바꾼다고 가정해 봅시다. 여기서 $R$은 임의의 rotation matrix입니다. 그렇다면 모델의 계산은 변하지 않을 것입니다. 행렬 $R$은 임의적이기 때문에 basis를 원하는 방식으로 변경할 수 있으며, 따라서 해당 basis는 privileged할 수 없습니다.

달리 말하자면, 만약 누군가 "residual stream의 47번째 요소가 특정 정보(예: 해당 시퀀스 위치에 있는 명사의 복수형 여부)를 인코딩하고 있다고 생각합니다"라고 주장한다면, 저는 그 주장이 틀렸다고 말할 수 있습니다. 왜냐하면 이 사고 실험은 transformer가 수행하는 계산을 근본적으로 바꾸지 않고도, 어떤 basis 방향이든 여러 다른 basis 방향의 linear combination으로 쉽게 회전 및 분산될 수 있음을 보여주기 때문입니다. 이는 neuron에는 적용되지 않는데, rotation이나 basis 변경이 neuron 상에서 수행되는 계산의 본질을 바꾸기 때문입니다.

**요약**

**어떤 것이 rotation-independent하지 않다면 그것은 privileged basis입니다.** 즉, 그 위에서 수행되는 계산의 본질이 **basis 방향에 특별한 의미를 부여함**을 뜻합니다.

흔한 오해: privileged basis가 interpretable basis와 동일하다는 것입니다. 이는 **사실이 아닙니다** (비록 개별 basis 방향이 어떤 interpretable한 의미를 갖기 위해 basis가 privileged해야 하는 것은 맞지만, 이는 필요조건일 뿐 충분조건은 아닙니다).

</details>

**superposition**과 **polysemanticity**의 차이점은 무엇입니까?

<details>
<summary>답변</summary>

polysemanticity는 하나의 neuron이 여러 feature에 대응할 때 발생합니다 (더 자세한 논의와 예시는 [here](https://distill.pub/2020/circuits/zoom-in/#:~:text=lot%20of%20effort.-,Polysemantic%20Neurons,-This%20essay%20may)을 참조하십시오). 만약 polysemanticity만 존재한다면, 이는 우리에게 큰 문제가 되지 않을 것입니다 (각 basis vector가 단일 feature에 대응하는 feature를 위한 basis가 존재할 수 있기 때문입니다).

superposition은 **dimension보다 feature가 더 많을 때** 발생합니다. 따라서 이는 polysemanticity를 함의합니다 (하나 이상의 feature를 나타내는 dimension이 반드시 존재해야 하므로). 하지만 그 역은 성립하지 않습니다.

</details>

feature의 **importance**와 **sparsity**란 무엇입니까? sparsity가 더 크다면 polysemantic neuron이 더 많아질 것으로 예상합니까, 적어질 것으로 예상합니까?

<details>
<summary>답변</summary>

**Importance** = 이 feature가 loss를 낮추는 데 얼마나 유용한가?

**Sparsity** = 입력 데이터에서 이 feature가 얼마나 빈번하게 나타나는가?

sparsity가 더 크다면, 더 많은 polysemantic neuron이 나타날 것으로 예상합니다. 이는 단일 neuron이 여러 개의 서로 다른 sparse feature를 표현할 여유가 있기 때문입니다 (보통 특정 시점에는 그중 하나만 표현하게 되므로 간섭이 발생하지 않습니다).
</details>

**feature**를 어떻게 정의하시겠습니까?

<details>
<summary>답변</summary>

이에 대한 단 하나의 정답은 없습니다. 많은 정의가 불행히도 순환 논리적입니다 (예: "feature란 neuron에 의해 표현될 수 있는 것입니다"). 몇 가지 가능한 정의는 Neel의 [Dynalist notes](https://dynalist.io/d/n2ZWtnoYHrU1s4vnFSAQ519J#q=feature)에 나오는 다음과 같은 정의입니다:

> feature는 모델의 입력, 또는 그 입력의 일부 부분집합(예: 언어 모델에 주어진 prompt의 token, 또는 이미지의 patch)의 속성입니다.

또는 Chris Olah의 [Distill circuits Thread](https://distill.pub/2020/circuits/zoom-in/)에 나오는 유사한 정의입니다:

> feature는 입력의 scalar function입니다. 이 에세이에서 신경망 feature는 방향(direction)이며, 종종 단순히 개별 neuron입니다. 우리는 신경망의 이러한 feature들이 일반적으로 엄격하게 연구될 수 있는 의미 있는 feature라고 주장합니다. **의미 있는 feature**란 곡선의 존재나 처진 귀와 같이, 입력의 명확하게 설명 가능한 속성에 실제로 반응하는 feature를 말합니다.
</details>

## 설정 (읽지 말고 실행만 하세요)

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter1_transformer_interp"
repo = "ARENA_3.0"
branch = "main"

# Install dependencies
try:
    import transformer_lens
except:
    %pip install einops datasets jaxtyping "sae-lens>=4.0.0,<5.0.0" tabulate eindex-callum transformer_lens==2.17.0
    %pip install --force-reinstall numpy pandas

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else "/root"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/{repo}-{branch}


if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

In [ ]:
import sys
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable, Literal

import einops
import numpy as np
import pandas as pd
import plotly.express as px
import torch as t
from IPython.display import HTML, display
from jaxtyping import Float
from torch import Tensor, nn
from torch.distributions.categorical import Categorical
from torch.nn import functional as F
from tqdm.auto import tqdm

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")

# Make sure exercises are in the path
chapter = "chapter1_transformer_interp"
section = "part54_toy_models_of_superposition_and_saes"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

import part54_toy_models_of_superposition_and_saes.tests as tests
import part54_toy_models_of_superposition_and_saes.utils as utils
from plotly_utils import imshow, line

MAIN = __name__ == "__main__"

# 1️⃣ TMS: 비특권 기저에서의 Superposition

> ##### 학습 목표
>
> - superposition의 개념과, 이것이 모델이 더 많은 feature 세트를 표현하는 데 어떻게 도움이 되는지 이해합니다.
> - superposition과 polysemanticity의 차이점을 이해합니다.
> - sparsity가 superposition에 어떻게 기여하는지 배웁니다.
> - feature importance curve의 개념을 이해합니다.
> - feature correlation이 superposition의 성격과 정도를 어떻게 변화시키는지 배웁니다.

## Toy Model 설정

이 섹션에서는 [Anthropic's paper](https://transformer-circuits.pub/2022/toy_model/index.html)에서 연구된 toy model을 살펴보고 실험을 진행하겠습니다.

[Demonstrating Superposition](https://transformer-circuits.pub/2022/toy_model/index.html#demonstrating) 섹션부터 논문을 함께 읽으실 수 있으며, 이 노트북의 섹션 순서와 대략적으로 일치합니다.

이 논문은 **bottleneck superposition**에 대한 매우 기본적인 모델을 제시했습니다. 이는 차원이 $n$인 벡터 공간에 $n$ 개 이상의 feature를 표현하려고 할 때 발생합니다. 모델의 구조는 다음과 같습니다:

* 5차원 입력 $x$를 가져옵니다.
* 이를 2D 공간으로 매핑합니다.
* 다시 5D 공간으로 매핑합니다 (첫 번째 행렬의 transpose를 사용합니다).
* bias를 더하고 ReLU를 적용합니다.

$$
\begin{aligned}
h &= W x \\
x' &= \operatorname{ReLU}(W^T h + b)
\end{aligned}
$$

### 이러한 설정의 동기는 무엇인가요?

입력 $x$은 다섯 개의 feature를 나타냅니다 (이들은 0과 1 사이에서 균등하게 샘플링됩니다).

각 feature는 **importance**와 **sparsity**를 가질 수 있습니다. 이전의 정의를 상기해 보겠습니다:

* **Importance** = 손실(loss)을 낮추는 데 이 feature가 얼마나 유용한가?
* **Sparsity** = 입력 데이터에서 이 feature가 얼마나 빈번하게 나타나는가?

이는 toy model에서 다음과 같이 구현됩니다:

* **Importance** = 모델 학습에 사용하는 입력과 출력 사이의 가중 평균 제곱 오차(weighted mean squared error)의 계수입니다.
    * 다시 말해, 손실 함수는 $L = \sum_x \sum_i I_i (x_i - x_i^\prime)^2$이며, 여기서 $I_i$은 feature $i$의 importance입니다.
* **Sparsity** = $x$의 해당 요소가 0이 될 확률입니다.
    * 다시 말해, 이는 학습 데이터가 생성되는 방식에 영향을 줍니다 (아래 `Module` 클래스의 `generate_batch` 메서드를 참조하십시오).
    * 우리는 흔히 sparsity보다는 **feature probability** (1에서 sparsity를 뺀 값)라고 부릅니다.

$W^T W$을 사용하는 근거는 다음과 같습니다. 우리는 $W$ (shape이 `(2, 5)`인 행렬)를 feature와 bottleneck 차원 사이의 "overlap values" 그리드로 생각할 수 있습니다. 5x5 행렬 $W^T W$의 값들은 각 feature 쌍의 2D 표현 사이의 dot product입니다. 이 직관을 더 명확히 하기 위해, $W$의 각 열이 unit vector라고 가정하면, $W^T W$은 feature들 사이의 cosine similarity 행렬이 됩니다 (feature 자신과의 유사도는 1이므로 대각 요소는 1이 됩니다). 이를 직접 확인해 보겠습니다:

In [ ]:
t.manual_seed(2)

W = t.randn(2, 5)
W_normed = W / W.norm(dim=0, keepdim=True)

imshow(
    W_normed.T @ W_normed,
    title="Cosine similarities of each pair of 2D feature embeddings",
    width=600,
)

다른 방식으로 말하자면, $W$의 열들이 orthogonal하다면 $W^T W$은 identity가 될 것입니다. $W$가 2x5 행렬이기 때문에 실제로 그럴 수는 없지만, 열들 간의 pairwise cosine similarity가 0에 가깝다는 의미에서 "거의 orthogonal"할 수는 있습니다.

<details>
<summary>질문 - <code>W</code>의 열 개수가 행 개수보다 많을 때 (또는 달리 말해, hidden dimension이 input dimension보다 엄격히 작을 때), <code>W.T @ W</code>가 identity가 될 수 없음을 증명할 수 있습니까?</summary>

증명 #1: 행렬 곱 $AB$의 rank는 두 인자 $A$과 $B$ 중 최댓값에 의해 upper-bounded 됩니다. $W^T W$의 경우, 두 행렬 모두 rank가 최대 2이므로, 곱의 rank는 최대 2입니다.

증명 #2: 임의의 벡터 $x$에 대해, $W^T W x = W^T (Wx)$는 rank가 2인 벡터 공간인 $W^T$의 열들의 span에 속합니다.

</details>

두 개의 bottleneck dimension을 사용하는 또 다른 장점은 출력을 시각화할 수 있다는 점입니다! 이를 위해 몇 가지 helper 함수를 준비했습니다.

In [ ]:
utils.plot_features_in_2d(
    W_normed.unsqueeze(0),  # shape [instances=1 d_hidden=2 features=5]
)

이 플롯을 위의 `imshow` 플롯과 비교하고, 여기서 어떤 일이 일어나고 있는지(그리고 두 플롯이 서로 어떻게 연관되는지) 확실히 이해하시기 바랍니다. 이후의 많은 연습 문제들이 모델의 feature와 bottleneck dimension에 대한 기하학적 해석이라는 아이디어를 바탕으로 진행됩니다.

<details>
<summary>도움말 - 이 플롯들이 어떻게 작동하는지 헷갈립니다.</summary>

앞서 언급했듯이, $W$를 다섯 개의 feature 각각에 대응하는 다섯 개의 2D 벡터 집합으로 볼 수 있습니다. heatmap은 이 벡터들의 각 쌍 사이의 cosine similarity를 보여주며, 두 번째 플롯은 이 다섯 개의 벡터를 2D 공간에 나타냅니다.

위의 예시에서, 두 쌍의 벡터(1번째와 2번째, 그리고 0번째와 4번째)가 매우 높은 cosine similarity를 가지고 있음을 알 수 있습니다. 이는 2D 플롯에 반영되어 있으며, 여기서 해당 feature들은 서로 매우 가깝게 위치합니다 (0번째 feature가 가장 어두운 색이고, 4번째 feature가 가장 밝은 색입니다).

</details>

### 모델 정의하기

아래는 모델을 위한 코드입니다 (대부분의 메서드는 아직 채워지지 않은 상태입니다). 이 코스는 앞서 간단한 neural network를 구축해 보셨다면 익숙하실 것입니다.

이미 작성되어 있는 초기화 메서드에 대한 몇 가지 참고 사항입니다:

#### Weights & instances

`Config` 클래스는 `n_inst` 클래스를 가지고 있습니다. 이는 단일 training loop에서 여러 모델을 동시에 최적화하기 위함입니다 (나중에 유용하게 사용될 것입니다). 이를 기본적으로 weight를 위한 batch dimension이라고 생각하시면 됩니다. 즉, 각 weight/bias는 실제로 0번째 차원을 따라 쌓인 `n_inst` 개의 별개 weight/bias가 되며, 이들 각각은 (동일한 optimizer를 사용하여) 서로 다른 데이터로 병렬적으로 독립적으로 학습됩니다.

우리는 Anthropic 논문의 $W$ 및 $b$에 해당하는 weight `W`와 `b_final`를 초기화합니다.

#### Sparsity & Importance

`feature_probability` 인자는 임의의 feature가 활성화될 확률을 나타냅니다. 여기에는 `feature_probability = 1 - sparsity` 관계가 성립합니다. 우리는 종종 매우 작은 feature 확률 $p = 1 - S \approx 0$, 즉 1에 가까운 sparsity를 다루게 됩니다. feature 확률은 training data를 생성하는 데 사용되며, importance는 loss function에서 사용됩니다 (두 사항 모두 나중에 설명합니다). 기본값은 `feature_probability = 0.01`이며, 이는 각 feature가 1%의 확률로 존재함을 의미합니다.

`importance` 인자는 loss를 계산할 때 사용됩니다 (이후 연습 문제 참조). 기본값은 `importance = None`이며, 이는 균일한 importance를 결과로 냅니다.

`__init__` 메서드에는 `feature_probability`와 `importance`를 broadcast 하는 코드가 있어, 최종적으로 두 값 모두 항상 `(n_inst, n_features)` shape를 갖게 됩니다.

### 연습 문제 - `forward` 구현하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-20 minutes on this exercise.
> ```

지금은 `forward` 메서드만 채우시면 됩니다. 연습 문제가 진행됨에 따라 더 많은 함수를 채우게 되겠지만, 현재로서는 다른 함수들은 무시하셔도 좋습니다.

In [ ]:
def linear_lr(step, steps):
    return 1 - (step / steps)


def constant_lr(*_):
    return 1.0


def cosine_decay_lr(step, steps):
    return np.cos(0.5 * np.pi * step / (steps - 1))


@dataclass
class ToyModelConfig:
    # We optimize n_inst models in a single training loop to let us sweep over sparsity or importance
    # curves efficiently. You should treat the number of instances `n_inst` like a batch dimension,
    # but one which is built into our training setup. Ignore the latter 3 arguments for now, they'll
    # return in later exercises.
    n_inst: int
    n_features: int = 5
    d_hidden: int = 2
    n_correlated_pairs: int = 0
    n_anticorrelated_pairs: int = 0
    feat_mag_distn: Literal["unif", "normal"] = "unif"


class ToyModel(nn.Module):
    W: Float[Tensor, "inst d_hidden feats"]
    b_final: Float[Tensor, "inst feats"]

    # Our linear map (for a single instance) is x -> ReLU(W.T @ W @ x + b_final)

    def __init__(
        self,
        cfg: ToyModelConfig,
        feature_probability: float | Tensor = 0.01,
        importance: float | Tensor = 1.0,
        device=device,
    ):
        super(ToyModel, self).__init__()
        self.cfg = cfg

        if isinstance(feature_probability, float):
            feature_probability = t.tensor(feature_probability)
        self.feature_probability = feature_probability.to(device).broadcast_to((cfg.n_inst, cfg.n_features))
        if isinstance(importance, float):
            importance = t.tensor(importance)
        self.importance = importance.to(device).broadcast_to((cfg.n_inst, cfg.n_features))

        self.W = nn.Parameter(nn.init.xavier_normal_(t.empty((cfg.n_inst, cfg.d_hidden, cfg.n_features))))
        self.b_final = nn.Parameter(t.zeros((cfg.n_inst, cfg.n_features)))
        self.to(device)

    def forward(
        self,
        features: Float[Tensor, "... inst feats"],
    ) -> Float[Tensor, "... inst feats"]:
        """
        Performs a single forward pass. For a single instance, this is given by:
            x -> ReLU(W.T @ W @ x + b_final)
        """
        raise NotImplementedError()

    def generate_batch(self, batch_size: int) -> Float[Tensor, "batch inst feats"]:
        """
        Generates a batch of data of shape (batch_size, n_instances, n_features).
        """
        # You'll fill this in later
        raise NotImplementedError()

    def calculate_loss(
        self,
        out: Float[Tensor, "batch inst feats"],
        batch: Float[Tensor, "batch inst feats"],
    ) -> Float[Tensor, ""]:
        """
        Calculates the loss for a given batch (as a scalar tensor), using this loss described in the
        Toy Models of Superposition paper:

            https://transformer-circuits.pub/2022/toy_model/index.html#demonstrating-setup-loss

        Note, `self.importance` is guaranteed to broadcast with the shape of `out` and `batch`.
        """
        # You'll fill this in later
        raise NotImplementedError()

    def optimize(
        self,
        batch_size: int = 1024,
        steps: int = 5_000,
        log_freq: int = 50,
        lr: float = 1e-3,
        lr_scale: Callable[[int, int], float] = constant_lr,
    ):
        """
        Optimizes the model using the given hyperparameters.
        """
        optimizer = t.optim.Adam(self.parameters(), lr=lr)

        progress_bar = tqdm(range(steps))

        for step in progress_bar:
            # Update learning rate
            step_lr = lr * lr_scale(step, steps)
            for group in optimizer.param_groups:
                group["lr"] = step_lr

            # Optimize
            optimizer.zero_grad()
            batch = self.generate_batch(batch_size)
            out = self(batch)
            loss = self.calculate_loss(out, batch)
            loss.backward()
            optimizer.step()

            # Display progress bar
            if step % log_freq == 0 or (step + 1 == steps):
                progress_bar.set_postfix(loss=loss.item() / self.cfg.n_inst, lr=step_lr)


tests.test_model(ToyModel)

<details><summary>솔루션</summary>

```python
def forward(
    self,
    features: Float[Tensor, "... inst feats"],
) -> Float[Tensor, "... inst feats"]:
    """
    Performs a single forward pass. For a single instance, this is given by:
        x -> ReLU(W.T @ W @ x + b_final)
    """
    h = einops.einsum(
        features, self.W, "... inst feats, inst hidden feats -> ... inst hidden"
    )
    out = einops.einsum(
        h, self.W, "... inst hidden, inst hidden feats -> ... inst feats"
    )
    return F.relu(out + self.b_final)
```

</details>

### 연습 문제 - `generate_batch` 구현하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

다음으로, 위의 `generate_batch` 함수를 구현해야 합니다. 이 함수는 `(n_batch, instances, features)` 형태의 tensor를 반환해야 하며, 조건은 다음과 같습니다:

* `instances` 및 `features` 값은 model config에서 가져옵니다.
* 각 feature는 `self.feature_probability`의 확률로 존재합니다.
* 존재하는 각 feature의 **magnitude**는 0과 1 사이의 균등 분포(uniform distribution)에서 샘플링됩니다.

이 함수를 충분히 이해하시기 바랍니다 (테스트를 통과한 후에도 솔루션을 확인하시는 것을 권장합니다). 상관관계(correlations) 섹션에서 이 함수의 더 복잡한 버전들을 만들 예정이기 때문입니다.

`model.feature_probability`의 shape가 `(n_inst, n_features)`이라고 가정해도 좋다는 점을 기억하십시오.

함수 구현을 완료했다면, 아래 코드를 실행하여 테스트하십시오.

In [ ]:
# Go back up and edit your `ToyModel.generate_batch` method, then run the test below

tests.test_generate_batch(ToyModel)

<details><summary>솔루션</summary>

```python
def generate_batch(self: ToyModel, batch_size: int) -> Float[Tensor, "batch inst feats"]:
    """
    Generates a batch of data of shape (batch_size, n_instances, n_features).
    """
    batch_shape = (batch_size, self.cfg.n_inst, self.cfg.n_features)
    feat_mag = t.rand(batch_shape, device=self.W.device)
    feat_seeds = t.rand(batch_shape, device=self.W.device)
    return t.where(feat_seeds <= self.feature_probability, feat_mag, 0.0)


ToyModel.generate_batch = generate_batch
```
</details>

## 모델 학습하기

학습의 세부 사항은 개념적으로 그리 중요하지 않으므로, 대부분의 코드를 제공해 드렸습니다 (`optimize` 메서드 내). 모델이 학습됨에 따라 learning rate를 조절하기 위해 **learning rate schedulers**를 사용합니다. 이는 나중에 RL 챕터에서 사용하시게 됩니다.

### 연습 문제 - `calculate_loss` 구현하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 5-10 minutes on this exercise.
> ```

아래의 `calculate_loss` 함수를 완성해야 합니다. **단일 인스턴스**에 대한 loss 함수는 다음과 같습니다:

$$
L=\frac{1}{BF}\sum_x \sum_i I_i\left(x_i-x_i^{\prime}\right)^2
$$

여기서:

* $B$는 batch size입니다,
* $F$는 feature의 수입니다,
* $x_i$은 입력값이고 $x_i'$은 모델의 출력값입니다,
* $I_i$은 feature $i$의 중요도입니다,
* $\sum_i$은 feature에 대한 합계입니다,
* $\sum_x$은 batch 내 요소들에 대한 합계입니다.

일반적인 경우, 모든 인스턴스에 대해 이 공식을 합산합니다.

<details>
<summary>질문 - 왜 feature와 batch 차원에 대해서는 평균을 내고, 인스턴스 차원에 대해서는 합계를 구한다고 생각하십니까?</summary>

batch size에 대해 평균을 내는 이유는 이것이 loss 함수의 표준 방식이기 때문입니다 (또한 이는 batch size가 달라져도 서로 다른 learning rate를 사용할 필요가 없음을 의미합니다).

feature 차원에 대해 평균을 내는 이유는 그것이 [normal for MSE loss](https://pytorch.org/docs/stable/generated/torch.nn.MSELoss.html)이기 때문입니다.

인스턴스 차원에 대해 합계를 구하는 이유는 각 인스턴스를 독립적으로, 그리고 단일 인스턴스를 훈련시킬 때와 동일한 속도로 훈련시키고 싶기 때문입니다.

</details>

In [ ]:
# Go back up and edit your `ToyModel.calculate_loss` method, then run the test below

tests.test_calculate_loss(ToyModel)

<details><summary>솔루션</summary>

```python
def calculate_loss(
    self: ToyModel,
    out: Float[Tensor, "batch inst feats"],
    batch: Float[Tensor, "batch inst feats"],
) -> Float[Tensor, ""]:
    """
    Calculates the loss for a given batch, using this loss described in the Toy Models paper:

        https://transformer-circuits.pub/2022/toy_model/index.html#demonstrating-setup-loss

    Remember, `self.importance` will always have shape (n_inst, n_features).
    """
    error = self.importance * ((batch - out) ** 2)
    loss = einops.reduce(error, "batch inst feats -> inst", "mean").sum()
    return loss


ToyModel.calculate_loss = calculate_loss
```
</details>

이제 서론에 나왔던 그림의 버전을 재현해 보겠습니다. 몇 가지 참고 사항입니다:

* `importance` 인자는 모든 인스턴스에 대해 동일합니다. 각 feature에 대해 1에서 ~0.66 사이의 값을 가집니다 (따라서 모든 인스턴스에서 어떤 feature는 다른 feature보다 더 중요하게 됩니다).
* `feature_probability`은 모든 feature에 대해 동일하지만, 인스턴스마다 다릅니다. 다시 말해, 우리는 여러 가지 서로 다른 실험을 동시에 진행하고 있으며, 이러한 실험들에서 더 큰 feature sparsity를 갖는 것의 효과를 비교할 수 있습니다.

In [ ]:
cfg = ToyModelConfig(n_inst=8, n_features=5, d_hidden=2)

# importance varies within features for each instance
importance = 0.9 ** t.arange(cfg.n_features)

# sparsity is the same for all features in a given instance, but varies over instances
feature_probability = 50 ** -t.linspace(0, 1, cfg.n_inst)

line(
    importance,
    width=600,
    height=400,
    title="Importance of each feature (same over all instances)",
    labels={"y": "Feature importance", "x": "Feature"},
)
line(
    feature_probability,
    width=600,
    height=400,
    title="Feature probability (varied over instances)",
    labels={"y": "Probability", "x": "Instance"},
)

model = ToyModel(
    cfg=cfg,
    device=device,
    importance=importance[None, :],
    feature_probability=feature_probability[:, None],
)
model.optimize()

In [ ]:
utils.plot_features_in_2d(
    model.W,
    colors=model.importance,
    title=f"Superposition: {cfg.n_features} features represented in 2D space",
    subplot_titles=[f"1 - S = {i:.3f}" for i in feature_probability.squeeze()],
)

### 연습 문제 - 이 다이어그램들을 해석해 보세요

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 10-20 minutes on this exercise.
> ```

모든 다이어그램에서 어두운 색상은 중요도가 낮고, 밝은 색상은 중요도가 높다는 점을 기억하십시오. 또한, 왼쪽에서 오른쪽으로 이동함에 따라 모든 feature의 sparsity가 증가합니다 (가장 왼쪽에는 sparsity가 없으며, 가장 오른쪽에서는 모든 feature의 확률이 5%, 즉 sparsity가 95%입니다).

<details>
<summary>힌트</summary>

낮은 sparsity의 경우, 5개의 feature가 항상 모두 존재한다면 모델이 무엇을 학습하게 될지 생각해 보십시오. 이 경우 모델이 할 수 있는 최선은 무엇이며, 그것이 **importance** 값과 어떤 관련이 있습니까?

높은 sparsity의 경우, 항상 정확히 하나의 feature만 존재한다면 모델이 무엇을 학습하게 될지 생각해 보십시오. 이것이 feature 간의 interference 문제를 덜하게 만듭니까?
</details>

<details>
<summary>정답 (직관적 설명)</summary>

sparsity가 없을 때, 모델은 2개 이상의 feature를 충실하게 표현할 수 없으므로, 가장 중요한 두 개의 feature만 표현하는 것이 합리적입니다. 모델은 이를 2D 공간에 orthogonal하게 저장하고, 나머지 3개의 feature는 0으로 설정합니다. 이렇게 하면 이 두 feature는 완벽하게 재구성할 수 있으며, 나머지는 모두 무시합니다.

sparsity가 높을 때는 오각형 구조가 나타납니다. 대부분의 경우 이 다섯 가지 feature 중 최대 하나만 활성화되며, 이는 feature 간의 **interference**를 피하는 데 도움이 됩니다. 2D 공간의 점을 이 다섯 가지 방향으로 projection 하여 초기 feature를 복구하려고 할 때, feature $i$이 존재한다면 $i$번째 feature 방향으로의 projection이 다른 feature의 존재에 영향을 받지 않고 오직 이 feature만을 캡처한다고 확신할 수 있는 경우가 많습니다. 수학적인 세부 사항은 여기에서 생략합니다.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/img/ch13-sparsity-diagram-tms.png" width="900">

여기서 핵심 아이디어는 모델 내에서 두 가지 힘이 경쟁하고 있다는 것입니다: **feature benefit** (더 많은 것을 표현하는 것이 좋다!)과 **interference** (비-orthogonal하게 표현하는 것은 나쁘다). sparsity가 높을수록 interference의 부정적인 영향을 더 많이 줄일 수 있으며, 따라서 트레이드오프는 "비-orthogonal하게 더 많은 feature를 표현하는 것" 쪽으로 기울게 됩니다.

</details>

또한 batch를 생성하고 그 embedding을 시각화할 수 있습니다. 가장 흥미로운 점은, sparsity가 높은 플롯(오른쪽)에서는 다섯 가지 feature 간의 interference가 매우 드물게 발생한다는 것을 확인할 수 있다는 점입니다. 왜냐하면 대부분의 경우 해당 feature 중 $\leq 1$개만 존재하며, 모델은 정보 손실 없이 해당 feature 차원을 따라 projection 함으로써 이를 복구할 수 있기 때문입니다.

In [ ]:
with t.inference_mode():
    batch = model.generate_batch(200)
    hidden = einops.einsum(
        batch,
        model.W,
        "batch instances features, instances hidden features -> instances hidden batch",
    )

utils.plot_features_in_2d(hidden, title="Hidden state representation of a random batch of data")

### 다양한 sparsity에 따른 feature 시각화

이제 pentagon plot을 통해 무엇이 일어나고 있는지에 대한 기하학적 직관을 얻기 시작했으므로, 규모를 키워보겠습니다! 이제는 시각화하기에 너무 큰 차원에서 작업하게 되지만, 우리가 얻은 직관이 그대로 적용되기를 기대합니다.

In [ ]:
cfg = ToyModelConfig(n_inst=10, n_features=100, d_hidden=20)

importance = 100 ** -t.linspace(0, 1, cfg.n_features)
feature_probability = 20 ** -t.linspace(0, 1, cfg.n_inst)

line(
    importance,
    width=600,
    height=400,
    title="Feature importance (same over all instances)",
    labels={"y": "Importance", "x": "Feature"},
)
line(
    feature_probability,
    width=600,
    height=400,
    title="Feature probability (varied over instances)",
    labels={"y": "Probability", "x": "Instance"},
)

model = ToyModel(
    cfg=cfg,
    device=device,
    importance=importance[None, :],
    feature_probability=feature_probability[:, None],
)
model.optimize(steps=10_000)

더 이상 feature들을 2D로 시각화할 수 없기 때문에, 다른 종류의 시각화를 사용하겠습니다:

* **하단 행의 그래프**는 모든 feature와 그에 해당하는 embedding norm $||W_i||$을 막대 그래프로 보여줍니다.
    * sparsity를 높일수록, 모델은 더 많은 feature를 표현할 수 있게 됩니다 (즉, embedding norm이 1에 가까운 feature가 더 많아집니다).
    * 또한 다른 feature들과 orthogonal한지 여부에 따라 막대 색상을 지정했습니다 (orthogonal하면 보라색, 아니면 노란색). 이를 통해 sparsity가 낮을 때는 대부분의 feature가 orthogonal하게 표현되지만 (위의 가장 왼쪽 그래프들처럼), sparsity를 높이면 모든 feature가 non-orthogonal하게 표현되는 상태로 전이됨을 알 수 있습니다 (위의 가장 오른쪽 오각형 그래프들처럼).
* **상단 행의 그래프**는 모든 feature 벡터 쌍 사이의 dot product를 보여줍니다 (이 섹션 시작 부분에서 그렸던 heatmap과 비슷합니다).
    * 이는 sparsity를 높임에 따라 feature 간의 interference가 증가하는 것을 시각화하는 또 다른 방법입니다.
    * 이 모든 오른쪽 그래프들은 **rank가 최대 `d_hidden=20`인 matrix**를 나타낸다는 점에 유의하십시오. 처음 몇 개는 identity의 submatrix와 거의 비슷하지만 (20개의 feature를 완벽하게 재구성하고 나머지를 삭제했기 때문), 이후 그래프들에서는 20개 이상의 값을 그리게 되면서 interference가 나타나기 시작합니다 (이 matrix들의 대각 성분에 0이 아닌 요소가 20개보다 많아집니다).

이 그래프에 대한 더 자세한 설명과 해석 방법은 [Basic Results](https://transformer-circuits.pub/2022/toy_model/index.html#demonstrating-basic-results) 섹션을 참조하십시오.

In [ ]:
utils.plot_features_in_Nd(
    model.W,
    height=800,
    width=1600,
    title="ReLU output model: n_features = 100, d_hidden = 20, I<sub>i</sub> = 0.9<sup>i</sup>",
    subplot_titles=[f"Feature prob = {i:.3f}" for i in feature_probability],
)

## 상관관계가 있는 superposition

> 참고: superposition 뒤에 숨겨진 핵심 아이디어만 빠르게 파악하고 이 연습 문제들을 넘어가고 싶으시다면, 아마 지금이 다음 섹션으로 건너뛰어도 좋은 시점일 것입니다! 여기서의 핵심 아이디어는 기본적으로 feature들 사이의 음의 상관관계가 더 많은 superposition으로 이어진다는 것입니다. 왜냐하면 모델이 간섭(두 feature가 동시에 활성화되는 경우)으로 인해 겪는 어려움이 줄어들기 때문입니다. 세부 사항에 관심이 있고 실제로 재현을 수행하고 싶으시다면 계속 읽어주시기 바랍니다!

우리의 실험에서 고려하지 않은 한 가지 주요 요소는 **상관관계(correlation)**입니다. 우리는 feature들이 **반상관(anticorrelated)** 관계일 때 superposition이 훨씬 더 흔하게 발생할 것이라고 추측할 수 있습니다(feature들이 sparse할 때 더 흔하게 발생하는 것과 유사한 이유입니다). 대부분의 실제 세계 feature들은 반상관 관계입니다 (예를 들어, "이것은 정렬된 Python 리스트이다"라는 feature와 "이것은 엣지 있는 십대 뱀파이어 로맨스 소설의 텍스트이다"라는 feature는 아마도 반상관 관계일 것입니다. 아주 이상한 팬픽을 읽고 계신 경우가 아니라면 말입니다).

이 섹션에서는 상관관계가 있는 feature들을 위한 새로운 데이터 생성 함수를 정의하고, 첫 번째 섹션과 동일한 실험을 수행합니다.

### 연습 문제 - `generate_correlated_batch` 구현하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> You should spend up to 20-40 minutes on this exercise.
> The exercise itself is a bit fiddly / delicate, so you should definitely look at the solutions if you get stuck.
> ```

이제 상관관계(correlated) 및 반상관관계(anticorrelated) 데이터를 생성하기 위해 만들어진 `Model` 클래스의 세 가지 메서드 `generate_correlated_features`, `generate_anticorrelated_features`, `generate_uncorrelated_features`를 채워 넣어야 합니다. 이 모든 메서드의 집계 결과를 반환하는 새로운 `generate_batch` 함수가 제공되었습니다.

참고로, 상관관계 및 반상관관계 케이스에서는 각 인스턴스의 모든 feature에 대해 feature 확률이 동일하다고 가정할 수 있습니다. 저희는 여러분을 위해 이 내용을 assert 하는 것으로 함수를 시작하며, 각 인스턴스에 대한 이 feature 확률을 포함하는 벡터 `p`를 생성합니다 (`model.feature_probability` 대신 이것을 사용해야 합니다). 이는 생성하는 무상관(uncorrelated) feature의 수가 `cfg.n_features`보다 적은 무상관 케이스에서도 마찬가지입니다 (그렇지 않다면 전체 `self.feature_probability` tensor를 사용해도 무방합니다).

또한 반상관관계 케이스에서는 확률 설정에 주의해야 합니다. 예를 들어, feature 1과 2의 쌍에 대해 다음과 같이 처리한다면:

```python
feat1_is_present = t.rand() < p
feat2_is_present = t.rand() < p & ~feat1_is_present
```

실제 `feat2` 확률은 의도한 `p`가 아니라 `p * (1 - p)`이 됩니다. 두 feature가 동시에 활성화되지 않도록 보장하면서, 동시에 두 feature 모두 확률 `p`를 갖도록 만들어야 합니다! 힌트에서 이를 구현하는 방법에 대한 가이드를 제공합니다 (다소 까다로운 작업이며 개념적으로 매우 중요한 부분은 아닙니다!).

더 자세한 내용은 상관관계 및 반상관관계 세트를 어떻게 설정했는지 설명하는 [experimental details in Anthropic's paper](https://transformer-circuits.pub/2022/toy_model/index.html#geometry-correlated-setup)에서 읽어보실 수 있습니다.

<details>
<summary>도움말 - 상관관계 feature 함수를 어떻게 구현해야 할지 헷갈립니다.</summary>

먼저 쌍이 존재하는지 여부를 나타내는 `(batch_size, n_inst, n_correlated_pairs)` shape의 boolean mask를 생성한 다음, `einops.repeat`를 사용하여 해당 mask를 feature 쌍 전체에 반복해 보십시오.

</details>

<details>
<summary>도움말 - 반상관관계 feature 함수를 어떻게 구현해야 할지 헷갈립니다.</summary>

다음은 제안하는 두 가지 방법입니다:

1. 확률 $2p$으로 `(batch_size, n_inst, n_anticorrelated_pairs)` shape의 boolean mask를 생성하여 *둘 중 하나*의 feature가 존재하는지 나타내고, true인 경우 쌍 중에서 존재하는 feature를 균등하게 무작위로 선택합니다. 이렇게 하면 두 feature 모두 확률 $2p \times 0.5 = p$을 갖게 되므로 유효합니다.
2. 각각 확률 $p$과 $p / (1 - p)$을 가진 `(batch_size, n_inst, n_anticorrelated_pairs)` shape의 boolean mask 2개 `M1, M2`를 생성합니다. `M1`가 true인 곳에 첫 번째 feature가 존재하도록 설정하고, `~M1 && M2`이 true인 곳에 두 번째 feature가 존재하도록 설정합니다. 이렇게 하면 첫 번째 feature는 확률 $p$를, 두 번째 feature는 확률 $\frac{(1 - p)p}{(1 - p)} = p$를 갖게 되므로 유효합니다.

솔루션에서는 (2)와 같은 방법을 사용하지만, 두 방법 모두 유효합니다.

</details>

In [ ]:
def generate_correlated_features(
    self: ToyModel, batch_size: int, n_correlated_pairs: int
) -> Float[Tensor, "batch inst 2*n_correlated_pairs"]:
    """
    Generates a batch of correlated features. For each pair `batch[i, j, [2k, 2k+1]]`, one of
    them is non-zero if and only if the other is non-zero.
    """
    assert t.all((self.feature_probability == self.feature_probability[:, [0]]))
    p = self.feature_probability[:, [0]]  # shape (n_inst, 1)

    # YOUR CODE HERE!
    raise NotImplementedError()


def generate_anticorrelated_features(
    self: ToyModel, batch_size: int, n_anticorrelated_pairs: int
) -> Float[Tensor, "batch inst 2*n_anticorrelated_pairs"]:
    """
    Generates a batch of anti-correlated features. For each pair `batch[i, j, [2k, 2k+1]]`, each
    of them can only be non-zero if the other one is zero.
    """
    assert t.all((self.feature_probability == self.feature_probability[:, [0]]))
    p = self.feature_probability[:, [0]]  # shape (n_inst, 1)

    assert p.max().item() <= 0.5, "For anticorrelated features, must have 2p < 1"

    # YOUR CODE HERE!
    raise NotImplementedError()


def generate_uncorrelated_features(self: ToyModel, batch_size: int, n_uncorrelated: int) -> Tensor:
    """
    Generates a batch of uncorrelated features.
    """
    if n_uncorrelated == self.cfg.n_features:
        p = self.feature_probability
    else:
        assert t.all((self.feature_probability == self.feature_probability[:, [0]]))
        p = self.feature_probability[:, [0]]  # shape (n_inst, 1)

    # YOUR CODE HERE!
    raise NotImplementedError()


def generate_batch(self: ToyModel, batch_size) -> Float[Tensor, "batch inst feats"]:
    """
    Generates a batch of data, with optional correlated & anticorrelated features.
    """
    n_corr_pairs = self.cfg.n_correlated_pairs
    n_anti_pairs = self.cfg.n_anticorrelated_pairs
    n_uncorr = self.cfg.n_features - 2 * n_corr_pairs - 2 * n_anti_pairs

    data = []
    if n_corr_pairs > 0:
        data.append(generate_correlated_features(self, batch_size, n_corr_pairs))
    if n_anti_pairs > 0:
        data.append(generate_anticorrelated_features(self, batch_size, n_anti_pairs))
    if n_uncorr > 0:
        data.append(generate_uncorrelated_features(self, batch_size, n_uncorr))
    batch = t.cat(data, dim=-1)
    return batch


ToyModel.generate_batch = generate_batch

<details><summary>솔루션</summary>

```python
def generate_correlated_features(
    self: ToyModel, batch_size: int, n_correlated_pairs: int
) -> Float[Tensor, "batch inst 2*n_correlated_pairs"]:
    """
    Generates a batch of correlated features. For each pair `batch[i, j, [2k, 2k+1]]`, one of
    them is non-zero if and only if the other is non-zero.
    """
    assert t.all((self.feature_probability == self.feature_probability[:, [0]]))
    p = self.feature_probability[:, [0]]  # shape (n_inst, 1)

    feat_mag = t.rand((batch_size, self.cfg.n_inst, 2 * n_correlated_pairs), device=self.W.device)
    feat_set_seeds = t.rand((batch_size, self.cfg.n_inst, n_correlated_pairs), device=self.W.device)
    feat_set_is_present = feat_set_seeds <= p
    feat_is_present = einops.repeat(
        feat_set_is_present,
        "batch instances features -> batch instances (features pair)",
        pair=2,
    )
    return t.where(feat_is_present, feat_mag, 0.0)


def generate_anticorrelated_features(
    self: ToyModel, batch_size: int, n_anticorrelated_pairs: int
) -> Float[Tensor, "batch inst 2*n_anticorrelated_pairs"]:
    """
    Generates a batch of anti-correlated features. For each pair `batch[i, j, [2k, 2k+1]]`, each
    of them can only be non-zero if the other one is zero.
    """
    assert t.all((self.feature_probability == self.feature_probability[:, [0]]))
    p = self.feature_probability[:, [0]]  # shape (n_inst, 1)

    assert p.max().item() <= 0.5, "For anticorrelated features, must have 2p < 1"

    feat_mag = t.rand((batch_size, self.cfg.n_inst, 2 * n_anticorrelated_pairs), device=self.W.device)
    seed = t.rand((batch_size, self.cfg.n_inst, n_anticorrelated_pairs), device=self.W.device)
    mask = einops.rearrange(t.stack([seed, 1 - seed], dim=-1), "... feat pair -> ... (feat pair)") <= p
    return feat_mag * mask


def generate_uncorrelated_features(self: ToyModel, batch_size: int, n_uncorrelated: int) -> Tensor:
    """
    Generates a batch of uncorrelated features.
    """
    if n_uncorrelated == self.cfg.n_features:
        p = self.feature_probability
    else:
        assert t.all((self.feature_probability == self.feature_probability[:, [0]]))
        p = self.feature_probability[:, [0]]  # shape (n_inst, 1)

    if n_uncorrelated == self.cfg.n_features:
        p = self.feature_probability
    else:
        assert t.all((self.feature_probability == self.feature_probability[:, [0]]))
        p = self.feature_probability[:, [0]]  # shape (n_inst, 1)

    feat_mag = t.rand((batch_size, self.cfg.n_inst, n_uncorrelated), device=self.W.device)
    feat_seeds = t.rand((batch_size, self.cfg.n_inst, n_uncorrelated), device=self.W.device)
    return t.where(feat_seeds <= p, feat_mag, 0.0)
```
</details>

아래 코드는 많은 수의 batch를 생성하고 이를 통계적으로 측정함으로써 작성하신 함수를 테스트합니다.

In [ ]:
cfg = ToyModelConfig(n_inst=30, n_features=4, d_hidden=2, n_correlated_pairs=1, n_anticorrelated_pairs=1)

feature_probability = 10 ** -t.linspace(0.5, 1, cfg.n_inst).to(device)

model = ToyModel(cfg=cfg, device=device, feature_probability=feature_probability[:, None])

# Generate a batch of 4 features: first 2 are correlated, second 2 are anticorrelated
batch = model.generate_batch(batch_size=100_000)
corr0, corr1, anticorr0, anticorr1 = batch.unbind(dim=-1)

assert ((corr0 != 0) == (corr1 != 0)).all(), "Correlated features should be active together"
assert ((corr0 != 0).float().mean(0) - feature_probability).abs().mean() < 0.002, (
    "Each correlated feature should be active with probability `feature_probability`"
)

assert not ((anticorr0 != 0) & (anticorr1 != 0)).any(), "Anticorrelated features should never be active together"
assert ((anticorr0 != 0).float().mean(0) - feature_probability).abs().mean() < 0.002, (
    "Each anticorrelated feature should be active with probability `feature_probability`"
)

우리는 또한 이러한 feature들을 바 차트 형태로 시각화할 수 있습니다. 상관관계가 있는 feature들은 항상 함께 나타나고, 반상관관계가 있는 feature들은 절대 함께 나타나지 않는 것을 확인할 수 있습니다.

In [ ]:
# Generate a batch of 4 features: first 2 are correlated, second 2 are anticorrelated
batch = model.generate_batch(batch_size=1)
correlated_feature_batch, anticorrelated_feature_batch = batch.split(2, dim=-1)

# Plot correlated features
utils.plot_correlated_features(
    correlated_feature_batch,
    title="Correlated feature pairs: should always co-occur",
)
utils.plot_correlated_features(
    anticorrelated_feature_batch,
    title="Anti-correlated feature pairs: should never co-occur",
)

이제 2쌍의 상관관계가 있는 feature가 있을 때(Anthropic 논문의 [first row of the correlation figure](https://transformer-circuits.pub/2022/toy_model/index.html#geometry-organization) 에 해당), 모델을 학습시키고 feature를 2D로 시각화해 보겠습니다.

In [ ]:
cfg = ToyModelConfig(n_inst=5, n_features=4, d_hidden=2, n_correlated_pairs=2)

# All same importance, very low feature probabilities (ranging from 5% down to 0.25%)
feature_probability = 400 ** -t.linspace(0.5, 1, cfg.n_inst)

model = ToyModel(
    cfg=cfg,
    device=device,
    feature_probability=feature_probability[:, None],
)
model.optimize(steps=10_000)

In [ ]:
utils.plot_features_in_2d(
    model.W,
    colors=["blue"] * 2 + ["limegreen"] * 2,
    title="Correlated feature sets are represented in local orthogonal bases",
    subplot_titles=[f"1 - S = {i:.3f}" for i in feature_probability],
)

### 연습 문제 - 더 많은 상관관계 feature 플롯 생성하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to ~10 minutes on this exercise.
> It should just involve changing the parameters in your code above.
> ```

이제 논문의 [correlation figure](https://transformer-circuits.pub/2022/toy_model/index.html#geometry-organization) 에서 두 번째와 세 번째 행을 재현해야 합니다. 논문과 완전히 동일한 결과가 나오지 않을 수도 있지만, 대략적으로는 일치해야 합니다 (예를 들어, 위의 코드에서는 antipodal pair가 보이지 않아야 하지만, anticorrelated set을 테스트할 때는 모든 pair가 antipodal은 아니더라도 최소한 일부는 보여야 합니다). 솔루션 colab을 통해 몇 가지 예시를 확인하실 수 있습니다.

<details>
<summary>질문 - anticorrelated feature 플롯의 경우, feature probability를 약 10% 정도로 높여야 하며, 그렇지 않으면 항상 antipodal pair가 형성되지는 않을 것입니다. 왜 그렇다고 생각하십니까?</summary>

antipodal pair가 anticorrelated feature를 처리하는 데 더 유리한 이유는, 모델이 이러한 antipodal pair 중 한 번에 하나만 활성화될 것임을 확신할 수 있어 서로 간섭하지 않기 때문입니다. 따라서 결과적으로 한 번에 최대 2개의 방향만 non-zero가 될 것임을 확신할 수 있으며, 이 2개의 방향이 동시에 나타난다면 서로 직교하는 subspace에 위치한 서로 다른 2개의 orthogonal pair에서 왔으므로 반드시 orthogonal 함이 보장됩니다. 따라서 loss를 0으로 만들 수 있습니다. 만약 antipodal pair가 없다면, 서로 다른 feature pair의 feature들 사이에 간섭이 발생할 수 있습니다 (그 방향들이 antipodal일 수 있기 때문입니다).

여기서 핵심은 antipodal pair가 더 나은 이유는 단지 간섭, 즉 두 feature pair가 모두 활성화되는 경우를 더 잘 처리하기 때문이라는 점입니다. 이는 $O(p^2)$ 의 확률로 발생합니다 (여기서 $p$ 은 feature probability입니다). 따라서 $p$ 의 값이 매우 작을 때는, antipodal 솔루션이 non-antipodal 솔루션보다 가지는 이점이 훨씬 작아지며, 결국 먼저 발견한 솔루션으로 결정될 수 있습니다.

</details>

In [ ]:
# YOUR CODE HERE - generate more correlated feature plots

<details>
<summary>솔루션 (예시 코드 및 확인해야 할 사항)</summary>

```python
# Anticorrelated feature pairs
cfg = ToyModelConfig(n_inst=5, n_features=4, d_hidden=2, n_anticorrelated_pairs=2)

# All same importance, not-super-low feature probabilities (all >10%)
feature_probability = 10 ** -t.linspace(0.5, 1, cfg.n_inst)

model = ToyModel(cfg=cfg, device=device, feature_probability=feature_probability[:, None])
model.optimize(steps=10_000)

utils.plot_features_in_2d(
    model.W,
    colors=["red"] * 2 + ["orange"] * 2,
    title="Anticorrelated feature sets are frequently represented as antipodal pairs",
    subplot_titles=[f"1 - S = {i:.3f}" for i in feature_probability],
)
```

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/media-1320/1320-E4.png" width="950">

```python
# 3 correlated feature pairs
cfg = ToyModelConfig(n_inst=5, n_features=6, d_hidden=2, n_correlated_pairs=3)

# All same importance, very low feature probabilities (ranging from 5% down to 0.25%)
feature_probability = 100 ** -t.linspace(0.5, 1, cfg.n_inst)

model = ToyModel(cfg=cfg, device=device, feature_probability=feature_probability[:, None])
model.optimize(steps=10_000)

utils.plot_features_in_2d(
    model.W,
    colors=["blue"] * 2 + ["limegreen"] * 2 + ["purple"] * 2,
    title="Correlated feature sets are side by side if they can't be orthogonal (and sometimes we get collapse)",
    subplot_titles=[f"1 - S = {i:.3f}" for i in feature_probability],
)
```

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/media-1320/1320-E5.png" width="950">

</details>

# 2️⃣ TMS: 특권 기저에서의 Superposition

> ##### 학습 목표
>
> - neuron superposition과 bottleneck superposition(또는 computational superposition과 representational superposition)의 차이점을 이해합니다.
> - 예시 `f(x) = abs(x)` 를 통해 모델이 superposition 상태에서 어떻게 계산을 수행할 수 있는지 학습합니다.

## 서론

지금까지 우리는 privileged basis가 없는 모델에서의 superposition을 살펴보았습니다. 우리는 hidden activation을 임의로 회전시킬 수 있으며, 모든 weight를 함께 회전시킨다면 정확히 동일한 모델 동작을 얻을 수 있습니다. 즉, weight가 $W$ 인 임의의 ReLU 출력 모델에 대해, 임의의 orthogonal matrix $O$ 를 선택하여 모델 $W' = OW$ 를 고려할 수 있습니다. $(OW)^T(OW) = W^T W$ 이므로, 그 결과는 동일한 모델이 됩니다!

privileged basis가 없는 모델은 우아하며, word embedding이나 transformer residual stream과 같이 privileged basis가 없는 특정 신경망 표현의 흥미로운 유사 사례가 될 수 있습니다. 하지만 우리는 (아마도 주로) transformer MLP layer나 conv net neuron과 같이 privileged basis를 강제하는 neuron이 존재하는 신경망 표현을 이해하고 싶어 합니다.

이 섹션의 목표는 우리에게 privileged basis를 제공하는 가장 단순한 toy model을 탐구하는 것입니다. 이를 수행하는 방법은 적어도 두 가지가 있습니다. activation function을 추가하거나 hidden layer에 $L_1$ regularization을 적용하는 것입니다. 우리가 이해하는 데 가장 관심이 있는 표현은 transformer MLP layer와 같이 neuron이 있는 hidden layer이므로, activation function을 추가하는 것에 집중하겠습니다.

이를 통해 다음과 같은 "ReLU hidden layer" 모델을 얻게 됩니다. 이는 privileged basis를 제공할 가능성이 높으면서도 우리가 사용할 수 있는 가장 단순한 모델입니다. 이전 설정에서 hidden layer에 ReLU를 적용하기만 하면 됩니다.

$$
\begin{aligned}
h & =\operatorname{ReLU}(W x) \\
x^{\prime} & =\operatorname{ReLU}\left(W^T h+b\right)
\end{aligned}
$$

### 연습 문제 - `NeuronModel` 구현하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to ~10 minutes on this exercise.
> ```

이 섹션에서는 privileged basis에서의 superposition 연구에 관한 Anthropic 논문의 [first set of results](https://transformer-circuits.pub/2022/toy_model/index.html#demonstrating-setup-loss:~:text=model%20and%20a-,ReLU%20hidden%20layer%20model,-%3A)를 재현합니다. 이를 위해 새로운 `NeuronModel` 클래스가 필요합니다. 이 클래스는 `Model` 클래스의 대부분의 메서드를 상속받을 수 있지만, 중간에 ReLU를 포함하도록 `forward` 메서드를 재정의해야 합니다.

In [ ]:
class NeuronModel(ToyModel):
    def forward(self, features: Float[Tensor, "... inst feats"]) -> Float[Tensor, "... inst feats"]:
        raise NotImplementedError()


tests.test_neuron_model(NeuronModel)

<details><summary>솔루션</summary>

```python
class NeuronModel(ToyModel):
    def forward(self, features: Float[Tensor, "... inst feats"]) -> Float[Tensor, "... inst feats"]:
        activations = F.relu(
            einops.einsum(features, self.W, "... inst feats, inst d_hidden feats -> ... inst d_hidden")
        )
        out = F.relu(
            einops.einsum(activations, self.W, "... inst d_hidden, inst d_hidden feats -> ... inst feats")
            + self.b_final
        )
        return out
```
</details>

이 테스트들을 통과했다면, 아래 셀들을 실행하여 이전과 동일한 방식으로 모델을 학습시킬 수 있습니다. 우리는 7개의 instance를 사용하며, 각 instance는 10개의 feature를 가집니다 (instance 전반에 걸쳐 확률은 $0.75$에서 $0.01$ 사이이며), 각 instance 내의 feature importance는 $0.75^{i}$에 따라 감소합니다.

또한 행렬 $W$을 시각화합니다. 이 플롯들에서, 우리는 첫 번째 행의 시각화가 $W^T W$가 아닌 $W$가 되도록 설정합니다. 이제는 (이전과 달리) $W$의 개별 요소들이 의미를 갖기 때문에 이렇게 처리할 수 있습니다. 우리는 **privileged basis**를 다루고 있으며, $W$은 feature를 basis-aligned neuron에 연결합니다.

In [ ]:
cfg = ToyModelConfig(n_inst=7, n_features=10, d_hidden=5)

importance = 0.75 ** t.arange(1, 1 + cfg.n_features)
feature_probability = t.tensor([0.75, 0.35, 0.15, 0.1, 0.06, 0.02, 0.01])

model = NeuronModel(
    cfg=cfg,
    device=device,
    importance=importance[None, :],
    feature_probability=feature_probability[:, None],
)
model.optimize(steps=10_000)

In [ ]:
utils.plot_features_in_Nd(
    model.W,
    height=600,
    width=1000,
    title=f"Neuron model: {cfg.n_features=}, {cfg.d_hidden=}, I<sub>i</sub> = 0.75<sup>i</sup>",
    subplot_titles=[f"1 - S = {i:.2f}" for i in feature_probability.squeeze()],
    neuron_plot=True,
)

### 연습 문제 - 이 플롯들을 해석해 보세요

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

첫 번째 행은 $W$의 플롯을 보여줍니다. 행은 feature이고, 열은 hidden dimension (neuron)입니다.

두 번째 행은 stacked weight 플롯을 보여줍니다. 즉, 각 열은 하나의 neuron이며, 열의 값들은 해당 neuron에 대한 feature들의 exposure입니다. 이 플롯에서 각 feature는 다른 feature와의 간섭 정도에 따라 다르게 색칠되어 있습니다 (진한 파란색은 해당 feature가 다른 모든 feature와 orthogonal함을 의미하며, 밝은 색은 다른 feature들과의 squared dot product의 합이 큼을 의미합니다).

이 플롯들에 대해 어떻게 해석하시겠습니까? monosemanticity / polysemanticity와 sparsity가 증가함에 따라 이것이 어떻게 변하는지에 대해 논의해야 합니다.

<details>
<summary>일부 플롯에 대한 설명</summary>

**낮은 sparsity / 높은 feature 확률**

sparsity가 매우 낮을 때 (feature prob $\approx 1$), superposition은 발생하지 않습니다. 모든 feature는 모델의 서로 다른 neuron에 의해 충실하게 표현되거나, 전혀 표현되지 않습니다. 즉, **순수한 monosemanticity** 상태입니다.

heatmap에서 우리는 (neuron의 재배열을 제외하면) 대각선 플롯을 볼 수 있습니다. 즉, 가장 중요한 5개의 feature 각각이 해당 feature만을 감지하고 다른 것은 감지하지 않는 대응하는 neuron을 가지고 있습니다.

bar chart에서도 이러한 monosemanticity가 나타납니다. 각 neuron에는 단 하나의 feature만 exposure되어 있습니다.

**중간 sparsity / 중간 feature 확률**

중간 값에서는 일부 monosemantic neuron과 일부 polysemantic neuron이 나타납니다. 다음과 같은 반복되는 블록 패턴을 볼 수 있을 것입니다 (행 및/또는 열의 재배열을 제외하고):

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/three_two2.png" width="130">

이것들이 어떤 기하학적 배치에 대응하는지 알 수 있습니까? 정답은 아래의 중첩 드롭다운에 있습니다.

<details>
<summary>정답</summary>

3x2 블록은 2D 공간에 embedding된 3개의 feature를 보여줍니다. 3개의 feature를 각각 $i, j, k$라고 할 때, $j$은 $(1, 1)$ 방향(다른 두 개와 orthogonal한 방향)으로 표현되고, $i, k$은 각각 $(-1, 1)$와 $(1, -1)$로 표현됨을 알 수 있습니다 (antipodal pair).

3x3 블록의 경우, 이는 사실 정사면체의 4개 점 중 3개입니다! 이는 다음 (선택 사항) 연습 문제 세트에서 살펴볼 중요한 사실을 암시합니다. **superposition은 feature들이 기하학적 구조로 스스로를 조직하게 만들며**, 이는 종종 uniform polyhedra를 나타냅니다.

</details>

bar chart를 보면 일부 neuron들이 하나 이상의 feature에 exposure되면서 polysemantic해지기 시작하는 것을 볼 수 있습니다.

**높은 sparsity / 낮은 feature 확률**

sparsity가 높으면 모든 neuron이 polysemantic하며, 대부분 또는 모든 feature가 어느 정도 표현됩니다. neuron들은 (neuron보다 feature가 훨씬 많기 때문에) orthogonal하지 않지만, orthogonal할 필요는 없습니다. 이전 섹션에서 높은 sparsity가 어떻게 dimension보다 더 많은 feature를 표현할 수 있게 하는지 살펴보았습니다. 이 경우에도 마찬가지입니다.

참고 - Anthropic은 [finds](https://transformer-circuits.pub/2022/toy_model/index.html#privileged-basis:~:text=The%20solutions%20are%20visualized%20below) sparsity가 매우 높으면 각 feature가 한 쌍의 neuron에 대응하게 된다고 언급했습니다. 하지만 여러분의 플롯에서는 이를 발견하지 못할 수도 있습니다 (저도 그러했습니다!). 이는 Anthropic이 언급했듯이, 그들의 toy model 설정에서 이러한 모델들이 다른 모델들보다 최적화하기 더 어려웠기 때문에 많은 별도의 인스턴스를 학습시키고 loss가 가장 작은 것들을 선택했기 때문입니다.

전반적으로, sparsity를 증가시킴에 따라 이전에 보았던 feature phase change와 마찬가지로 **neuron 수준에서 monosemantic에서 polysemantic으로의 phase change**가 일어나는 것처럼 보입니다.

</details>

다양한 설정(sparsity, importance)으로 시도해 보세요. 어떤 결과가 나오나요?

### 연습 문제 (선택 사항) - 그래프를 더 정확하게 재현하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵⚪⚪⚪⚪
> 
> You should spend up to 10-25 minutes on this exercise, if you choose to do it.
> ```

Anthropic은 논문에서 1000개의 인스턴스를 학습시킨 후 loss가 가장 낮은 것들을 선택했다고 언급했습니다. 이것이 여러분의 결과가 그들의 결과와 다를 수 있는 이유이며, 특히 sparsity가 매우 높거나 feature 확률이 매우 낮을 때 더욱 그렇습니다.

여러분의 클래스에 이 "최저 loss 선택" 방법을 구현할 수 있습니까? 몇 가지 제안입니다:

* 가장 기본적인 방법은 `optimize` 함수가 인스턴스당 loss를 반환하도록 수정하고, for 루프를 사용하여 `optimize` 호출을 여러 번 실행한 뒤 마지막에 각 sparsity 레벨에 대해 최적의 인스턴스를 선택하는 것입니다.
* 훨씬 더 좋은 방법은 한 번에 더 많은 인스턴스를 학습시키고 (예: sparsity 레벨당 `N` 개의 인스턴스), 각 sparsity 레벨에 대해 마지막에 `N`에 대해 argmax를 수행하여 단일 인스턴스를 얻는 것입니다. 이 방법이 훨씬 빠를 것입니다 (다만, 1000개의 인스턴스를 한 번에 학습시키지 않도록 주의하십시오. GPU가 지원하지 않을 수 있습니다!).
* 더 정교하게 구현하려면, argmax를 수행할 이 `N` 차원에 대응하는 차원을 weight 행렬에 추가할 수도 있습니다. 그렇게 하면 이 "최저 loss 인스턴스 선택" 동작이 자동으로 이루어집니다.

## superposition에서의 계산

위의 예시는 흥미로웠지만, 어떤 면에서는 한계가 있었습니다. 여기서 핵심적인 문제는 **모델이 ReLU hidden layer로부터 이득을 얻지 못한다**는 점입니다. ReLU를 추가하면 모델이 특권 기저(privileged basis)를 갖도록 유도하지만, 모델이 입력을 재구성(즉, 선형 함수인 identity)하려고 하기 때문에 실제로 ReLU를 사용할 필요가 없으며, 모든 뉴런을 선형적으로 동작하는 양수 영역으로 이동시키는 bias를 학습하는 등 ReLU를 우회하기 위해 가능한 모든 방법을 시도할 것입니다. 이는 superposition을 연구하기 위해 이 toy model을 사용하는 것에 있어 단점이 됩니다.

이 점을 더 확장해 보겠습니다. 우리는 identity와 같은 지루한 선형 함수를 연구하고 싶은 것이 아니라, **모델이 superposition에서 (비선형) 계산을 어떻게 수행하는지**를 연구하고 싶습니다. transformer의 MLP layer는 단순히 정보를 충실하게 표현하고 복구하는 수단이 아니라, 그 정보에 대해 계산을 수행하는 수단입니다. 따라서 다음 섹션에서는 비선형 계산을 수행하는 모델을 학습시켜 보겠습니다. 구체적으로, **입력값의 절대값을 계산하도록 $x$** 모델을 학습시킬 것입니다.

이제 데이터 $x$는 $[0, 1]$ 대신 $[-1, 1]$ 범위에서 샘플링됩니다 (그렇지 않으면 절대값을 계산하는 것이 입력을 재구성하는 것과 동일해지기 때문입니다). $abs(x)$는 $\operatorname{ReLU}(x) + \operatorname{ReLU}(-x)$와 동일하므로, 이는 비선형 함수가 가질 수 있는 가장 단순한 형태입니다. 하지만 비선형이기 때문에, 모델이 반드시 hidden layer의 ReLU를 사용해야 한다는 점을 확신할 수 있습니다.

### 연습 문제 - `NeuronComputationModel` 구현하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 20-30 minutes on this exercise.
> ```

아래의 `NeuronComputationModel` 클래스를 완성해야 합니다. 구체적으로, `forward`, `generate_batch` 및 `calculate_loss` 메서드를 채워 넣어야 합니다. 몇 가지 안내 사항은 다음과 같습니다:

* 모델의 **forward 함수**가 다릅니다 - forward 함수 내에 ReLU hidden layer가 포함되어 있습니다 (위의 설명 및 논문에 기술된 바와 같습니다).
* 모델의 **데이터**가 다릅니다 - 위의 논의 내용을 참조하십시오. `generate_batch` 함수를 다시 작성해야 합니다 - 이 함수는 여러분이 작성한 첫 번째 버전(즉, correlation이 없는 버전)과 동일하지만, 한 가지 차이점이 있습니다: 값은 $[0, 1]$ 대신 $[-1, 1]$ 범위에서 균등하게 샘플링됩니다.
* 모델의 **loss 함수**가 다릅니다. 입력 $x$과 출력 $x'$ 사이의 importance-weighted $L_2$ 오차를 계산하는 대신, $\operatorname{abs}(x)$과 $x'$ 사이의 importance-weighted $L_2$ 오차를 계산합니다. 이는 단 한 줄만 수정하면 됩니다. `optimize` 함수는 그대로 유지해도 되지만, 이제 이 새로운 loss 함수를 최적화하게 됩니다.

In [ ]:
class NeuronComputationModel(ToyModel):
    W1: Float[Tensor, "inst d_hidden feats"]
    W2: Float[Tensor, "inst feats d_hidden"]
    b_final: Float[Tensor, "inst feats"]

    def __init__(
        self,
        cfg: ToyModelConfig,
        feature_probability: float | Tensor = 1.0,
        importance: float | Tensor = 1.0,
        device=device,
    ):
        super(ToyModel, self).__init__()
        self.cfg = cfg

        if isinstance(feature_probability, float):
            feature_probability = t.tensor(feature_probability)
        self.feature_probability = feature_probability.to(device).broadcast_to((cfg.n_inst, cfg.n_features))
        if isinstance(importance, float):
            importance = t.tensor(importance)
        self.importance = importance.to(device).broadcast_to((cfg.n_inst, cfg.n_features))

        self.W1 = nn.Parameter(nn.init.kaiming_uniform_(t.empty((cfg.n_inst, cfg.d_hidden, cfg.n_features))))
        self.W2 = nn.Parameter(nn.init.kaiming_uniform_(t.empty((cfg.n_inst, cfg.n_features, cfg.d_hidden))))
        self.b_final = nn.Parameter(t.zeros((cfg.n_inst, cfg.n_features)))
        self.to(device)

    def forward(self, features: Float[Tensor, "... inst feats"]) -> Float[Tensor, "... inst feats"]:
        raise NotImplementedError()

    def generate_batch(self, batch_size) -> Float[Tensor, "batch instances features"]:
        raise NotImplementedError()

    def calculate_loss(
        self,
        out: Float[Tensor, "batch instances features"],
        batch: Float[Tensor, "batch instances features"],
    ) -> Float[Tensor, ""]:
        raise NotImplementedError()


tests.test_neuron_computation_model(NeuronComputationModel)

<details>
<summary><code>forward</code></summary>에 대한 솔루션입니다.

```python
def forward(self, features: Float[Tensor, "... inst feats"]) -> Float[Tensor, "... inst feats"]:
    activations = F.relu(
        einops.einsum(features, self.W1, "... inst feats, inst d_hidden feats -> ... inst d_hidden")
    )
    out = F.relu(
        einops.einsum(activations, self.W2, "... inst d_hidden, inst feats d_hidden -> ... inst feats")
        + self.b_final
    )
    return out
```

</details>

<details>
<summary><code>generate_batch</code></summary>에 대한 솔루션입니다.

```python
def generate_batch(self, batch_size) -> Float[Tensor, "batch instances features"]:
    feat_mag = 2 * t.rand((batch_size, self.cfg.n_inst, self.cfg.n_features), device=self.W1.device) - 1
    feat_seed = t.rand(
        (batch_size, self.cfg.n_inst, self.cfg.n_features),
        device=self.W1.device,
    )
    batch = t.where(feat_seed < self.feature_probability, feat_mag, 0.0)
    return batch
```

</details>

<details>
<summary><code>calculate_loss</code></summary>에 대한 솔루션입니다.

```python
def calculate_loss(
    self,
    out: Float[Tensor, "batch instances features"],
    batch: Float[Tensor, "batch instances features"],
) -> Float[Tensor, ""]:
    error = self.importance * ((batch.abs() - out) ** 2)
    loss = einops.reduce(error, "batch inst feats -> inst", "mean").sum()
    return loss
```

</details>

이 테스트들을 통과했다면, 아래 코드를 실행하여 위와 동일한 시각화를 만들 수 있습니다.

비슷한 패턴을 확인하실 수 있을 것입니다. sparsity가 매우 낮을 때는 대부분 또는 모든 neuron이 monosemantic하지만, sparsity가 증가함에 따라 더 많은 polysemantic neuron이 나타납니다 (모든 neuron이 polysemantic해질 때까지). 또 다른 흥미로운 관찰 결과는 다음과 같습니다. monosemantic(또는 대부분 monosemantic)인 경우, 임의의 feature에 대해 해당 feature에 양의 exposure를 가진 neuron과 음의 exposure를 가진 neuron이 모두 존재합니다. 이는 일부 neuron은 값 $\operatorname{ReLU}(x_i)$ 을 표현하고, 다른 neuron들은 값 $\operatorname{ReLU}(-x_i)$ 을 표현하기 때문입니다 (앞서 논의한 것처럼, 절대값을 계산하기 위해서는 이 두 가지가 모두 필요합니다).

In [ ]:
cfg = ToyModelConfig(n_inst=7, n_features=100, d_hidden=40)

importance = 0.8 ** t.arange(1, 1 + cfg.n_features)
feature_probability = t.tensor([1.0, 0.3, 0.1, 0.03, 0.01, 0.003, 0.001])

model = NeuronComputationModel(
    cfg=cfg,
    device=device,
    importance=importance[None, :],
    feature_probability=feature_probability[:, None],
)
model.optimize()

In [ ]:
utils.plot_features_in_Nd(
    model.W1,
    height=800,
    width=1600,
    title=f"Neuron computation model: n_features = {cfg.n_features}, d_hidden = {cfg.d_hidden}, I<sub>i</sub> = 0.75<sup>i</sup>",
    subplot_titles=[f"1 - S = {i:.3f}" for i in feature_probability.squeeze()],
    neuron_plot=True,
)

이 현상이 실제로 일어나고 있는지 더 확실히 확인하기 위해, 바 차트의 값을 feature의 polysemanticity에 따라 연속적으로 색칠하는 대신, feature별로 이산적으로 색칠할 수 있습니다. 이 시각화를 위해 feature probability를 50%로 설정하겠습니다. 이는 각 neuron이 monosemantic임을 보장하기에 충분히 높은 수치입니다. input weights $W_1$ 가 서로 반대 방향의 neuron 쌍(즉, 해당 feature 방향에 대해 양수/음수 노출을 가진 neuron들)을 형성하지만, 이 두 neuron 모두 해당 feature에 대해 양수 output weights $W_2$ 를 가지고 있음을 확인할 수 있을 것입니다.

In [ ]:
cfg = ToyModelConfig(n_inst=6, n_features=20, d_hidden=10)

importance = 0.8 ** t.arange(1, 1 + cfg.n_features)
feature_probability = 0.5

model = NeuronComputationModel(
    cfg=cfg,
    device=device,
    importance=importance[None, :],
    feature_probability=feature_probability,
)
model.optimize()

In [ ]:
utils.plot_features_in_Nd_discrete(
    W1=model.W1,
    W2=model.W2,
    title="Neuron computation model (colored discretely, by feature)",
    legend_names=[f"I<sub>{i}</sub> = {importance.squeeze()[i]:.3f}" for i in range(cfg.n_features)],
)

## 보너스 - 비대칭 중첩(asymmetric superposition) 모티프

Anthropic 논문의 [Asymmetric Superposition Motif](https://transformer-circuits.pub/2022/toy_model/index.html#computation-asymmetric-motif) 섹션에서, 그들은 이 toy model의 특이한 점에 대해 자세히 논의합니다. 해당 섹션에서는 여기에서 다루는 것보다 더 상세하게(시각적 설명 포함) 설명하고 있지만, 여기서는 상대적으로 간략한 설명을 제공하겠습니다.

> 모델의 sparsity를 높여 superposed feature들이 나타나기 시작할 때, 항상 어떤 feature $i$에 대해 $\operatorname{ReLU}(x_i)$ 또는 $\operatorname{ReLU}(-x_i)$를 각각 계산하는 monosemantic neuron을 갖게 되는 것은 아닙니다. 대신, 때때로 **비대칭 중첩(asymmetric superposition)**이 발생하는데, 이는 단일 neuron이 두 개의 서로 다른 feature $i$와 $j$를 감지하고, 이 feature들을 서로 다른 크기로 저장하는 경우를 말합니다 (feature $i$에 대한 $W_1$ 벡터가 훨씬 더 크다고 가정합니다). $W_2$ 벡터들은 크기가 반전되어 있습니다 (즉, $j$에 대한 벡터가 훨씬 더 큽니다). $i$이 존재하고 $j$이 존재하지 않을 때는 문제가 없습니다. feature $i$에 대한 출력은 `large * small` (정확한 크기)이고, $j$에 대한 출력은 `small * small` (0에 가까움)이기 때문입니다. 하지만 $j$이 존재하고 $i$이 존재하지 않을 때, feature $j$에 대한 출력은 `small * large` (정확한 크기)이지만, $i$에 대한 출력은 `large * large` (있어야 할 크기보다 훨씬 큼)이 됩니다. 특히 $i$에 대한 출력의 부호가 양수일 때 이것은 문제가 됩니다. 모델은 $j$이 존재하고 $i$가 존재하지 않는 경우를 수정하기 위해 다른 neuron을 재용도화하여 이를 해결합니다. 정확한 메커니즘은 생략하지만, 이는 모델의 맨 마지막에 ReLU가 있다는 점을 이용합니다. 따라서 feature에 대한 출력이 매우 크고 음수여도 상관없지만 (loss가 0에서 절단됨), 크고 양수인 것은 매우 좋지 않습니다.

자세한 내용은 링크된 Anthropic 논문의 섹션을 읽어보시기 바랍니다. 아래에 이 그래프의 결과를 재현할 수 있는 코드를 제공했습니다. 일부 그래프는 위에서 설명한 비대칭 중첩의 형태를 보여주는 반면, 다른 그래프들은 단순히 각 feature에 대해 한 쌍의 neuron만을 가질 수 있습니다. Anthropic의 그래프와 정확히 유사한 결과를 관찰하려면 몇 가지 random seed를 실행해 보아야 할 수도 있습니다. 출력이 무엇을 나타내는지 이해하시겠습니까? 하이퍼파라미터(예: 서로 다른 feature 확률 또는 중요도)를 조정하며 이 동작이 어떻게 변하는지 확인해 보시겠습니까?

In [ ]:
cfg = ToyModelConfig(n_inst=6, n_features=10, d_hidden=10)

importance = 0.8 ** t.arange(1, 1 + cfg.n_features)
feature_probability = 0.35  # slightly lower feature probability, to encourage a small degree of superposition

model = NeuronComputationModel(
    cfg=cfg,
    device=device,
    importance=importance[None, :],
    feature_probability=feature_probability,
)
model.optimize()

In [ ]:
utils.plot_features_in_Nd_discrete(
    W1=model.W1,
    W2=model.W2,
    title="Neuron computation model (colored discretely, by feature)",
    legend_names=[f"I<sub>{i}</sub> = {importance.squeeze()[i]:.3f}" for i in range(cfg.n_features)],
)

## 요약 - 무엇을 배웠습니까?

이와 같은 toy 모델을 사용할 때는 단순히 학습 설정의 세부 사항이 아니라, 일반화 가능한 교훈을 얻는 것이 중요합니다.

이 논문에서 얻어야 할 핵심 내용은 다음과 같습니다:

* superposition이란 무엇인가
* feature 중요도와 sparsity에 따라 superposition이 어떻게 변하는가
* 상관관계(correlated) 또는 반상관관계(anticorrelated) feature가 있을 때 어떻게 변하는가
* neuron superposition과 bottleneck superposition의 차이 (또는 동일하게 "computational 및 representational supervision"의 차이)

# 3️⃣ Feature Geometry

> ##### 학습 목표
>
> - 특정 feature에 차원의 어느 정도 비율이 할당되었는지를 측정하는 *dimensionality*에 대해 학습합니다.
> - superposition 뒤에 숨겨진 기하학적 직관을 이해하고, 이것이 더 큰 모델에서의 일반적인 superposition 개념과 어떻게 연관되는지 이해합니다.

> 참고 - 이 섹션은 여기서 사용하는 구체적인 문제 설정에 대해 매우 상세하게 다루므로 선택 사항입니다. 원하신다면 다음 섹션으로 넘어가셔도 좋습니다.

## 차원 (Dimensionality)

우리는 superposition이 모델로 하여금 추가적인 feature들을 표현할 수 있게 하며, sparsity를 높일수록 추가되는 feature의 수가 증가한다는 것을 확인했습니다. 이 섹션에서는 이 관계를 더 자세히 조사하며, feature들이 오각형이나 사면체와 같은 기하학적 구조로 스스로를 조직하는 것처럼 보인다는 예상치 못한 기하학적 이야기를 발견해 보겠습니다!

아래 코드는 모든 importance가 동일한 세 번째 실험을 실행합니다. 우리는 먼저 모델이 표현하도록 학습한 feature의 수에 관심을 가집니다. 이는 가중치 행렬의 **Frobenius norm**의 제곱 $W$, 즉 $||W||_F^2 = \sum_{ij}W_{ij}^2$ 로 잘 표현됩니다.

<details>
<summary>질문 - 이것이 표현된 feature의 수를 측정하는 좋은 지표인 이유를 알 수 있습니까?</summary>

합산 순서를 변경함으로써, Frobenius norm의 제곱이 각 2D embedding 벡터의 제곱 norm의 합과 같음을 보일 수 있습니다:

$$
\big\|W\big\|_F^2 = \sum_j \left(\sum_i W_{ij}^2\right) = \sum_{j}\big\|W_{[:, j]}\big\|^2
$$

각 embedding 벡터는 feature가 표현된 경우 제곱 norm이 대략 $1$ 이고, 표현되지 않은 경우 $0$ 입니다. 따라서 이는 대략적으로 표현된 전체 feature의 수와 같습니다.
</details>

아래 코드를 실행하면, "feature당 차원 수"의 총합인 $m/\big\|W\big\|_F^2$ 도 함께 그래프로 그리게 됩니다.

In [ ]:
cfg = ToyModelConfig(n_features=200, d_hidden=20, n_inst=20)

# For this experiment, use constant importance across features (but still vary sparsity across instances)
feature_probability = 20 ** -t.linspace(0, 1, cfg.n_inst)

model = ToyModel(
    cfg=cfg,
    device=device,
    feature_probability=feature_probability[:, None],
)
model.optimize(steps=10_000)

In [ ]:
utils.plot_feature_geometry(model)

놀랍게도, 우리는 이 그래프가 $1$와 $1/2$에서 "sticky"하다는 것을 발견했습니다. 살펴보면, 이 $1/2$ "sticky point"는 feature들이 서로 정확히 음수 관계인 "antipodal pairs"로 나타나는 정밀한 기하학적 배치에 대응하는 것으로 보이며, 이를 통해 각 hidden dimension에 두 개의 feature를 채워 넣을 수 있게 됩니다. antipodal pairs가 매우 효율적이기 때문에, 모델이 광범위한 sparsity regime에서 이를 우선적으로 사용하는 것으로 보입니다.

알고 보니 antipodal pairs는 빙산의 일각에 불과했습니다. 이 곡선 아래에는 매우 구체적인 feature들의 기하학적 구성들이 숨겨져 있습니다.

이러한 기하학적 구성들을 어떻게 발견할 수 있을까요? 저자들이 feature의 **dimensionality**라고 명명한 다음 metric을 고려해 보겠습니다:

$$
D_i = \frac{\big\|W_i\big\|^2}{\sum_{j} \big( \hat{W_i} \cdot W_j \big)^2}
$$

직관적으로, 이것은 특정 feature가 "차원의 어느 정도 비율"을 차지하는지를 측정하는 척도입니다. 이 metric에 대한 몇 가지 직관을 얻어 보겠습니다:

* 0보다 작을 수 없습니다.
    * 벡터가 zero vector일 때, 즉 feature가 표현되지 않았을 때에만 0과 같습니다.
* 1보다 클 수 없습니다 ($j = i$일 때, 분모 합의 항이 분자와 같기 때문입니다).
    * $i$번째 feature vector $W_i$이 다른 모든 feature와 orthogonal할 때에만 1과 같습니다 (이 경우 분모 합에서 $j=i$이 유일한 항이 되기 때문입니다).
    * 직관적으로, 이 경우 해당 feature는 하나의 차원 전체를 독점하게 됩니다.
* 서로 평행하고 다른 모든 feature와 orthogonal한 $k$개의 feature가 있다면, 이들은 dimensionality를 동일하게 "공유"하며, 즉 각각 $D_i = 1/k$가 됩니다.
* 모든 $D_i$의 합은 전체 feature 수인 $m$보다 클 수 없으며, 모든 벡터가 orthogonal할 때에만 등호가 성립합니다.

### 연습 문제 - 차원 계산하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-20 minutes on this exercise.
> ```

$W$의 shape은 `(n_inst, d_hidden, n_features)`임을 기억하십시오. 벡터 $W_i$은 feature 벡터를 의미하며 (즉, 길이는 `d_hidden`입니다), `n_inst` 차원에 대해 계산을 broadcast 해야 합니다.

In [ ]:
@t.inference_mode()
def compute_dimensionality(
    W: Float[Tensor, "n_inst d_hidden n_features"],
) -> Float[Tensor, "n_inst n_features"]:
    raise NotImplementedError()


tests.test_compute_dimensionality(compute_dimensionality)

<details><summary>솔루션</summary>

```python
@t.inference_mode()
def compute_dimensionality(
    W: Float[Tensor, "n_inst d_hidden n_features"],
) -> Float[Tensor, "n_inst n_features"]:
    W_norms = W.norm(dim=1, keepdim=True)
    numerator = W_norms.squeeze() ** 2

    # Compute denominator terms
    W_normalized = W / (W_norms + 1e-8)
    denominator = einops.einsum(W_normalized, W, "i h f1, i h f2 -> i f1 f2").pow(2).sum(-1)

    return numerator / denominator
```
</details>

아래 코드는 여러 인스턴스에 걸쳐 sparsity 레벨이 증가함에 따른 차원의 비율을 그래프로 나타냅니다.

In [ ]:
W = model.W.detach()
dim_fracs = compute_dimensionality(W)

utils.plot_feature_geometry(model, dim_fracs=dim_fracs)

여기서 무슨 일이 일어나고 있는 것일까요? 모델은 특정한 weight geometry를 생성하는 경향이 있으며, 서로 다른 configuration 사이를 일종의 도약하듯 이동하는 것으로 나타납니다. 예를 들어:

* sparsity가 0(또는 매우 낮을) 때, feature basis는 그 어떤 것보다 우선시되지 않으며, 따라서 모델은 대신 임의의 방향으로 feature를 표현합니다. 일부 feature는 충실하게 표현되고 다른 feature는 그렇지 않아야 할 이유가 없습니다.
* sparsity 수준이 높아지면, feature basis가 우선시됩니다. 이에 따라 모델은 일부 feature를 antipodal pair로 표현하는 상태로 phase-transition하며, 나머지는 해석되지 않습니다.
* sparsity가 더욱 증가하면, 다른 geometry로 전이됩니다 (아래 다이어그램 참조).

교훈은 무엇일까요? superposition은 정확히 정의하기 매우 어렵습니다! 차원 0(feature를 학습하지 않음)과 1(feature에 차원을 할당함) 사이에는 많은 지점이 존재합니다. 비유를 들자면, 우리는 흔히 물이 얼음, 물, 수증기의 세 가지 상태로만 존재한다고 생각합니다. 하지만 이는 단순화한 것이며, 실제로는 얼음에도 다양한 상태가 존재하며 이는 종종 서로 다른 결정 구조(예: 육방정계 vs 입방정계 얼음)에 대응합니다. 이와 비슷하게, neural network feature 또한 "superposition"이라는 일반적인 범주 내에서 많은 다른 phase를 가지는 것으로 보입니다.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/grid_all.png" width="900">

이 결과들에 너무 큰 의미를 부여하지 않도록 주의해야 합니다. 많은 부분이 실험 설정의 세부 사항에 민감하게 의존합니다 (예를 들어, 우리는 positive semidefinite matrix인 $W^T W$을 사용했으며, 이러한 저차원 symmetric pos-semidef matrix와 위 그래프에서 본 polytope 종류 사이에는 대응 관계가 있습니다). 하지만 이를 통해 feature를 더 적은 차원에 패킹할 때 고려해야 할 관련 사항들에 대해 감을 잡으셨기를 바랍니다.

# 4️⃣ Superposition & Deep Double Descent

> ##### 학습 목표
>
> - deep double descent 현상을 이해하고 특징을 정의합니다.
> - double descent의 서로 다른 단계들을 superposition 개념과 연결합니다.
> - 가이드가 적은 환경에서 논문의 결과를 직접 재현하는 연습을 합니다.

> 참고 - 이 섹션은 구조화된 연습 문제 세트라기보다, 가이드가 제공되는 재현 과정에 가깝습니다. 논문을 재현하는 능력(특히 toy 모델과 일부 저수준 수학 및 ML 관련 논문)을 향상시키는 데 관심이 있다면 시도해 보시는 것을 추천합니다. 빠른 피드백 루프를 통해 연습 문제 세트를 진행하며 superposition 및 SAEs에 대해 최대한 많이 배우는 것에 더 관심이 있다면, 이 섹션의 핵심 결과만 읽거나 아예 건너뛰셔도 좋습니다.

이번 추천 재현 실험에서는 Double Descent와 superposition에 관한 [Anthropic paper](https://transformer-circuits.pub/2023/toy-double-descent/index.html)을 살펴보겠습니다. 이 논문은 [double descent](https://openai.com/research/deep-double-descent) 현상을 superposition 모델과 연결 짓습니다. 이 논문에서 제시하는 이론은 대략 다음과 같습니다:

* 초기에 모델은 데이터 포인트들이 superposition 상태로 표현되는 **memorising solution**을 학습합니다. 이는 일반화되지 않으므로, training loss는 낮지만 test loss는 높게 나타납니다.
* 이후 모델은 feature들이 학습되고 superposition 상태로 표현되는 **generalizing solution**을 학습합니다. 이는 일반화되므로, training loss와 test loss가 모두 낮게 나타납니다.
* 이 두 솔루션 사이에서 loss가 급증하는 현상은 모델이 memorising solution에서 generalizing solution으로 전환될 때 발생합니다.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/ddd-superposn.png" width="700">

데이터 포인트를 superposition 상태로 표현한다는 것은 무엇을 의미할까요? 이전 섹션에서 상관관계가 있는(correlated) 또는 반상관관계가 있는(anticorrelated) feature에 대한 연습 문제를 풀었다면, 반상관관계 feature들이 서로 간섭하지 않기 때문에 superposition 상태로 표현하기 더 쉽다는 것을 알고 계실 것입니다. 특히 feature들이 단순히 반상관관계인 것을 넘어 **상호 배타적(mutually exclusive)**일 때 더욱 그렇습니다. Anthropic 논문의 내용은 다음과 같습니다:

> 텍스트를 그대로 암기하는 언어 모델의 경우를 생각해 보십시오. 모델은 이를 어떻게 수행할 수 있을까요? 한 가지 단순한 생각은 뉴런을 사용하여 시퀀스를 임의의 연속 텍스트로 매핑하는 lookup table을 만드는 것입니다. 암기하려는 모든 token 시퀀스에 대해, 해당 시퀀스를 감지하는 뉴런 하나를 할당하고, 그 뉴런이 활성화될 때 임의의 동작을 구현하는 방식입니다. 이 접근 방식의 문제는 매우 비효율적이라는 점입니다. 하지만 각 케이스가 상호 배타적이며 간섭할 수 없으므로, superposition을 적용하기에 완벽한 후보로 보입니다.

우리는 toy model의 맥락에서 이 이론을 연구하겠습니다. 구체적으로, 이 논문의 첫 번째 섹션에서 다루었던 toy model을 사용하되, 매우 특정한 방식으로 학습시킬 것입니다. 즉, 무작위 데이터 배치를 하나 생성한 다음, 전체 학습 과정 동안 동일한 배치를 사용하는 방식입니다. 배치 크기가 변할 때, 그리고 feature의 수가 변할 때 어떤 일이 일어나는지 살펴보겠습니다. 이론에 따르면, 배치 크기가 feature 수보다 작을 때는 모델이 데이터 포인트를 superposition 상태로 표현해야 하며, 배치 크기가 feature 수보다 클 때는 feature를 superposition 상태로 표현해야 합니다.

완료해야 할 일련의 연습 문제를 제공하는 대신, 이 섹션은 개방형으로 남겨두겠습니다. 이를 구조화된 연습 문제라기보다 논문 재현 실험으로 생각하시기 바랍니다. 다만, 몇 가지 팁을 드리겠습니다:

* Adam optimizer 대신 논문에서는 기본 weight decay가 `WEIGHT_DECAY = 1e-2`인 AdamW를 권장합니다.
    * Weight decay는 weight의 norm을 제한하여 너무 커지지 않게 합니다. weight decay가 없다면, 이론적으로 임의의 매우 많은 데이터 포인트를 암기하고 이를 단위 원(unit circle) 주변에 균등하게 배치하여 표현할 수 있습니다. 그러면 이를 투영할 수 있는 충분히 큰 weight 벡터만 있다면 완벽하게 재구성할 수 있습니다.
* 논문에서는 `NUM_WARMUP_STEPS = 2500`까지 선형 warmup(즉, learning rate를 0부터 `LEARNING_RATE = 1e-3`까지 선형적으로 증가시킴)을 거친 후, 학습 종료 시점인 `NUM_BATCH_UPDATES = 50_000`까지 cosine decay를 적용하는 learning rate를 권장합니다.
* 논문에서는 feature에 대해 0.999의 sparsity와 총 10,000개의 feature를 사용할 것을 권장합니다. 하지만 저희는 대신 `SPARSITY = 0.99`과 `N_FEATURES = 1000`을 사용할 것을 권장합니다 (Marius Hobbhahn의 재현 실험을 따름). 이렇게 하면 근본적으로 동일한 패턴을 관찰하면서도 모델 학습 속도를 높일 수 있습니다.
* 데이터 배치를 생성할 때, 데이터를 정규화해야 합니다 (즉, 주어진 배치 인덱스와 인스턴스에 대한 각 벡터가 unit norm을 갖도록 합니다). 나머지 데이터 생성 과정은 이 노트북의 첫 번째 섹션과 동일해야 합니다.
* 기술적으로는 인스턴스가 하나만 있어도 충분합니다. 하지만 학습 종료 시점에 loss가 가장 낮은 인스턴스를 선택할 수 있도록 몇 개(예: 5-10개)를 사용하는 것을 권장합니다. 이는 (우리의 친절한 친구인 무작위성 덕분에) 모든 인스턴스가 반드시 최적의 솔루션을 학습하는 것은 아니기 때문입니다. 저희의 구현(아래 코드)에서는 `optimize` 함수를 수정하여 마지막에 `(batch_inst, W_inst)`을 반환하도록 했습니다. 여기서 `batch_inst`은 학습 종료 시점에 loss가 가장 낮았던 배치이며, `W_inst`는 해당 인스턴스에 대해 학습된 weight입니다. 이것이 바로 논문에 등장하는 2D feature plot을 그리는 데 필요한 데이터입니다.
* feature geometry 섹션에서 **dimensionality**를 계산하는 함수를 재활용할 수 있습니다. feature의 차원뿐만 아니라 데이터 포인트의 차원까지 측정하는 일반화된 dimensionality 함수에 대한 논의는 논문을 참조하십시오.

시작을 돕기 위해, 유용하게 사용할 수 있는 몇 가지 상수를 제공합니다:

In [ ]:
NUM_WARMUP_STEPS = 2500
NUM_BATCH_UPDATES = 50_000

WEIGHT_DECAY = 1e-2
LEARNING_RATE = 1e-3

BATCH_SIZES = [3, 5, 6, 8, 10, 15, 30, 50, 100, 200, 500, 1000, 2000]

N_FEATURES = 1000
N_INSTANCES = 5
N_HIDDEN = 2
SPARSITY = 0.99
FEATURE_PROBABILITY = 1 - SPARSITY

또한, 시각화에 도움이 필요하시다면, 아래 코드를 통해 [this figure](https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/fig-2d.png) 하단에서 볼 수 있는 것과 같은 2D feature 시각화 결과물을 모든 batch size에 대해 가로로 쌓아서 생성할 수 있습니다. 이때 다음 사항을 가정합니다:

* `features_list`은 단일 instance에 대한 detached `W`-matrix들의 리스트이며, 즉 각각의 shape은 `(2, n_features)`입니다 (이들은 첫 번째 행의 파란색 plot을 생성하는 데 사용됩니다).
* `data_list`는 동일한 instance의 hidden direction으로 투영된 batch 데이터의 리스트이며, 즉 각각의 shape은 `(2, batch_size)`입니다 (이들은 두 번째 행의 빨간색 plot을 생성하는 데 사용됩니다).

아래에 데모가 제공됩니다 (값들은 의미가 없으며, 이 함수를 사용하는 방법을 보여주기 위해 무작위로 생성되었습니다).

In [ ]:
features_list = [t.randn(size=(2, 100)) for _ in BATCH_SIZES]
hidden_representations_list = [t.randn(size=(2, batch_size)) for batch_size in BATCH_SIZES]

utils.plot_features_in_2d(
    features_list + hidden_representations_list,
    colors=[["blue"] for _ in range(len(BATCH_SIZES))] + [["red"] for _ in range(len(BATCH_SIZES))],
    title="Double Descent & Superposition (num features = 100)",
    subplot_titles=[f"Features (batch={bs})" for bs in BATCH_SIZES] + [f"Data (batch={bs})" for bs in BATCH_SIZES],
    allow_different_limits_across_subplots=True,
    n_rows=2,
)

아래에 구현 코드를 제공해 드립니다. 하지만 코드를 너무 많이 살펴보기 전에 먼저 직접 시도해 보시는 것을 권장합니다. 어려움을 겪고 계신다면, 다양한 드롭다운 메뉴를 통해 단계별 힌트를 확인하실 수 있습니다.

In [ ]:
# YOUR CODE HERE - replicate the results from the Superposition & Deep Double Descent paper!

구현 과정에서 특히 어려움을 겪는 부분이 있다면, 처음 4개의 힌트가 구체적인 코드 조각들을 제공합니다:

<details>
<summary>힌트 (기본 설정 코드)</summary>

기본 import 및 상수:

```python
import math
from typing import Any
import pandas as pd
import plotly.express as px

NUM_WARMUP_STEPS = 2500
NUM_BATCH_UPDATES = 50_000
# EVAL_N_DATAPOINTS = 1_000

WEIGHT_DECAY = 1e-2
LEARNING_RATE = 1e-3

BATCH_SIZES = [3, 4, 5, 6, 8, 10, 15, 20, 30, 50, 100, 200, 300, 500, 1000, 2000, 3000]
# SMALLER_BATCH_SIZES = [3, 6, 10, 30, 100, 500, 2000]

N_FEATURES = 1000
N_INSTANCES = 10
D_HIDDEN = 2
SPARSITY = 0.99
FEATURE_PROBABILITY = 1 - SPARSITY
```

Anthropic의 설명서에 따른 새로운 scheduler들입니다:

```python
def linear_warmup_lr(step, steps):
    """Increases linearly from 0 to 1."""
    return step / steps

def anthropic_lr(step, steps):
    """As per the description in the paper: 2500 step linear warmup, followed by cosine decay to zero."""
    if step < NUM_WARMUP_STEPS:
        return linear_warmup_lr(step, NUM_WARMUP_STEPS)
    else:
        return cosine_decay_lr(step - NUM_WARMUP_STEPS, steps - NUM_WARMUP_STEPS)
```

</details>

<details>
<summary>힌트 (<code>ToyModel</code> 클래스의 새로운 버전 코드)</summary>

```python
class DoubleDescentModel(ToyModel):
    W: Float[Tensor, "inst d_hidden feats"]
    b_final: Float[Tensor, "inst feats"]
    # Our linear map (for a single instance) is x -> ReLU(W.T @ W @ x + b_final)

    @classmethod
    def dimensionality(
        cls, data: Float[Tensor, "... batch d_hidden"]
    ) -> Float[Tensor, "... batch"]:
        """
        Calculates dimensionalities of data. Assumes data is of shape ... batch d_hidden, i.e. if it's 2D then
        it's a batch of vectors of length `d_hidden` and we return the dimensionality as a 1D tensor of length
        `batch`. If it has more dimensions at the start, we assume this means separate calculations for each
        of these dimensions (i.e. they are independent batches of vectors).
        """
        # Compute the norms of each vector (this will be the numerator)
        squared_norms = einops.reduce(data.pow(2), "... batch d_hidden -> ... batch", "sum")
        # Compute the denominator (i.e. get the dot product then sum over j)
        data_normed = data / data.norm(dim=-1, keepdim=True)
        interference = einops.einsum(
            data_normed, data, "... batch_i d_hidden, ... batch_j d_hidden -> ... batch_i batch_j"
        )
        polysemanticity = einops.reduce(
            interference.pow(2), "... batch_i batch_j -> ... batch_i", "sum"
        )
        assert squared_norms.shape == polysemanticity.shape

        return squared_norms / polysemanticity

    def generate_batch(self, batch_size: int) -> Float[Tensor, "batch inst feats"]:
        """
        New function for generating batch, so we can normalize it.
        """
        # Get batch from prev method
        batch = super().generate_batch(batch_size)

        # Normalize the batch (i.e. so each vector for a particular batch & instance has norm 1)
        # (need to be careful about vectors with norm zero)
        norms = batch.norm(dim=-1, keepdim=True)
        norms = t.where(norms.abs() < 1e-6, t.ones_like(norms), norms)
        batch_normed = batch / norms
        return batch_normed

    def calculate_loss(
        self,
        out: Float[Tensor, "batch inst feats"],
        batch: Float[Tensor, "batch inst feats"],
        per_inst: bool = False,
    ) -> Float[Tensor, "inst"]:
        """
        New function to calculate loss, because we need a "loss per instance" option to find the best
        instance at the end of our optimization.
        """
        error = self.importance * ((batch - out) ** 2)
        loss = einops.reduce(error, "batch inst feats -> inst", "mean")
        return loss if per_inst else loss.sum()

    def optimize(
        self,
        batch_size: int,
        steps: int = NUM_BATCH_UPDATES,
        log_freq: int = 100,
        lr: float = LEARNING_RATE,
        lr_scale: Callable[[int, int], float] = anthropic_lr,
        weight_decay: float = WEIGHT_DECAY,
    ) -> tuple[Tensor, Tensor]:
        optimizer = t.optim.AdamW(list(self.parameters()), lr=lr, weight_decay=weight_decay)

        progress_bar = tqdm(range(steps))

        # Same batch for each step
        batch = self.generate_batch(batch_size)  # [batch_size inst n_features]

        for step in progress_bar:
            # Update learning rate
            step_lr = lr * lr_scale(step, steps)
            for group in optimizer.param_groups:
                group["lr"] = step_lr

            # Optimize
            optimizer.zero_grad()
            out = self.forward(batch)
            loss = self.calculate_loss(out, batch)
            loss.backward()
            optimizer.step()

            # Display progress bar
            if (step % log_freq == 0) or (step + 1 == steps):
                progress_bar.set_postfix(loss=loss.item() / self.cfg.n_inst, lr=step_lr)

        # Generate one final batch to compute the loss (we want only the best instance!)
        with t.inference_mode():
            out = self.forward(batch)
            loss_per_inst = self.calculate_loss(out, batch, per_inst=True)
            best_inst = loss_per_inst.argmin()
            print(f"Best instance = #{best_inst}, with loss {loss_per_inst[best_inst].item():.4e}")

        return batch[:, best_inst], self.W[best_inst].detach()
```

</details>

<details>
<summary>힌트 (모델 학습 및 2D feature plot 재현 코드)</summary>

```python
features_list = []
hidden_representations_list = []

for batch_size in tqdm(BATCH_SIZES):
    # Define our model
    cfg = ToyModelConfig(n_features=N_FEATURES, n_inst=N_INSTANCES, d_hidden=D_HIDDEN)
    model = DoubleDescentModel(cfg, feature_probability=FEATURE_PROBABILITY).to(device)

    # Optimize, and return the best batch & weight matrix
    batch_inst, W_inst = model.optimize(steps=15_000, batch_size=batch_size)

    # Calculate the hidden feature representations, and add both this and weight matrix to our lists of data
    with t.inference_mode():
        hidden = einops.einsum(
            batch_inst, W_inst, "batch features, hidden features -> hidden batch"
        )
    features_list.append(W_inst.cpu())
    hidden_representations_list.append(hidden.cpu())
```

2D feature plot 시각화:

```python
utils.plot_features_in_2d(
    features_list + hidden_representations_list,
    colors=[["blue"] for _ in range(len(BATCH_SIZES))] + [["red"] for _ in range(len(BATCH_SIZES))],
    title="Double Descent & Superposition (num features = 1000)",
    subplot_titles=[f"Features (batch={bs})" for bs in BATCH_SIZES] + [f"Data (batch={bs})" for bs in BATCH_SIZES],
    allow_different_limits_across_subplots=True,
    n_rows=2,
)
```

다음과 같은 결과를 얻어야 합니다:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/ddd_fig1.png" width="1400">

</details>

<details>
<summary>힌트 (dimensionality plot 재현 코드)</summary>

```python
df_data = {"Batch size": [], "Dimensionality": [], "Data": []}

for batch_size, model_W, hidden in zip(BATCH_SIZES, features_list, hidden_representations_list):
    # Get x-axis data (batch size), and color (blue or red)
    df_data["Batch size"].extend([batch_size] * (N_FEATURES + batch_size))
    df_data["Data"].extend(["features"] * N_FEATURES + ["hidden"] * batch_size)

    # Calculate dimensionality of model.W[inst].T, which has shape [d_hidden=2 N_FEATURES]
    feature_dim = DoubleDescentModel.dimensionality(model_W.T)
    assert feature_dim.shape == (N_FEATURES,)

    # Calculate dimensionality of model's batch data hidden representation. This has shape [d_hidden=2 batch_size]
    data_dim = DoubleDescentModel.dimensionality(hidden.T)
    assert data_dim.shape == (batch_size,)

    # Add them both to the data
    df_data["Dimensionality"].extend(feature_dim.tolist() + data_dim.tolist())


df = pd.DataFrame(df_data)
eps = 0.01
xline1, xline2 = (100 * 200) ** 0.5, (500 * 1000) ** 0.5
vrect_kwargs: dict[str, Any] = dict(opacity=0.5, layer="below", line_width=0)
xrange = [math.log10(1.5), math.log10(5000)]
fig = (
    px.strip(
        df,
        x="Batch size",
        y="Dimensionality",
        color="Data",
        color_discrete_sequence=["rgba(0,0,255,0.3)", "rgba(255,0,0,0.3)"],
        log_x=True,
        template="simple_white",
        width=1000,
        height=600,
        title="Dimensionality of features & hidden representation of training examples",
    )
    .update_traces(marker=dict(opacity=0.5))
    .update_layout(
        xaxis=dict(range=xrange, tickmode="array", tickvals=BATCH_SIZES),
        yaxis=dict(range=[-0.05, 1.0]),
    )
    .add_vrect(x0=1, x1=(1 - eps) * xline1, fillcolor="#ddd", **vrect_kwargs)
    .add_vrect(x0=(1 + eps) * xline1, x1=(1 - eps) * xline2, fillcolor="#ccc", **vrect_kwargs)
    .add_vrect(x0=(1 + eps) * xline2, x1=10_000, fillcolor="#bbb", **vrect_kwargs)
    .add_scatter(
        x=BATCH_SIZES,
        y=[2 / b for b in BATCH_SIZES],
        mode="lines",
        line=dict(shape="spline", dash="dot", color="#333", width=1),
        name="d_hidden / batch_size",
    )
)

fig.show()
```

다음과 같은 결과를 얻어야 합니다:

<img src="https://info-arena.github.io/ARENA_img/misc/media-1320/1320-D.png" width="1000">

</details>

마지막으로, 여기에서 전체 솔루션 코드를 확인할 수 있습니다:

<details>
<summary>솔루션 (전체)</summary>

```python
import math
from typing import Any

import pandas as pd
import plotly.express as px

NUM_WARMUP_STEPS = 2500
NUM_BATCH_UPDATES = 50_000
# EVAL_N_DATAPOINTS = 1_000

WEIGHT_DECAY = 1e-2
LEARNING_RATE = 1e-3

BATCH_SIZES = [3, 4, 5, 6, 8, 10, 15, 20, 30, 50, 100, 200, 300, 500, 1000, 2000, 3000]
# SMALLER_BATCH_SIZES = [3, 6, 10, 30, 100, 500, 2000]

N_FEATURES = 1000
N_INSTANCES = 10
D_HIDDEN = 2
SPARSITY = 0.99
FEATURE_PROBABILITY = 1 - SPARSITY


def linear_warmup_lr(step, steps):
    """Increases linearly from 0 to 1."""
    return step / steps


def anthropic_lr(step, steps):
    """As per the description in the paper: 2500 step linear warmup, followed by cosine decay to zero."""
    if step < NUM_WARMUP_STEPS:
        return linear_warmup_lr(step, NUM_WARMUP_STEPS)
    else:
        return cosine_decay_lr(step - NUM_WARMUP_STEPS, steps - NUM_WARMUP_STEPS)


class DoubleDescentModel(ToyModel):
    W: Float[Tensor, "inst d_hidden feats"]
    b_final: Float[Tensor, "inst feats"]
    # Our linear map (for a single instance) is x -> ReLU(W.T @ W @ x + b_final)

    @classmethod
    def dimensionality(cls, data: Float[Tensor, "... batch d_hidden"]) -> Float[Tensor, "... batch"]:
        """
        Calculates dimensionalities of data. Assumes data is of shape ... batch d_hidden, i.e. if
        it's 2D then it's a batch of vectors of length `d_hidden` and we return the dimensionality
        as a 1D tensor of length `batch`. If it has more dimensions at the start, we assume this
        means separate calculations for each of these dimensions (i.e. they are independent batches
        of vectors).
        """
        # Compute the norms of each vector (this will be the numerator)
        squared_norms = einops.reduce(data.pow(2), "... batch d_hidden -> ... batch", "sum")
        # Compute the denominator (i.e. get the dot product then sum over j)
        data_normed = data / data.norm(dim=-1, keepdim=True)
        interference = einops.einsum(
            data_normed, data, "... batch_i d_hidden, ... batch_j d_hidden -> ... batch_i batch_j"
        )
        polysemanticity = einops.reduce(interference.pow(2), "... batch_i batch_j -> ... batch_i", "sum")
        assert squared_norms.shape == polysemanticity.shape

        return squared_norms / polysemanticity

    def generate_batch(self, batch_size: int) -> Float[Tensor, "batch inst feats"]:
        """
        New function for generating batch, so we can normalize it.
        """
        # Get batch from prev method
        batch = super().generate_batch(batch_size)

        # Normalize the batch (i.e. so each vector for a particular batch & instance has norm 1)
        # (need to be careful about vectors with norm zero)
        norms = batch.norm(dim=-1, keepdim=True)
        norms = t.where(norms.abs() < 1e-6, t.ones_like(norms), norms)
        batch_normed = batch / norms
        return batch_normed

    def calculate_loss(
        self,
        out: Float[Tensor, "batch inst feats"],
        batch: Float[Tensor, "batch inst feats"],
        per_inst: bool = False,
    ) -> Float[Tensor, " inst"]:
        """
        New function to calculate loss, because we need a "loss per instance" option to find the
        best instance at the end of our optimization.
        """
        error = self.importance * ((batch - out) ** 2)
        loss = einops.reduce(error, "batch inst feats -> inst", "mean")
        return loss if per_inst else loss.sum()

    def optimize(
        self,
        batch_size: int,
        steps: int = NUM_BATCH_UPDATES,
        log_freq: int = 100,
        lr: float = LEARNING_RATE,
        lr_scale: Callable[[int, int], float] = anthropic_lr,
        weight_decay: float = WEIGHT_DECAY,
    ) -> tuple[Tensor, Tensor]:
        optimizer = t.optim.AdamW(list(self.parameters()), lr=lr, weight_decay=weight_decay)

        progress_bar = tqdm(range(steps))

        # Same batch for each step
        batch = self.generate_batch(batch_size)  # [batch_size inst n_features]

        for step in progress_bar:
            # Update learning rate
            step_lr = lr * lr_scale(step, steps)
            for group in optimizer.param_groups:
                group["lr"] = step_lr

            # Optimize
            optimizer.zero_grad()
            out = self.forward(batch)
            loss = self.calculate_loss(out, batch)
            loss.backward()
            optimizer.step()

            # Display progress bar
            if (step % log_freq == 0) or (step + 1 == steps):
                progress_bar.set_postfix(loss=loss.item() / self.cfg.n_inst, lr=step_lr)

        # Generate one final batch to compute the loss (we want only the best instance!)
        with t.inference_mode():
            out = self.forward(batch)
            loss_per_inst = self.calculate_loss(out, batch, per_inst=True)
            best_inst = loss_per_inst.argmin()
            print(f"Best instance = #{best_inst}, with loss {loss_per_inst[best_inst].item():.4e}")

        return batch[:, best_inst], self.W[best_inst].detach()


# ! Results, part 1/2

features_list = []
hidden_representations_list = []

for batch_size in tqdm(BATCH_SIZES):
    # Define our model
    cfg = ToyModelConfig(n_features=N_FEATURES, n_inst=N_INSTANCES, d_hidden=D_HIDDEN)
    model = DoubleDescentModel(cfg, feature_probability=FEATURE_PROBABILITY).to(device)

    # Optimize, and return the best batch & weight matrix
    batch_inst, W_inst = model.optimize(steps=15_000, batch_size=batch_size)

    # Calculate the hidden feature representations, and add both this and weight matrix to our
    # lists of data
    with t.inference_mode():
        hidden = einops.einsum(batch_inst, W_inst, "batch features, hidden features -> hidden batch")
    features_list.append(W_inst.cpu())
    hidden_representations_list.append(hidden.cpu())

utils.plot_features_in_2d(
    features_list + hidden_representations_list,
    colors=[["blue"] for _ in range(len(BATCH_SIZES))] + [["red"] for _ in range(len(BATCH_SIZES))],
    title="Double Descent & Superposition (num features = 1000)",
    subplot_titles=[f"Features (batch={bs})" for bs in BATCH_SIZES] + [f"Data (batch={bs})" for bs in BATCH_SIZES],
    allow_different_limits_across_subplots=True,
    n_rows=2,
)

# ! Results, part 2/2

df_data = {"Batch size": [], "Dimensionality": [], "Data": []}

for batch_size, model_W, hidden in zip(BATCH_SIZES, features_list, hidden_representations_list):
    # Get x-axis data (batch size), and color (blue or red)
    df_data["Batch size"].extend([batch_size] * (N_FEATURES + batch_size))
    df_data["Data"].extend(["features"] * N_FEATURES + ["hidden"] * batch_size)

    # Calculate dimensionality of model.W[inst].T, which has shape [d_hidden=2 N_FEATURES]
    feature_dim = DoubleDescentModel.dimensionality(model_W.T)
    assert feature_dim.shape == (N_FEATURES,)

    # Calculate dimensionality of model's batch data hidden representation.
    # This has shape [d_hidden=2 batch_size]
    data_dim = DoubleDescentModel.dimensionality(hidden.T)
    assert data_dim.shape == (batch_size,)

    # Add them both to the data
    df_data["Dimensionality"].extend(feature_dim.tolist() + data_dim.tolist())

df = pd.DataFrame(df_data)
eps = 0.01
xline1, xline2 = (100 * 200) ** 0.5, (500 * 1000) ** 0.5
vrect_kwargs: dict[str, Any] = dict(opacity=0.5, layer="below", line_width=0)
xrange = [math.log10(1.5), math.log10(5000)]
fig = (
    px.strip(
        df,
        x="Batch size",
        y="Dimensionality",
        color="Data",
        color_discrete_sequence=["rgba(0,0,255,0.3)", "rgba(255,0,0,0.3)"],
        log_x=True,
        template="simple_white",
        width=1000,
        height=600,
        title="Dimensionality of features & hidden representation of training examples",
    )
    .update_traces(marker=dict(opacity=0.5))
    .update_layout(
        xaxis=dict(range=xrange, tickmode="array", tickvals=BATCH_SIZES),
        yaxis=dict(range=[-0.05, 1.0]),
    )
    .add_vrect(x0=1, x1=(1 - eps) * xline1, fillcolor="#ddd", **vrect_kwargs)
    .add_vrect(x0=(1 + eps) * xline1, x1=(1 - eps) * xline2, fillcolor="#ccc", **vrect_kwargs)
    .add_vrect(x0=(1 + eps) * xline2, x1=10_000, fillcolor="#bbb", **vrect_kwargs)
    .add_scatter(
        x=BATCH_SIZES,
        y=[2 / b for b in BATCH_SIZES],
        mode="lines",
        line=dict(shape="spline", dash="dot", color="#333", width=1),
        name="d_hidden / batch_size",
    )
)

fig.show()
```

</details>

# 5️⃣ Toy Model에서의 Sparse Autoencoders

> ##### 학습 목표
>
> - sparse autoencoder에 대해 배우고, 이것이 superposition으로 표현된 feature들을 어떻게 disentangle하는 데 사용될 수 있는지 학습합니다.
> - 이전 섹션의 toy model을 사용하여 직접 SAE를 훈련시키고, feature reconstruction 과정을 시각화합니다.
> - 중요한 SAE 훈련 전략(예: resampling)과 아키텍처 변형(예: Gated, Jump ReLU)을 이해합니다.

이제 sparse autoencoder로 넘어가겠습니다. 이는 Anthropic이 [recent paper](https://transformer-circuits.pub/2023/monosemantic-features/index.html)에서 탐구한 최근 연구 방향이며, 현재 mechanistic interpretability에서 가장 흥미로운 연구 분야 중 하나입니다.

다음 연습 문제 세트에서 여러분은 다음을 수행하게 됩니다:

- 직접 sparse autoencoder를 구축하고, 그 architecture와 loss function을 작성합니다.
- 이전에 정의한 `Model` class의 hidden activation에 대해 SAE를 학습시킵니다 (Anthropic 논문의 설정과 차이가 있음에 유의하십시오. 논문에서는 MLP layer에서 SAE를 학습시켰지만, 여기서는 non-privileged basis에서 학습시킵니다).
- SAE에서 feature를 추출하고, 이것이 모델이 학습한 feature와 동일한지 확인합니다.

Anthropic의 dictionary learning 논문(위의 링크)을 읽으십시오. 서론과 첫 번째 섹션(problem setup)부터 "Sparse Autoencoder Setup" 섹션까지 읽으시면 됩니다. 적어도 다음 질문들에 답할 수 있는지 확인하십시오:

<details>
<summary>autoencoder란 무엇이며, 무엇을 하도록 학습됩니까?</summary>

autoencoder는 라벨이 없는 데이터의 효율적인 encoding / representation을 학습하는 신경망의 한 종류입니다. 이는 입력을 어떤 방식으로든 **latent representation**으로 압축한 다음, 이를 다시 원래의 입력 공간으로 매핑하도록 학습됩니다. 입력과 재구성된 입력 사이의 reconstruction loss를 최소화함으로써 학습됩니다.

"encoding" 부분은 보통 latent space가 입력보다 저차원인 경우를 의미합니다. 하지만 sparse autoencoder에서 보게 되겠지만, 항상 그런 것은 아닙니다.

<img src="https://raw.githubusercontent.com/chloeli-15/ARENA_img/main/img/sae-diagram-2.png" width="900">

</details>

<details>
<summary>MLP layer에서 SAE를 학습시킬 때, 왜 autoencoder의 hidden dimension이 activation의 수보다 더 큽니까?</summary>

이전 드롭다운에서 언급했듯이, 보통 latent vector는 저차원이기 때문에 입력의 압축된 representation이 됩니다. 하지만 latent vector에 sparsity 제약을 가한다면(이는 어떤 의미에서 유효 차원을 줄이는 것입니다), 고차원이라 하더라도 여전히 압축된 representation이 될 수 있습니다.

특히 우리의 autoencoder 유스케이스에서 이렇게 하는 이유는, **neuron보다 feature가 더 많은** 경우에 superposition으로부터 feature를 복구하려고 하기 때문입니다. 우리는 autoencoder가 **overcomplete feature basis**를 학습하기를 기대합니다.

</details>

<details>
<summary>왜 L1 penalty는 sparsity를 유도합니까? (이 내용은 논문에 구체적으로 언급되어 있지 않지만, 이해해야 할 중요한 사항입니다.)</summary>

$L_2$ penalty와 달리, $L_1$ penalty는 실제로 값들을 0으로 밀어냅니다. 이는 통계학에서 잘 알려진 결과이며, 아래에서 가장 잘 설명되어 있습니다:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/l1-viz.png" width="450">

더 자세한 설명은 [this Google ML page](https://developers.google.com/machine-learning/crash-course/regularization-for-sparsity/l1-regularization)을 참조하십시오 (맥락 없이도 이해하기 좋은 애니메이션이 포함되어 있습니다!).

</details>

### 문제 설정

이전 모델의 설정을 다시 떠올려 보겠습니다:

$$
\begin{aligned}
h &= W x \\
x' &= \operatorname{ReLU}(W^T h + b)
\end{aligned}
$$

우리는 autoencoder가 hidden state activation $h$을 입력받아, 이를 더 큰 (overcomplete) hidden state $z$로 매핑한 다음, $z$로부터 원래의 hidden state $h$을 재구성하도록 학습시킬 것입니다.

$$
\begin{aligned}
z &= \operatorname{ReLU}(W_{enc}(h - b_{dec}) + b_{enc}) \\
h' &= W_{dec}z + b_{dec}
\end{aligned}
$$

가중치 행렬을 공유(tied)하지 않고 encoder와 decoder의 가중치 행렬을 다르게 선택했다는 점에 유의하십시오. 이에 대해서는 나중에 더 자세히 논의하겠습니다.

autoencoder와 모델의 표기법을 혼동하지 않는 것이 중요합니다. 모델은 feature $x$을 입력받아 **저차원** 벡터 $h$로 매핑한 다음, 이를 $x'$로 재구성한다는 점을 기억하십시오. 반면 autoencoder는 이러한 hidden state $h$를 입력받아 **고차원이지만 sparse한** 벡터 $z$로 매핑한 다음, 이를 $h'$로 재구성합니다. 우리의 희망은 $z$의 요소들이 $x$의 feature들에 대응하는 것입니다.

또 다른 참고 사항으로, 여기서 $b_{dec}$를 사용하는 것이 이상하게 보일 수 있습니다. 시작 부분에서 이를 뺐다가 마지막에 다시 더해주기 때문입니다. 우리가 이 항을 처리하는 방식은 **hidden state를 위한 중심화 항(centralizing term)**으로 사용하는 것입니다. 이는 학습된 평균 벡터를 빼줌으로써 $W_{enc}$가 중심화된 벡터에 작용할 수 있게 하며, 모델의 마지막 단계에서 이 항이 재구성된 hidden state에 다시 더해집니다.

### 표기법

autoencoder의 hidden activation은 여러 이름으로 불립니다. 때로는 **neurons**라고 불리기도 합니다 (MLP layer의 neurons처럼 activation function이 적용되어 privileged basis가 되기 때문입니다). 때로는 **features**라고도 불리는데, SAE의 아이디어는 이러한 hidden activation이 데이터의 특정 feature를 나타내도록 하는 것이기 때문입니다. 하지만 feature라는 단어는 약간 [overloaded](https://www.lesswrong.com/posts/9Nkb389gidsozY9Tf/lewis-smith-s-shortform#fd64ALuWK8rXdLKz6) - 이상적으로는 "feature"라는 말을 데이터 자체의 속성을 지칭하는 데 사용하고 싶을 것입니다. 만약 SAE의 weights가 무작위로 초기화되었다면, 이를 feature라고 부르는 것이 타당할까요?!

이러한 이유로, 우리는 autoencoder의 hidden activation을 **SAE latents**라고 부르겠습니다. 하지만 사람들이 때때로 "SAE features"나 "neurons"라는 용어를 대신 사용하므로 혼동하지 않도록 주의하시기 바랍니다 (예를 들어, 많은 사람들이 SAE의 weights를 리샘플링하는 것을 "neuron resampling"이라고 부릅니다).

이 섹션에서 채택할 새로운 표기법은 다음과 같습니다:

- `d_sae`: SAE의 hidden layer에 있는 activation의 수 (즉, latent dimension)입니다. SAE latents가 원래 데이터의 features와 대응되기를 원하므로 `d_sae >= n_features`가 필요합니다 (보통 이 섹션에서는 두 값이 같습니다).
- `d_in`: SAE input dimension입니다. SAE가 모델의 hidden activation을 재구성하므로 이는 이전 섹션의 `d_hidden`와 동일합니다. 하지만 SAE의 문맥에서 이를 `d_hidden`라고 부르는 것은 혼란스러울 수 있습니다. 보통 이 섹션에서는 결과를 시각화하기 위해 `d_in = d_hidden = 2`를 사용합니다.

<details>
<summary>질문 - 위의 공식들("Problem setup" 섹션)에서 x, x', z, h, h'의 shape은 무엇입니까?</summary>

batch 및 instance dimension을 제외하면 다음과 같습니다:

- `x`와 `x'`은 shape이 `(n_features,)`인 벡터입니다.
- `z`은 shape이 `(d_sae,)`인 벡터입니다.
- `h`와 `h'`은 shape이 `(d_in,)`인 벡터이며, 이는 이전 섹션의 `d_hidden`와 같습니다.

batch 및 instance dimension을 포함하면, 모든 shape은 앞에 추가적인 dimension `(batch_size, n_inst, d)`를 가집니다.

</details>

### SAE 클래스

아래에 `ToySAEConfig` 클래스를 제공했습니다. 인자들은 다음과 같습니다 (이후 연습 문제에서만 필요한 인자들은 생략합니다):

- `n_inst`: `ToyModel` 클래스에서와 동일한 의미입니다.
- `d_in`: SAE의 입력 크기입니다 (`ToyModel` 클래스의 `d_hidden`와 동일합니다).
- `d_sae`: SAE의 latent dimension 크기입니다.
- `sparsity_coeff`: loss function에서 사용됩니다.
- `weight_normalize_eps`: 가중치를 정규화할 때마다 분모에 더해지는 값입니다.
- `tied_weights`: encoder와 decoder 가중치가 tied 되었는지 여부를 결정하는 boolean 값입니다.
- `ste_epsilon`: 이후에 등장하는 JumpReLU SAE에만 해당됩니다.

또한 `ToySAE` 클래스도 제공했습니다. 앞으로 4개의 연습 문제에서 여러분이 할 일은 `__init__`, `W_dec_normalized`, `generate_batch` 및 `forward` 메서드를 완성하는 것입니다.

In [ ]:
@dataclass
class ToySAEConfig:
    n_inst: int
    d_in: int
    d_sae: int
    sparsity_coeff: float = 0.2
    weight_normalize_eps: float = 1e-8
    tied_weights: bool = False
    ste_epsilon: float = 0.01


class ToySAE(nn.Module):
    W_enc: Float[Tensor, "inst d_in d_sae"]
    _W_dec: Float[Tensor, "inst d_sae d_in"] | None
    b_enc: Float[Tensor, "inst d_sae"]
    b_dec: Float[Tensor, "inst d_in"]

    def __init__(self, cfg: ToySAEConfig, model: ToyModel) -> None:
        super(ToySAE, self).__init__()

        assert cfg.d_in == model.cfg.d_hidden, "Model's hidden dim doesn't match SAE input dim"
        self.cfg = cfg
        self.model = model.requires_grad_(False)
        self.model.W.data[1:] = self.model.W.data[0]
        self.model.b_final.data[1:] = self.model.b_final.data[0]

        raise NotImplementedError()

        self.to(device)

    @property
    def W_dec(self) -> Float[Tensor, "inst d_sae d_in"]:
        return self._W_dec if self._W_dec is not None else self.W_enc.transpose(-1, -2)

    @property
    def W_dec_normalized(self) -> Float[Tensor, "inst d_sae d_in"]:
        """
        Returns decoder weights, normalized over the autoencoder input dimension.
        """
        # You'll fill this in later
        raise NotImplementedError()

    def generate_batch(self, batch_size: int) -> Float[Tensor, "batch inst d_in"]:
        """
        Generates a batch of hidden activations from our model.
        """
        # You'll fill this in later
        raise NotImplementedError()

    def forward(
        self, h: Float[Tensor, "batch inst d_in"]
    ) -> tuple[
        dict[str, Float[Tensor, "batch inst"]],
        Float[Tensor, "batch inst"],
        Float[Tensor, "batch inst d_sae"],
        Float[Tensor, "batch inst d_in"],
    ]:
        """
        Forward pass on the autoencoder.

        Args:
            h: hidden layer activations of model

        Returns:
            loss_dict:       dict of different loss terms, each having shape (batch_size, n_inst)
            loss:            total loss (i.e. sum over terms of loss dict), same shape as loss terms
            acts_post:       autoencoder latent activations, after applying ReLU
            h_reconstructed: reconstructed autoencoder input
        """
        # You'll fill this in later
        raise NotImplementedError()

    def optimize(
        self,
        batch_size: int = 1024,
        steps: int = 10_000,
        log_freq: int = 100,
        lr: float = 1e-3,
        lr_scale: Callable[[int, int], float] = constant_lr,
        resample_method: Literal["simple", "advanced", None] = None,
        resample_freq: int = 2500,
        resample_window: int = 500,
        resample_scale: float = 0.5,
        hidden_sample_size: int = 256,
    ) -> list[dict[str, Any]]:
        """
        Optimizes the autoencoder using the given hyperparameters.

        Args:
            model:              we reconstruct features from model's hidden activations
            batch_size:         size of batches we pass through model & train autoencoder on
            steps:              number of optimization steps
            log_freq:           number of optimization steps between logging
            lr:                 learning rate
            lr_scale:           learning rate scaling function
            resample_method:    method for resampling dead latents
            resample_freq:      number of optimization steps between resampling dead latents
            resample_window:    number of steps needed for us to classify a neuron as dead
            resample_scale:     scale factor for resampled neurons
            hidden_sample_size: size of hidden value sample we add to the logs (for visualization)

        Returns:
            data_log:           dictionary containing data we'll use for visualization
        """
        assert resample_window <= resample_freq

        optimizer = t.optim.Adam(self.parameters(), lr=lr)  # betas=(0.0, 0.999)
        frac_active_list = []
        progress_bar = tqdm(range(steps))

        # Create lists of dicts to store data we'll eventually be plotting
        data_log = []

        for step in progress_bar:
            # Resample dead latents
            if (resample_method is not None) and ((step + 1) % resample_freq == 0):
                frac_active_in_window = t.stack(frac_active_list[-resample_window:], dim=0)
                if resample_method == "simple":
                    self.resample_simple(frac_active_in_window, resample_scale)
                elif resample_method == "advanced":
                    self.resample_advanced(frac_active_in_window, resample_scale, batch_size)

            # Update learning rate
            step_lr = lr * lr_scale(step, steps)
            for group in optimizer.param_groups:
                group["lr"] = step_lr

            # Get a batch of hidden activations from the model
            with t.inference_mode():
                h = self.generate_batch(batch_size)

            # Optimize
            loss_dict, loss, acts, _ = self.forward(h)
            loss.mean(0).sum().backward()
            optimizer.step()
            optimizer.zero_grad()

            # Normalize decoder weights by modifying them directly (if not using tied weights)
            if not self.cfg.tied_weights:
                self.W_dec.data = self.W_dec_normalized.data

            # Calculate the mean sparsities over batch dim for each feature
            frac_active = (acts.abs() > 1e-8).float().mean(0)
            frac_active_list.append(frac_active)

            # Display progress bar, and log a bunch of values for creating plots / animations
            if step % log_freq == 0 or (step + 1 == steps):
                progress_bar.set_postfix(
                    lr=step_lr,
                    loss=loss.mean(0).sum().item(),
                    frac_active=frac_active.mean().item(),
                    **{k: v.mean(0).sum().item() for k, v in loss_dict.items()},  # type: ignore
                )
                with t.inference_mode():
                    loss_dict, loss, acts, h_r = self.forward(h := self.generate_batch(hidden_sample_size))
                data_log.append(
                    {
                        "steps": step,
                        "frac_active": (acts.abs() > 1e-8).float().mean(0).detach().cpu(),
                        "loss": loss.detach().cpu(),
                        "h": h.detach().cpu(),
                        "h_r": h_r.detach().cpu(),
                        **{name: param.detach().cpu() for name, param in self.named_parameters()},
                        **{name: loss_term.detach().cpu() for name, loss_term in loss_dict.items()},
                    }
                )

        return data_log

    @t.no_grad()
    def resample_simple(
        self,
        frac_active_in_window: Float[Tensor, "window inst d_sae"],
        resample_scale: float,
    ) -> None:
        """
        Resamples dead latents, by modifying the model's weights and biases inplace.

        Resampling method is:
            - For each dead neuron, generate a random vector of size (d_in,), and normalize these vecs
            - Set new values of W_dec and W_enc to be these normalized vecs, at each dead neuron
            - Set b_enc to be zero, at each dead neuron
        """
        raise NotImplementedError()

    @t.no_grad()
    def resample_advanced(
        self,
        frac_active_in_window: Float[Tensor, "window inst d_sae"],
        resample_scale: float,
        batch_size: int,
    ) -> None:
        """
        Resamples latents that have been dead for `dead_feature_window` steps, according to `frac_active`.

        Resampling method is:
            - Compute the L2 reconstruction loss produced from the hidden state vecs `h`
            - Randomly choose values of `h` with probability proportional to their reconstruction loss
            - Set new values of W_dec & W_enc to be these centered & normalized vecs, at each dead neuron
            - Set b_enc to be zero, at each dead neuron
        """
        raise NotImplementedError()

### 연습 문제 - `__init__` 구현하기

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 5-15 minutes on this exercise.
> ```

아래의 `__init__` 메서드를 구현해야 합니다. 여기에서 가중치 `b_enc`, `b_dec`, `W_enc` 및 `_W_dec`을 정의해야 합니다. 가중치 초기화를 위해 [Kaiming uniform](https://pytorch.org/docs/stable/nn.init.html#torch.nn.init.kaiming_uniform_)을 사용하고, bias는 0으로 초기화하십시오.

참고로, tied weights 케이스를 처리하기 위해 `_W_dec`을 사용합니다. tied weights인 경우 `None`가 되어야 하며, 그렇지 않은 경우에는 적절한 parameter가 되어야 합니다. 위 클래스에서 제공된 `W_dec` 프로퍼티가 이 두 가지 케이스를 모두 처리해 줄 것입니다.

<details>
<summary>왜 가중치를 tie하고 싶거나, 혹은 하고 싶지 않을까요?</summary>

우리의 `Model` 구현에서는 가중치와 그 transpose를 사용했습니다. 직관적으로 encoder와 decoder의 latent 벡터 모두 "원래 모델의 hidden dimension에서의 어떤 feature의 방향"을 나타내도록 의도되었으므로, encoder와 decoder 가중치가 서로의 transposed copy가 되는 것이 타당하다고 생각할 수 있습니다.

가중치를 tie하고 싶지 않은 이유는 꽤 미묘합니다. encoder의 역할은 어떤 의미에서 superposition으로부터 feature를 복구하는 것이지만, decoder의 역할은 해당 feature가 존재할 때 이를 충실하게 표현하는 것뿐입니다 (우리의 SAE 목표는 입력을 `W_dec` 벡터들의 선형 결합으로 쓰는 것이기 때문입니다). 이것이 가중치가 untied일 때 일반적으로 decoder 가중치를 feature의 "진정한 방향"으로 보는 이유입니다.

아래 다이어그램이 이 개념을 이해하는 데 도움이 될 것입니다 (원하신다면 우리의 toy model 설정을 사용하여 이 다이어그램의 결과를 재현해 볼 수 있습니다!).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/w-dec-explained.png" width="700">

이 toy model과 같은 단순한 설정에서는 가중치를 untying함으로써 얻는 이득이 많지 않을 수 있으며, 오히려 가중치를 tying하는 것이 최적화 과정에서 성가신 local minima를 피하는 데 도움이 될 수 있습니다. 하지만 대부분의 연습 문제에서는 SAE 개념을 더 명확하게 설명하기 위해 untied weights를 사용할 것입니다.

</details>

또한, init 함수에서 `self.cfg`과 `self.model`를 정의해 두었음을 유의하십시오. 후자의 경우, 모델의 가중치를 freeze했습니다 (SAE를 훈련할 때 base model의 gradient를 추적하고 싶지 않기 때문입니다). 또한 모델의 가중치들이 모두 첫 번째 인스턴스와 일치하도록 수정했습니다 (이는 훈련을 마친 후 생성할 SAE plot을 더 쉽게 해석하기 위함입니다).

In [ ]:
# Go back up and edit your `ToySAE.__init__` method, then run the test below

tests.test_sae_init(ToySAE)

<details><summary>솔루션</summary>

```python
def __init__(self: ToySAE, cfg: ToySAEConfig, model: ToyModel) -> None:
    super(ToySAE, self).__init__()

    assert cfg.d_in == model.cfg.d_hidden, "Model's hidden dim doesn't match SAE input dim"
    self.cfg = cfg
    self.model = model.requires_grad_(False)
    self.model.W.data[1:] = self.model.W.data[0]
    self.model.b_final.data[1:] = self.model.b_final.data[0]

    self.W_enc = nn.Parameter(nn.init.kaiming_uniform_(t.empty((cfg.n_inst, cfg.d_in, cfg.d_sae))))
    self._W_dec = (
        None
        if self.cfg.tied_weights
        else nn.Parameter(nn.init.kaiming_uniform_(t.empty((cfg.n_inst, cfg.d_sae, cfg.d_in))))
    )
    self.b_enc = nn.Parameter(t.zeros(cfg.n_inst, cfg.d_sae))
    self.b_dec = nn.Parameter(t.zeros(cfg.n_inst, cfg.d_in))

    self.to(device)


ToySAE.__init__ = __init__
```
</details>

### 연습 문제 - `W_dec_normalized` 구현하기

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend 5-10 minutes on this exercise.
> ```

이제 decoder 가중치를 반환하는 `W_dec_normalized` property를 채워야 합니다. 이때 가중치는 autoencoder 입력 차원에 대해 정규화(L2 norm 사용)되어야 합니다. `W_dec` property가 존재한다는 것은 `_W_dec`에 대해 더 이상 걱정하지 않고 이 속성을 안전하게 참조할 수 있음을 의미합니다. 또한, 분모에 `cfg.weight_normalize_eps`를 더하는 것을 잊지 마십시오 (이는 0으로 나누는 오류를 방지하는 데 도움이 됩니다).

<details>
<summary>왜 <code>W_dec_normalized</code>가 필요한가요?</summary>

모델이 편법을 쓰는 것을 막기 위해 `W_dec`를 정규화합니다! 만약 `W_dec`를 정규화하지 않는다고 가정해 보겠습니다. 모델은 `W_enc`를 10배 작게 만들고, `W_dec`를 10배 크게 만들 수 있습니다. 출력값은 동일하겠지만(reconstruction error를 일정하게 유지하면서), latent activation은 10배 작아지게 되어 모델이 유용한 것을 학습하지 않고도 sparsity penalty(L1 loss 항)를 줄일 수 있게 됩니다.

`W_dec`의 열(column)들을 L2-정규화하면 latent activation의 크기를 더 명확하게 해석할 수 있습니다. 정규화를 통해, 이 값들은 "단위 길이의 각 feature가 얼마나 존재하는가?"라는 질문에 답하게 됩니다.

</details>

In [ ]:
# Go back up and edit your `ToySAE.W_dec_normalized` method, then run the test below

tests.test_sae_W_dec_normalized(ToySAE)

<details><summary>솔루션</summary>

```python
@property
def W_dec_normalized(self: ToySAE) -> Float[Tensor, "inst d_sae d_in"]:
    """Returns decoder weights, normalized over the autoencoder input dimension."""
    return self.W_dec / (self.W_dec.norm(dim=-1, keepdim=True) + self.cfg.weight_normalize_eps)


ToySAE.W_dec_normalized = W_dec_normalized
```
</details>

### 연습 문제 - `generate_batch` 구현하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend 5-15 minutes on this exercise.
> ```

앞서 언급했듯이, 이제 데이터는 더 이상 `ToyModel.generate_batch`에서 직접 오지 않습니다. 대신, `Model.generate_batch`을 사용하여 모델 입력 $x$를 가져온 다음, 모델의 `W` 행렬을 적용하여 hidden activation $h=Wx$을 얻습니다. 우리는 "Superposition in a Nonprivileged Basis" 모델을 사용하고 있으므로, $h$을 얻기 위해 적용할 ReLU 함수가 없다는 점에 유의하십시오.

이제 `generate_batch` 메서드를 채운 다음 테스트를 실행하십시오. 주의 사항으로, `model` 대신 `self.model`를 사용해야 함을 기억하십시오!

In [ ]:
# Go back up and edit your `ToySAE.generate_batch` method, then run the test below

tests.test_sae_generate_batch(ToySAE)

<details><summary>솔루션</summary>

```python
def generate_batch(self: ToySAE, batch_size: int) -> Float[Tensor, "batch inst d_in"]:
    """
    Generates a batch of hidden activations from our model.
    """
    return einops.einsum(
        self.model.generate_batch(batch_size),
        self.model.W,
        "batch inst feats, inst d_in feats -> batch inst d_in",
    )


ToySAE.generate_batch = generate_batch
```
</details>

### 연습 문제 - `forward` 구현하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵🔵
> 
> You should spend up to 25-40 minutes on this exercise.
> ```

autoencoder의 hidden state activation을 $z = \operatorname{ReLU}(W_{enc}(h - b_{dec}) + b_{enc})$로 계산한 다음, 출력을 $h' = W_{dec}z + b_{dec}$으로 재구성해야 합니다. 몇 가지 참고 사항은 다음과 같습니다:

- 반환하는 **첫 번째 변수**는 `loss_dict`이며, 손실 함수 내 두 항에 대한 `(batch_size, n_inst)` 형태의 loss tensor를 포함합니다 (L1 계수를 곱하기 전 단계입니다). 이는 로깅에 사용되며, 나중에 neuron resampling 방법에서도 사용됩니다. 이 아키텍처에서 key는 `"L_reconstruction"` 및 `"L_sparsity"`이어야 합니다.
- 반환하는 **두 번째 변수**는 `loss` 항이며, 이 또한 `(batch_size, n_inst)` 형태를 가집니다. 이는 `loss_dict`의 loss들을 합산하여 생성됩니다 (sparsity loss에는 `cfg.sparsity_coeff`을 곱합니다). gradient descent를 수행할 때, batch 차원에 대해서는 평균을 내고 instance 차원에 대해서는 합산합니다 (instance들을 독립적으로 그리고 병렬로 학습시키기 때문입니다).
- 반환하는 **세 번째 변수**는 hidden state activation인 `acts`이며, 이는 나중에 neuron resampling(및 활성화된 latent의 개수 로깅)에 사용됩니다.
- 반환하는 **네 번째 변수**는 재구성된 hidden state인 `h_reconstructed`, 즉 autoencoder의 실제 출력입니다.

손실 항과 관련하여 중요한 참고 사항이 있습니다. reconstruction loss는 입력과 출력 사이의 제곱 차이를 `d_in` 차원에 대해 **평균(average)** 낸 것이지만, sparsity penalty는 hidden activation의 L1 norm을 `d_sae` 차원에 대해 **합산(sum)** 한 것입니다. 왜 하나는 평균을 내고 다른 하나는 합산하는지 이해하시겠습니까?

<details>
<summary>힌트</summary>

L1 loss 또한 평균을 냈다고 가정해 봅시다. 단일 latent가 reconstruction loss와 sparsity penalty로부터 받는 gradient를 생각했을 때, `d_sae`가 매우 커지는 극한 상황에서 이들은 어떤 모습일까요?

</details>

<details>
<summary>정답 - 왜 L2 loss는 <code>d_in</code>에 대해 평균을 내고, L1 loss는 <code>d_sae</code>에 대해 합산하는가</summary>

논의를 위해 L1 loss 또한 평균을 냈다고 가정해 보겠습니다. latent 차원을 두 배로 늘리되, 다른 모든 SAE 하이퍼파라미터는 동일하게 유지한다고 상상해 보십시오. reconstruction loss로 인한 hidden unit당 gradient는 여전히 동일할 것입니다 (단일 hidden unit의 encoder 또는 decoder 벡터를 변경하는 것이 출력에 미치는 영향은 이전과 같기 때문입니다). 하지만 sparsity penalty로 인한 hidden unit당 gradient는 절반으로 줄어들 것입니다 (sparsity penalty를 `d_sae`에 대해 평균 내기 때문입니다). 이는 극한 상황에서 sparsity penalty가 전혀 중요하지 않게 되며, 오직 reconstruction loss를 0으로 만드는 것만이 중요해진다는 것을 의미합니다.

</details>

참고 - forward 함수에서 `self.W_dec` 대신 `self.W_dec_normalized`을 사용하고 있는지 확인하십시오. tied weights를 사용하는 경우 `W_dec`를 inplace로 직접 정규화할 수 없지만, 여전히 정규화된 버전을 사용하고 싶기 때문입니다.

In [ ]:
# Go back up and edit your `ToySAE.forward` method, then run the test below

tests.test_sae_forward(ToySAE)

<details><summary>솔루션</summary>

```python
def forward(
    self: ToySAE, h: Float[Tensor, "batch inst d_in"]
) -> tuple[
    dict[str, Float[Tensor, "batch inst"]],
    Float[Tensor, "batch inst"],
    Float[Tensor, "batch inst d_sae"],
    Float[Tensor, "batch inst d_in"],
]:
    """
    Forward pass on the autoencoder.

    Args:
        h: hidden layer activations of model

    Returns:
        loss_dict:       dict of different loss terms, each dict value having shape (batch_size, n_inst)
        loss:            total loss (i.e. sum over terms of loss dict), same shape as loss_dict values
        acts_post:       autoencoder latent activations, after applying ReLU
        h_reconstructed: reconstructed autoencoder input
    """
    h_cent = h - self.b_dec

    # Compute latent (hidden layer) activations
    acts_pre = einops.einsum(h_cent, self.W_enc, "batch inst d_in, inst d_in d_sae -> batch inst d_sae") + self.b_enc
    acts_post = F.relu(acts_pre)

    # Compute reconstructed input
    h_reconstructed = (
        einops.einsum(acts_post, self.W_dec_normalized, "batch inst d_sae, inst d_sae d_in -> batch inst d_in")
        + self.b_dec
    )

    # Compute loss terms
    L_reconstruction = (h_reconstructed - h).pow(2).mean(-1)
    L_sparsity = acts_post.abs().sum(-1)
    loss_dict = {"L_reconstruction": L_reconstruction, "L_sparsity": L_sparsity}
    loss = L_reconstruction + self.cfg.sparsity_coeff * L_sparsity

    return loss_dict, loss, acts_post, h_reconstructed


ToySAE.forward = forward
```
</details>

## SAE 학습하기

`optimize` 메서드가 제공되었습니다. 이전 모델과 어떻게 다른지에 대한 몇 가지 참고 사항입니다:

- 각 최적화 단계 전에 **neuron resampling**을 구현합니다. 이에 대해서는 나중에 자세히 다루겠습니다.
- `data_log` 딕셔너리를 통해 더 많은 로깅을 수행하며, 이는 시각화에 사용됩니다.
- [Anthropic's Feb 2024 update](https://transformer-circuits.pub/2024/feb-update/index.html#dict-learning-loss)의 설명과 일치시키기 위해 `betas=(0.0, 0.999)`를 사용했습니다. 문서에는 특히 큰 모델에서 더 잘 작동한다고 되어 있지만, 여기에서도 이를 맞추는 것이 좋습니다.

먼저, 모델을 정의하고 학습시킨 후, 모델 가중치와 `sae.generate_batch`에서 반환된 데이터(학습된 모델의 hidden state representation이며, SAE 학습에 사용됩니다)를 시각화해 보겠습니다.

이후의 모든 실습에서는 feature 확률을 2.5%로 사용하며(feature 간의 독립성을 가정함), 이를 적용할 것임을 유의하시기 바랍니다.

In [ ]:
d_hidden = d_in = 2
n_features = d_sae = 5
n_inst = 16

# Create a toy model, and train it to convergence
cfg = ToyModelConfig(n_inst=n_inst, n_features=n_features, d_hidden=d_hidden)
model = ToyModel(cfg=cfg, device=device, feature_probability=0.025)
model.optimize()

sae = ToySAE(cfg=ToySAEConfig(n_inst=n_inst, d_in=d_in, d_sae=d_sae), model=model)

h = sae.generate_batch(512)

utils.plot_features_in_2d(model.W[:8], title="Base model")
utils.plot_features_in_2d(
    einops.rearrange(h[:, :8], "batch inst d_in -> inst d_in batch"),
    title="Hidden state representation of a random batch of data",
)

이제 SAE를 학습시키고, loss가 가장 낮은 인스턴스들을 시각화해 보겠습니다! 또한 시간에 따른 학습 과정을 애니메이션으로 생성하는 함수 `animate_features_in_2d` 를 만들었습니다. 만약 인라인 표시가 작동하지 않는다면, 브라우저에서 저장된 HTML 파일을 열어 확인해야 할 수도 있습니다.

In [ ]:
data_log = sae.optimize(steps=20_000)

utils.animate_features_in_2d(
    data_log,
    instances=list(range(8)),  # only plot the first 8 instances
    rows=["W_enc", "_W_dec"],
    filename=str(section_dir / "animation-training.html"),
    title="SAE on toy model",
)

# If this display code doesn't work, try saving & opening animation in your browser
with open(section_dir / "animation-training.html") as f:
    display(HTML(f.read()))

다시 말해, autoencoder는 일반적으로 모델의 hidden state를 재구성하는 데 성공하며, 때로는 완전히 monosemantic한 솔루션(feature당 하나의 latent)을 학습하기도 하지만, 더 자주 **polysemantic latent**와 **dead latent**(결코 activation되지 않는 latent)의 조합을 학습합니다. 이는 학습 중에 gradient를 전혀 받지 못하기 때문에 시간이 지난다고 해서 스스로 해결되는 문제가 아니며, 매우 큰 문제가 됩니다. 아래 코드에서 학습 과정에 따른 feature 확률을 그래프로 그려 dead latent의 존재 여부를 확인할 수 있습니다. 다음과 같은 점을 발견하실 수 있을 것입니다:

1. 일부 latent는 학습 기간의 대부분 또는 전체 동안 dead 상태입니다 ("fraction of datapoints active"가 0인 경우).
2. 일부 latent는 목표 feature 확률인 2.5%보다 더 빈번하게 fire합니다 (이들은 보통 polysemantic하며, 즉 하나 이상의 서로 다른 feature에서 fire합니다).
3. 일부 latent는 목표 확률과 거의 비슷하거나 약간 낮은 수준으로 fire합니다 (이들은 보통 monosemantic합니다). 만약 위의 사례 중 어떤 것이 완전한 monosemantic 솔루션(즉, 2D hidden dimension 주변에 latent들이 균일하게 배치된 경우)을 학습했다면, 해당 사례의 5개 latent 모두가 이 세 번째 범주에 속하는 것을 확인할 수 있을 것입니다.

In [ ]:
utils.frac_active_line_plot(
    frac_active=t.stack([data["frac_active"] for data in data_log]),
    title="Probability of sae features being active during training",
    avg_window=20,
)

## Resampling

Anthropic의 논문에서 발췌한 내용입니다 (우리가 사용하는 용어에 맞춰 "dead neurons"를 "dead latents"로 변경하였습니다):

> 둘째로, 우리는 학습 과정에서 일부 latents가 매우 많은 수의 데이터 포인트에 대해서도 더 이상 activate되지 않는다는 것을 발견했습니다. 우리는 학습 중에 이러한 dead latents를 "resampling" 하는 것이, 주어진 autoencoder hidden layer dimension에 대해 모델이 더 많은 feature를 표현할 수 있게 함으로써 더 나은 결과를 낸다는 것을 발견했습니다. 우리의 resampling 절차는 [Autoencoder Resampling](https://transformer-circuits.pub/2023/monosemantic-features/index.html#appendix-autoencoder-resampling)에 자세히 설명되어 있지만, 간단히 말하면 주기적으로 상당한 단계 동안 firing되지 않은 latents를 확인하고, dead latents의 encoder weights를 autoencoder가 현재 잘 표현하지 못하는 데이터 포인트에 맞게 리셋합니다.

다음 작업은 이 resampling 절차를 구현하는 것입니다.

### 연습 문제 - `resample_simple` 구현하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 20-30 minutes on this exercise.
> ```

Anthropic이 설명하는 SAE latent 리샘플링 과정은 상당히 복잡하므로, 우선 더 간단한 버전을 구현해 보겠습니다. 구체적으로, 각 인스턴스 `inst` 에 대해 다음 알고리즘을 구현합니다:

* 모든 dead latent를 찾습니다 (즉, `frac_active_in_window[:, inst, d]` 가 모두 0인 값 `(inst, d)` ).
* 이들 각각에 대해 다음을 수행합니다:
    * 길이 `d_in` 의 새로운 랜덤 벡터 `v` 를 생성합니다.
    * decoder weights `W_dec[inst, d, :]` 를 정규화된 이 새로운 벡터 `v` 로 설정합니다.
    * encoder weights `W_enc[inst, :, d]` 를 norm이 `resample_scale` 이 되도록 스케일링된 이 새로운 벡터 `v` 로 설정합니다.
    * encoder biases `b_enc[inst, d]` 를 0으로 설정합니다.

제공된 테스트 함수는 여러분의 함수가 올바른 weight를 교체하거나 0으로 만드는지 확인합니다.

In [ ]:
# Go back up and edit your `ToySAE.resample_simple` method, then run the test below

tests.test_resample_simple(ToySAE)

<details><summary>솔루션</summary>

```python
@t.no_grad()
def resample_simple(
    self: ToySAE,
    frac_active_in_window: Float[Tensor, "window inst d_sae"],
    resample_scale: float,
) -> None:
    """
    Resamples dead latents, by modifying the model's weights and biases inplace.

    Resampling method is:
        - For each dead neuron, generate a random vector of size (d_in,), and normalize these vecs
        - Set new values of W_dec and W_enc to be these normalized vectors, at each dead neuron
        - Set b_enc to be zero, at each dead neuron

    This function performs resampling over all instances at once, using batched operations.
    """
    # Get a tensor of dead latents
    dead_latents_mask = (frac_active_in_window < 1e-8).all(dim=0)  # [instances d_sae]
    n_dead = int(dead_latents_mask.int().sum().item())

    # Get our random replacement values of shape [n_dead d_in], and scale them
    replacement_values = t.randn((n_dead, self.cfg.d_in), device=self.W_enc.device)
    replacement_values_normed = replacement_values / (
        replacement_values.norm(dim=-1, keepdim=True) + self.cfg.weight_normalize_eps
    )

    # Change the corresponding values in W_enc, W_dec, and b_enc
    self.W_enc.data.transpose(-1, -2)[dead_latents_mask] = resample_scale * replacement_values_normed
    self.W_dec.data[dead_latents_mask] = replacement_values_normed
    self.b_enc.data[dead_latents_mask] = 0.0


ToySAE.resample_simple = resample_simple
```
</details>

테스트를 통과했다면, 모델을 다시 학습시키고 애니메이션을 통해 neuron resampling이 학습 과정에 어떻게 도움이 되었는지 확인하십시오. resampled 된 neuron들은 빨간색으로 표시되는 것을 볼 수 있습니다.

In [ ]:
resampling_sae = ToySAE(cfg=ToySAEConfig(n_inst=n_inst, d_in=d_in, d_sae=d_sae), model=model)

resampling_data_log = resampling_sae.optimize(steps=20_000, resample_method="simple")

utils.animate_features_in_2d(
    resampling_data_log,
    rows=["W_enc", "_W_dec"],
    instances=list(range(8)),  # only plot the first 8 instances
    filename=str(section_dir / "animation-training-resampling.html"),
    color_resampled_latents=True,
    title="SAE on toy model (with resampling)",
)

utils.frac_active_line_plot(
    frac_active=t.stack([data["frac_active"] for data in resampling_data_log]),
    title="Probability of sae features being active during training",
    avg_window=20,
)

훨씬 좋아졌습니다!

이제 feature들에 대해 거의 완전한 reconstruction을 달성했으므로, 그 reconstruction을 시각화해 보겠습니다! `animate_features_in_2d` 함수는 hidden state reconstruction과 그것이 시간에 따라 어떻게 변화하는지를 plot하는 기능도 제공합니다. hidden state reconstruction이 시간에 따라 어떻게 진화하는지 살펴보면, 예를 들어 다음과 같은 상황들을 이해하는 데 도움이 됩니다:

- SAE는 이상적인 솔루션으로 수렴하기 전에 종종 non-sparse한 솔루션(예: 균일하게 배치된 4개의 polysemantic latent와 1개의 dead latent)을 학습합니다.
    - 참고로, LLM에서 SAE를 학습시킬 때도 유사한 현상이 관찰됩니다. 먼저 reconstruction loss가 작은 non-sparse한 솔루션을 찾은 다음, 더 sparse한 솔루션을 학습합니다(L0가 감소합니다).
- hidden state 위에 마우스를 올리면 다음과 같은 점들을 관찰할 수 있습니다:
    - magnitude가 낮은 hidden state는 종종 0으로 reconstruction됩니다. 이는 SAE가 다른 feature들로부터 오는 interference로부터 이를 분리해낼 수 없기 때문입니다.
    - 올바르게 reconstruction된 feature라 하더라도, hidden state의 magnitude는 일반적으로 실제 hidden state보다 작습니다. 이를 **shrinkage**라고 하며, 다음 섹션에서 자세히 다루겠습니다.

In [ ]:
utils.animate_features_in_2d(
    resampling_data_log,
    rows=["W_enc", "h", "h_r"],
    instances=list(range(4)),  # plotting fewer instances for a smaller animation file size
    color_resampled_latents=True,
    filename=str(section_dir / "animation-training-reconstructions.html"),
    title="SAE on toy model (showing hidden states & reconstructions)",
)

### 연습 문제 - `resample_advanced` 구현하기


> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Importance: 🔵🔵⚪⚪⚪
> 
> You should spend up to 20-40 minutes on this exercise, if you choose to do it.
> ```

위에서 더 간단한 버전의 `resample`를 이미 구현했다면 이 섹션은 선택 사항으로 간주해도 좋습니다. 하지만 [Anthropic's methodology](https://transformer-circuits.pub/2023/monosemantic-features/index.html#appendix-autoencoder-resampling)에 더 가까운 버전에 관심이 있다면, 이 연습 문제가 유용할 것입니다.

주요 차이점은 리샘플링된 값들을 선택하는 방식에 있습니다. 단순히 분포에서 무작위로 추출하여 정규화하는 대신, **입력 activation 세트 $h$에서 복원 추출(sampling with replacement)을 수행하며, 이때 샘플링 확률은 각 입력에 대한 autoencoder의 squared $L_2$ loss에 의해 가중치를 둡니다**. 직관적으로, 이는 리샘플링된 neuron이 현재 autoencoder가 제대로 표현하지 못하고 있는 feature 방향을 나타낼 가능성을 더 높여줍니다.

새로운 리샘플링 알고리즘은 다음과 같습니다. 각 instance에 대해 다음을 수행합니다:

* SAE로부터 hidden data 배치 `h`를 생성하고 squared reconstruction loss `l2_squared`를 계산합니다. 이는 `(batch_size, n_inst)` 형태여야 합니다. 만약 이 instance `l2_squared[:, inst]`에 대한 L2 loss가 모든 곳에서 0이라면, 이 instance는 건너뛸 수 있습니다.
* 이 instance에 대한 dead latent를 찾습니다 (즉, `frac_active_in_window[:, inst, d]`가 모두 0인 instance `inst`와 latent index `d`).
* 이들 각각에 대해 다음을 수행합니다:
    * `l2_squared[:, inst]`에 비례하는 확률 분포에 따라 `0 <= x < batch_size`를 선택하여 벡터 `v = h[x, inst, :]`를 무작위로 샘플링합니다.
    * decoder weight `W_dec[inst, d, :]`를 정규화된 이 새로운 벡터 `v`로 설정합니다.
    * encoder weight `W_enc[inst, :, d]`를 norm이 `resample_scale * avg_W_enc_alive_norm`가 되도록 스케일링된 이 새로운 벡터 `v`로 설정합니다 (여기서 `avg_W_enc_alive_norm`은 해당 instance에서 살아있는 neuron들의 encoder weight의 평균 norm입니다).
    * encoder bias `b_enc[inst, d]`를 0으로 설정합니다.

결과적으로 변경 사항은 단 두 가지입니다: encoder weight를 위해 `avg_W_enc_alive_norm`를 추가로 사용한 것과, 벡터 `v`를 얻기 위해 L2 기반 분포에서 샘플링하는 것입니다. 이 함수는 다소 복잡해질 수 있으므로, 모든 instance를 한꺼번에 리샘플링하려 하기보다 instance를 하나씩 순회하며 처리하는 것을 권장합니다.

샘플링의 경우, `torch.distributions.categorical.Categorical`를 사용하여 확률 분포를 정의하고, `sample` 메서드를 사용하여 샘플링하는 것을 권장합니다. 아래에 이 함수를 사용하는 방법에 대한 예시를 포함했습니다.

<details>
<summary><code>Categorical</code> 사용 예시</summary>

```python
from torch.distributions.categorical import Categorical

# Define a prob distn over (0, 1, 2, 3, 4) with probs proportional to (4, 3, 2, 1, 0)
values = t.arange(5).flip(0)
probs = values.float() / values.sum()
distribution = Categorical(probs = probs)

# Sample a single value from it
distribution.sample() # tensor(1)

# Sample multiple values with replacement (values will mostly be in the lower end of the range)
distribution.sample((10,)) # tensor([1, 1, 3, 0, 0, 1, 0, 3, 2, 2])
```

여러 번 샘플링할 때는 스칼라 대신 1D tensor를 전달해야 합니다.

</details>

이 리샘플링 메서드를 구현한 후, 테스트를 실행하십시오:

In [ ]:
# Go back up and edit your `ToySAE.resample_advanced` method, then run the test below

tests.test_resample_advanced(ToySAE)

<details><summary>솔루션</summary>

```python
@t.no_grad()
def resample_advanced(
    self: ToySAE,
    frac_active_in_window: Float[Tensor, "window inst d_sae"],
    resample_scale: float,
    batch_size: int,
) -> None:
    """
    Resamples latents that have been dead for 'dead_feature_window' steps, according to `frac_active`.

    Resampling method is:
        - Compute the L2 reconstruction loss produced from the hidden state vectors `h`
        - Randomly choose values of `h` with probability proportional to their reconstruction loss
        - Set new values of W_dec & W_enc to be these centered & normalized vecs, at each dead neuron
        - Set b_enc to be zero, at each dead neuron

    Returns colors and titles (useful for creating the animation: resampled neurons appear in red).
    """
    h = self.generate_batch(batch_size)
    l2_loss = self.forward(h)[0]["L_reconstruction"]

    for instance in range(self.cfg.n_inst):
        # Find the dead latents in this instance. If all latents are alive, continue
        is_dead = (frac_active_in_window[:, instance] < 1e-8).all(dim=0)
        dead_latents = t.nonzero(is_dead).squeeze(-1)
        n_dead = dead_latents.numel()
        if n_dead == 0:
            continue  # If we have no dead features, then we don't need to resample

        # Compute L2 loss for each element in the batch
        l2_loss_instance = l2_loss[:, instance]  # [batch_size]
        if l2_loss_instance.max() < 1e-6:
            continue  # If we have zero reconstruction loss, we don't need to resample

        # Draw `d_sae` samples from [0, 1, ..., batch_size-1], with probabilities proportional to
        # the values of l2_loss
        distn = Categorical(probs=l2_loss_instance.pow(2) / l2_loss_instance.pow(2).sum())
        replacement_indices = distn.sample((n_dead,))  # type: ignore

        # Index into the batch of hidden activations to get our replacement values
        replacement_values = (h - self.b_dec)[replacement_indices, instance]  # [n_dead d_in]
        replacement_values_normalized = replacement_values / (
            replacement_values.norm(dim=-1, keepdim=True) + self.cfg.weight_normalize_eps
        )

        # Get the norm of alive neurons (or 1.0 if there are no alive neurons)
        W_enc_norm_alive_mean = self.W_enc[instance, :, ~is_dead].norm(dim=0).mean().item() if (~is_dead).any() else 1.0

        # Lastly, set the new weights & biases (W_dec is normalized, W_enc needs specific scaling,
        # b_enc is zero)
        self.W_dec.data[instance, dead_latents, :] = replacement_values_normalized
        self.W_enc.data[instance, :, dead_latents] = (
            replacement_values_normalized.T * W_enc_norm_alive_mean * resample_scale
        )
        self.b_enc.data[instance, dead_latents] = 0.0


ToySAE.resample_advanced = resample_advanced
```
</details>

테스트를 통과한 후, SAE를 다시 학습시키고 시각화해 볼 수 있습니다. 2차원에서는 이 resampling 방법으로 큰 개선을 발견하지 못할 수도 있지만, 훨씬 더 고차원인 공간에서는 뉴런을 더 타겟팅된 방식으로 resample 하는 것이 매우 유익합니다.

## Gated & JumpReLU SAEs

이 섹션들에서는 표준 모델보다 성능 향상을 제공하는 것으로 보이는 두 가지 대안적인 SAE 아키텍처에 대해 논의하겠습니다. 두 아키텍처 모두 유사한 직관을 가지고 있으며 (특정 가정 하에서는 실제로 수학적으로 거의 동일합니다), JumpReLU로 넘어가기 전에 Gated SAE에 먼저 집중하겠습니다. 이것이 반드시 Gated SAE가 개념적으로 더 간단하기 때문은 아니며 (JumpReLU가 더 간단하다는 주장도 있습니다), 단지 학습시키기가 더 쉽기 때문입니다. 하지만 이 섹션을 진행하는 동안 두 아키텍처 모두 중요하고 효과적이며, 한 쪽의 직관이 다른 쪽으로 이어지는 경우가 많다는 점을 기억해 두는 것이 좋습니다.

### Gated SAEs

현재 많은 다양한 SAE architecture 변형들이 탐구되고 있습니다. 특히 흥미로운 것 중 하나는 [DeepMind](https://arxiv.org/pdf/2404.16014)의 논문에서 자세히 설명된 **Gated SAE**입니다. 두 가지 관찰 결과를 통해 이 architecture의 필요성을 설명할 수 있습니다.

1. **경험적으로, feature들은 보통 binary 형태가 되려는 경향이 있습니다.** 예를 들어, "이것이 농구에 관한 내용인가"와 같은 feature들은 0에서 1 사이의 연속적인 범위를 갖기보다 "off" 또는 "on"으로 생각하는 것이 더 적절합니다. 실제로 정확한 계수를 재구성하는 것은 중요하며, 이는 특정 feature가 존재한다는 모델의 신뢰도 등을 나타내는 데 중요한 역할을 하는 것으로 보입니다. 하지만 여전히, 우리는 이러한 불연속성을 학습할 수 있는 architecture를 이상적으로 원합니다.

한 가지 쉬운 옵션은 SAE의 hidden layer에 **Jump ReLU**와 같은 불연속 activation function을 사용하는 것입니다. 이 activation은 특정 값 $\theta$에서 jump가 발생하며, 이를 통해 이러한 nonlinearity를 표현할 수 있습니다.

<img src="https://res.cloudinary.com/lesswrong-2-0/image/upload/f_auto,q_auto/v1/mirroredImages/wZqqQysfLrt2CFx4T/zzrdot3xexvcz3mqghn8" width="240">

하지만 Jump ReLU만으로는 해결할 수 없는 또 다른 문제가 있습니다:

2. **SAE는 [shrinkage](https://www.alignmentforum.org/posts/3JuSjTZyMzaSeTxKk/addressing-feature-suppression-in-saes) 문제를 겪습니다.** 우리가 실제로 원하는 목적은 hidden layer의 L0 "norm"(0이 아닌 요소의 수)이 작은 것이며, 이를 위해 L1 norm을 대리 지표로 사용한다는 점을 상기하십시오. SAE loss function의 두 loss 항은 서로 충돌하는 목표를 가집니다. reconstruction 항은 autoencoder가 입력을 잘 재구성하도록 만들려 하고, sparsity 항은 hidden layer의 크기를 줄이려 합니다. 이는 단 하나의 hidden unit만 활성화되어도 완벽한 reconstruction이 가능한 상황에서조차, sparsity loss가 이 hidden unit의 크기를 0으로 편향시켜 reconstruction 성능을 떨어뜨린다는 것을 의미합니다.

**_참고로, JumpReLU 단독으로는 shrinkage 문제를 해결하지 못하지만, JumpReLU에 L0 penalty <u>를 더하면 </u> shrinkage 문제를 해결할 수 있습니다. 이에 대해서는 챕터 뒷부분에서 논의하겠습니다._**

여기서 **Gated SAEs**가 등장합니다. Gated SAE는 불연속성을 적용하는 Heaviside 항을 두고, 이 항을 magnitude 항과 분리함으로써 두 가지 문제를 모두 해결하는 것으로 보입니다. SAE activation을 계산하는 표준 함수 대신:

$$
\mathbf{f}(\mathbf{x}):=\operatorname{ReLU}\left(\mathbf{W}_{\mathrm{enc}}\left(\mathbf{x}-\mathbf{b}_{\mathrm{dec}}\right)+\mathbf{b}_{\mathrm{enc}}\right)
$$

다음과 같은 함수를 사용합니다:

$$
\tilde{\mathbf{f}}(\mathbf{x}):=\underbrace{\mathbf{1} [\overbrace{\mathbf{W}_{\text {gate }}\left(\mathbf{x}-\mathbf{b}_{\text {dec }}\right)+\mathbf{b}_{\text {gate }}}^{\pi_{\text {gate }}(\mathbf{x})}>0]}_{\mathbf{f}_{\text {gate }}(\mathbf{x})} \odot \underbrace{\operatorname{ReLU}\left(\mathbf{W}_{\text {mag }}\left(\mathbf{x}-\mathbf{b}_{\text {dec }}\right)+\mathbf{b}_{\text {mag }}\right)}_{\mathbf{f}_{\text {mag }}(\mathbf{x})}
$$
여기서 $\mathbf{1}[\cdot > 0]$은 pointwise Heaviside step function이며, $\odot$는 elementwise multiplication입니다. feature의 gate와 activation magnitude는 weight matrix인 $W_{\text{mag}}$과 $W_{\text{gate}}$에 의해 계산됩니다. 흥미롭게도, gated weight와 magnitude weight를 $\left(\mathbf{W}_{\text {mag }}\right)_{i j}:=\left(\exp \left(\mathbf{r}_{\text {mag }}\right)\right)_i \cdot\left(\mathbf{W}_{\text {gate }}\right)_{i j}$와 같이 묶는다면, 이것이 기본적으로 파라미터화된 threshold 값 $\theta$을 가진 Jump ReLU activation function과 동일함을 보일 수 있습니다 (독자 여러분을 위한 연습 문제로 남겨둡니다!).

이 SAE를 어떻게 학습시킬 수 있을지 궁금하실 것입니다. 이상적으로는 activation이 0이 될지 여부를 결정하는 항인 $f_{\text{gate}}(\mathbf{x})$에 sparsity penalty를 부여하고 싶을 것입니다. 불행히도 Heaviside function은 불연속적이기 때문에 gradient가 전파되지 않아 그렇게 할 수 없습니다. 대신, 우리는 preactivation $\pi_{\text {gate }}(\mathbf{x})$에 sparsity penalty를 적용합니다. 따라서 loss function은 다음과 같습니다:

$$
\mathcal{L}_{\text {gated }}(\mathbf{x}):=\underbrace{\|\mathbf{x}-\hat{\mathbf{x}}(\tilde{\mathbf{f}}(\mathbf{x}))\|_2^2}_{\mathcal{L}_{\text {reconstruct }}}+\underbrace{\lambda\left\|\operatorname{ReLU}\left(\boldsymbol{\pi}_{\text {gate }}(\mathbf{x})\right)\right\|_1}_{\mathcal{L}_{\text {sparsity }}}
$$

하지만 여기서 문제가 발생합니다. preactivation 값 $\pi_{\text {gate }}(\mathbf{x})$이 양수인 한, 이 값을 줄이면 reconstruction loss를 변경하지 않고도 sparsity penalty를 줄일 수 있습니다 (reconstruction에서 중요한 것은 preactivation 값이 양수인지 음수인지 여부뿐이기 때문입니다). 결국 이 값들은 0에 도달하게 되고, 더 이상 gradient를 받지 못하게 됩니다 (그 시점부터 모델의 출력은 항상 0이 되기 때문입니다). 이를 해결하기 위해, 실제 latent activation 대신 preactivation 값 $\pi_{\text {gate }}(\mathbf{x})$을 사용했을 때의 reconstruction loss와 동일한 auxiliary loss 항을 추가합니다. 이는 preactivation을 위로 밀어 올리는 gradient를 추가하여, 값을 0으로만 밀어 내리려는 sparsity loss function을 상쇄합니다. 이제 최종 loss function은 다음과 같습니다:

$$
\mathcal{L}_{\text {gated }}(\mathbf{x}):=\underbrace{\|\mathbf{x}-\hat{\mathbf{x}}(\tilde{\mathbf{f}}(\mathbf{x}))\|_2^2}_{\mathcal{L}_{\text {reconstruct }}}+\underbrace{\lambda\left\|\operatorname{ReLU}\left(\boldsymbol{\pi}_{\text {gate }}(\mathbf{x})\right)\right\|_1}_{\mathcal{L}_{\text {sparsity }}}+\underbrace{\left\|\mathbf{x}-\hat{\mathbf{x}}_{\text {frozen }}\left(\operatorname{ReLU}\left(\boldsymbol{\pi}_{\text {gate }}(\mathbf{x})\right)\right)\right\|_2^2}_{\mathcal{L}_{\text {aux }}}
$$

### 연습 문제 - Gated SAE 구현하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 60 minutes on this exercise.
> ```

이제 Gated SAE를 구현하고 표준 모델과 비교하는 데 필요한 모든 정보를 갖추었습니다. 아래에 `GatedToySAE` 클래스를 제공하며, 위 설명에 따라 `ToySAE` 메서드들의 수정된 버전을 구현해야 합니다.

*참고 - 이를 구현하는 또 다른 방법은 `ToySAE` 클래스를 수정하여 gated 및 표준 아키텍처를 모두 지원하도록 만드는 것입니다. 예를 들어, SAE config 클래스에 `architecture` 인자를 도입하는 방식입니다. 좋은 연습이 될 것이라고 생각하신다면 보너스 연습 문제로 시도해 보시기 바랍니다!*

몇 가지 팁입니다:

- forward pass와 loss function의 경우, [DeepMind paper](https://arxiv.org/pdf/2404.16014)의 34페이지에 있는 Appendix G를 참고하시기 바랍니다. 가장 쉬운 방법이므로 해당 부록에서 사용된 명명 규칙을 따르는 것을 권장합니다.
- Gated 아키텍처를 사용하는 경우, 서로 다른 가중치를 생성하고 *다시 샘플링(resample)* 해야 함을 기억하십시오. 예를 들어, Gated 방식이라면 모든 dead latent에서 `b_mag`, `b_gate`, `r_mag`을 0으로 만들어야 합니다.
- 기본적으로 gate 가중치와 magnitude 가중치를 논문에서처럼 $\left(\mathbf{W}_{\text {mag }}\right)_{i j}:=\exp \left(\mathbf{r}_{\text {mag }}\right)_i \times \left(\mathbf{W}_{\text {gate }}\right)_{i j}$와 같이 묶는(tie) 것을 권장합니다. 이러한 방식의 tying은 encoder와 decoder 가중치를 묶는 것보다 훨씬 덜 부자연스럽습니다. 만약 가중치를 *추가로* 묶는다면, 이를 $W_{\text{dec}} = W_{\text{gate}}$로 해석할 수 있습니다.

<details>
<summary>도움말 - 이 weight tying을 어떻게 구현해야 할지 모르겠습니다.</summary>

다음과 같이 property를 사용하는 것을 권장합니다:

```python
@property
def W_mag(self) -> Float[Tensor, "inst d_in d_sae"]:
    assert self.cfg.architecture == "gated", "W_mag only available for gated model"
    return self.r_mag.exp().unsqueeze(1) * self.W_gate
```

이렇게 하면 `r_mag`와 `W_gate`만 정의하면 됩니다. 주의할 점은, `W_mag`의 값을 직접 설정할 수 없으므로 resampling 시 주의해야 한다는 것입니다.

</details>

In [ ]:
class GatedToySAE(ToySAE):
    W_gate: Float[Tensor, "inst d_in d_sae"]
    b_gate: Float[Tensor, "inst d_sae"]
    r_mag: Float[Tensor, "inst d_sae"]
    b_mag: Float[Tensor, "inst d_sae"]
    _W_dec: Float[Tensor, "inst d_sae d_in"] | None
    b_dec: Float[Tensor, "inst d_in"]

    def __init__(self, cfg: ToySAEConfig, model: ToyModel):
        super(ToySAE, self).__init__()

        # YOUR CODE HERE - initialize the Gated model's weights & biases
        raise NotImplementedError()

        self.to(device)

    @property
    def W_dec(self) -> Float[Tensor, "inst d_sae d_in"]:
        # YOUR CODE HERE - return the decoder weights. Depending on what you name your
        # weights in __init__, this may not differ from the `ToySAE` implementation.
        raise NotImplementedError()

    @property
    def W_mag(self) -> Float[Tensor, "inst d_in d_sae"]:
        # YOUR CODE HERE - implement the magnitude weights getter (tied as described above).
        raise NotImplementedError()

    def forward(
        self, h: Float[Tensor, "batch inst d_in"]
    ) -> tuple[
        dict[str, Float[Tensor, "batch inst"]],
        Float[Tensor, ""],
        Float[Tensor, "batch inst d_sae"],
        Float[Tensor, "batch inst d_in"],
    ]:
        """
        Same as previous forward function, but allows for gated case as well (in which case we have
        different functional form, as well as a new term "L_aux" in the loss dict).
        """
        # YOUR CODE HERE - implement the Gated forward function. This will be similar
        # to the standard forward function, but with the gating mechanism included
        # (plus a new loss term "L_aux" in the loss dict).
        raise NotImplementedError()

        assert sorted(loss_dict.keys()) == ["L_aux", "L_reconstruction", "L_sparsity"]
        return loss_dict, loss, acts_post, h_reconstructed

    @t.no_grad()
    def resample_simple(self, frac_active_in_window: Float[Tensor, "window inst d_sae"], resample_scale: float) -> None:
        # YOUR CODE HERE - implement the resample_simple function for the Gated SAE.
        # This will be identical to the ToySAE implementation, except that it will
        # apply to different weights & biases.
        raise NotImplementedError()

    @t.no_grad()
    def resample_advanced(
        self,
        frac_active_in_window: Float[Tensor, "window inst d_sae"],
        resample_scale: float,
        batch_size: int,
    ) -> None:
        # YOUR CODE HERE - implement the resample_advanced function for the Gated SAE.
        # This will be identical to the ToySAE implementation, except that it will
        # apply to different weights & biases.
        raise NotImplementedError()

<details><summary>솔루션</summary>

```python
class GatedToySAE(ToySAE):
    W_gate: Float[Tensor, "inst d_in d_sae"]
    b_gate: Float[Tensor, "inst d_sae"]
    r_mag: Float[Tensor, "inst d_sae"]
    b_mag: Float[Tensor, "inst d_sae"]
    _W_dec: Float[Tensor, "inst d_sae d_in"] | None
    b_dec: Float[Tensor, "inst d_in"]

    def __init__(self, cfg: ToySAEConfig, model: ToyModel):
        super(ToySAE, self).__init__()

        assert cfg.d_in == model.cfg.d_hidden, "ToyModel's hidden dim doesn't match SAE input dim"
        self.cfg = cfg
        self.model = model.requires_grad_(False)
        self.model.W.data[1:] = self.model.W.data[0]
        self.model.b_final.data[1:] = self.model.b_final.data[0]

        self._W_dec = (
            None
            if self.cfg.tied_weights
            else nn.Parameter(nn.init.kaiming_uniform_(t.empty((cfg.n_inst, cfg.d_sae, cfg.d_in))))
        )
        self.b_dec = nn.Parameter(t.zeros(cfg.n_inst, cfg.d_in))

        self.W_gate = nn.Parameter(nn.init.kaiming_uniform_(t.empty((cfg.n_inst, cfg.d_in, cfg.d_sae))))
        self.b_gate = nn.Parameter(t.zeros(cfg.n_inst, cfg.d_sae))
        self.r_mag = nn.Parameter(t.zeros(cfg.n_inst, cfg.d_sae))
        self.b_mag = nn.Parameter(t.zeros(cfg.n_inst, cfg.d_sae))

        self.to(device)

    @property
    def W_dec(self) -> Float[Tensor, "inst d_sae d_in"]:
        return self._W_dec if self._W_dec is not None else self.W_gate.transpose(-1, -2)

    @property
    def W_mag(self) -> Float[Tensor, "inst d_in d_sae"]:
        return self.r_mag.exp().unsqueeze(1) * self.W_gate

    def forward(
        self, h: Float[Tensor, "batch inst d_in"]
    ) -> tuple[
        dict[str, Float[Tensor, "batch inst"]],
        Float[Tensor, ""],
        Float[Tensor, "batch inst d_sae"],
        Float[Tensor, "batch inst d_in"],
    ]:
        """
        Same as previous forward function, but allows for gated case as well (in which case we have
        different functional form, as well as a new term "L_aux" in the loss dict).
        """
        h_cent = h - self.b_dec

        # Compute the gating terms (pi_gate(x) and f_gate(x) in the paper)
        gating_pre_activation = (
            einops.einsum(h_cent, self.W_gate, "batch inst d_in, inst d_in d_sae -> batch inst d_sae") + self.b_gate
        )
        active_features = (gating_pre_activation > 0).float()

        # Compute the magnitude term (f_mag(x) in the paper)
        magnitude_pre_activation = (
            einops.einsum(h_cent, self.W_mag, "batch inst d_in, inst d_in d_sae -> batch inst d_sae") + self.b_mag
        )
        feature_magnitudes = F.relu(magnitude_pre_activation)

        # Compute the hidden activations (f˜(x) in the paper)
        acts_post = active_features * feature_magnitudes

        # Compute reconstructed input
        h_reconstructed = (
            einops.einsum(acts_post, self.W_dec, "batch inst d_sae, inst d_sae d_in -> batch inst d_in") + self.b_dec
        )

        # Compute loss terms
        gating_post_activation = F.relu(gating_pre_activation)
        via_gate_reconstruction = (
            einops.einsum(
                gating_post_activation,
                self.W_dec.detach(),
                "batch inst d_sae, inst d_sae d_in -> batch inst d_in",
            )
            + self.b_dec.detach()
        )
        loss_dict = {
            "L_reconstruction": (h_reconstructed - h).pow(2).mean(-1),
            "L_sparsity": gating_post_activation.sum(-1),
            "L_aux": (via_gate_reconstruction - h).pow(2).sum(-1),
        }

        loss = loss_dict["L_reconstruction"] + self.cfg.sparsity_coeff * loss_dict["L_sparsity"] + loss_dict["L_aux"]

        assert sorted(loss_dict.keys()) == ["L_aux", "L_reconstruction", "L_sparsity"]
        return loss_dict, loss, acts_post, h_reconstructed

    @t.no_grad()
    def resample_simple(self, frac_active_in_window: Float[Tensor, "window inst d_sae"], resample_scale: float) -> None:
        dead_latents_mask = (frac_active_in_window < 1e-8).all(dim=0)  # [instances d_sae]
        n_dead = int(dead_latents_mask.int().sum().item())

        replacement_values = t.randn((n_dead, self.cfg.d_in), device=self.W_gate.device)
        replacement_values_normed = replacement_values / (
            replacement_values.norm(dim=-1, keepdim=True) + self.cfg.weight_normalize_eps
        )

        # New names for weights & biases to resample
        self.W_gate.data.transpose(-1, -2)[dead_latents_mask] = resample_scale * replacement_values_normed
        self.W_dec.data[dead_latents_mask] = replacement_values_normed
        self.b_mag.data[dead_latents_mask] = 0.0
        self.b_gate.data[dead_latents_mask] = 0.0
        self.r_mag.data[dead_latents_mask] = 0.0

    @t.no_grad()
    def resample_advanced(
        self,
        frac_active_in_window: Float[Tensor, "window inst d_sae"],
        resample_scale: float,
        batch_size: int,
    ) -> None:
        h = self.generate_batch(batch_size)
        l2_loss = self.forward(h)[0]["L_reconstruction"]

        for instance in range(self.cfg.n_inst):
            is_dead = (frac_active_in_window[:, instance] < 1e-8).all(dim=0)
            dead_latents = t.nonzero(is_dead).squeeze(-1)
            n_dead = dead_latents.numel()
            if n_dead == 0:
                continue

            l2_loss_instance = l2_loss[:, instance]  # [batch_size]
            if l2_loss_instance.max() < 1e-6:
                continue

            distn = Categorical(probs=l2_loss_instance.pow(2) / l2_loss_instance.pow(2).sum())
            replacement_indices = distn.sample((n_dead,))  # type: ignore

            replacement_values = (h - self.b_dec)[replacement_indices, instance]  # [n_dead d_in]
            replacement_values_normalized = replacement_values / (
                replacement_values.norm(dim=-1, keepdim=True) + self.cfg.weight_normalize_eps
            )

            W_gate_norm_alive_mean = (
                self.W_gate[instance, :, ~is_dead].norm(dim=0).mean().item() if (~is_dead).any() else 1.0
            )

            # New names for weights & biases to resample
            self.W_dec.data[instance, dead_latents, :] = replacement_values_normalized
            self.W_gate.data[instance, :, dead_latents] = (
                replacement_values_normalized.T * W_gate_norm_alive_mean * resample_scale
            )
            self.b_mag.data[instance, dead_latents] = 0.0
            self.b_gate.data[instance, dead_latents] = 0.0
            self.r_mag.data[instance, dead_latents] = 0.0
```
</details>

이제 아래 코드를 실행하여 Gated SAE를 학습시키고 결과를 시각화할 수 있습니다. 다만, 여기서는 상위 4/16개 인스턴스(마지막 10개 샘플 배치에 대해 평균을 낸 loss 기준)만 플롯한다는 점에 유의하시기 바랍니다. 일반적으로 toy 모델에서 thresholding을 사용하는 SAE는 local minima에 더 쉽게 빠지는 경향이 있기 때문입니다 (제 생각에는 thresholding이 loss landscape를 평탄하게 만들어 더 많은 탐색과 local minima 발견을 가능하게 하는 반면, 단순한 SAE 구조는 global minimum으로 더 직접적으로 유도되기 때문인 것으로 보입니다).

In [ ]:
gated_sae = GatedToySAE(
    cfg=ToySAEConfig(
        n_inst=n_inst,
        d_in=d_in,
        d_sae=d_sae,
        sparsity_coeff=1.0,
    ),
    model=model,
)
gated_data_log = gated_sae.optimize(steps=20_000, resample_method="advanced")

# Animate the best instances, ranked according to average loss near the end of training
n_inst_to_plot = 4
n_batches_for_eval = 10
avg_loss = t.concat([d["loss"] for d in gated_data_log[-n_batches_for_eval:]]).mean(0)
best_instances = avg_loss.topk(n_inst_to_plot, largest=False).indices.tolist()

utils.animate_features_in_2d(
    gated_data_log,
    rows=["W_gate", "_W_dec", "h", "h_r"],
    instances=best_instances,
    filename=str(section_dir / "animation-training-gated.html"),
    color_resampled_latents=True,
    title="SAE on toy model",
)

### 연습 문제 - Gated 모델의 장점 증명하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> This is a quite long and unguided exercise, we recommend you come back to it after you've gone through the other content in this notebook.
> ```

Gated 및 JumpReLU와 같은 thresholding 모델이 표준 SAE보다 더 나은 성능을 낼 수 있는 이유를 생각할 때, 머릿속에 그려야 할 그림은 DeepMind의 Gated SAEs 논문 부록에 있는 아래 그래프입니다. 왼쪽 히스토그램은 특정 feature 방향을 따른 분포를 보여줍니다. 파란색은 해당 feature가 꺼져 있지만 다른 비직교(non-orthogonal) feature들이 켜져 있을 때 발생하는 간섭(interference) 분포를 나타내며, 빨간색은 해당 feature가 켜져 있을 때의 분포를 나타냅니다. 이 분포들은 명확한 bimodal 패턴을 형성하며, 오른쪽 그림을 통해 ReLU나 Gated 모델이 제공하는 jump discontinuity가 어떻게 더 많은 간섭 사례(파란색)를 0으로 정확하게 재구성함으로써 이러한 불연속성을 더 잘 모델링할 수 있는지 확인할 수 있습니다.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/distn-gated.png" width="900">

우리의 데이터 분포가 여기의 분포와 완전히 동일하지는 않지만, 여전히 bimodal입니다. "$f$이 활성화되었을 때의 feature 방향 $f$를 따른 projection"의 히스토그램은 "$f$가 비활성화되었을 때의 feature 방향 $f$를 따른 projection"의 히스토그램보다 훨씬 더 큰 평균값을 가질 것입니다. 실제로, 여러분이 직접 이 그래프를 재현하여 Gated 모델이 표준 모델보다 어떻게 더 나은 성능을 보이는지 정확하게 보여줄 수 있습니다.

이 연습 문제는 단순히 빈 함수를 채우는 방식이 아니라 비교적 개방형으로 남겨두었습니다. 시도해보고 싶으시다면, 시각화를 위해 Claude나 ChatGPT의 도움을 받는 것을 추천합니다. 중요한 점은 그래프를 충분히 이해하여 이를 재현하기 위해 어떤 데이터를 수집해야 하는지 아는 것입니다. 또한, 우리의 toy model 설정이 논문의 설정과 약간 다르다는 점에 유의하십시오. 우리는 5개의 독립적인 feature를 사용하므로 "X off" 분포는 다른 feature들로부터 오는 간섭에 의해 결정되지만, 논문에서는 단일 feature만을 고려하고 "X on"과 "X off" 분포를 미리 정의합니다. docstring이 우리가 여기서 어떤 그래프를 만들고 있는지 이해하는 데 도움이 될 것입니다.

원하신다면, 함수 `generate_batch`을 확장하여 대부분의 확률 질량이 `[0, 1]` 범위에 있는 정규 분포를 지원하도록 만들 수 있습니다 (이것이 `ToyModelConfig` 클래스의 `feat_mag_distn` 필드가 존재하는 이유입니다). 이렇게 하면 논문의 toy model 설정에 있는 분포와 더 밀접하게 일치하게 됩니다. 하지만 핵심 결과를 재현하기 위해 반드시 이 작업을 수행해야 하는 것은 아닙니다.

In [ ]:
# YOUR CODE HERE - replicate figure 15a & 15b from the paper

<details><summary>솔루션</summary>

```python
def generate_batch(self: ToyModel, batch_size: int) -> Float[Tensor, "batch inst feats"]:
    """
    Generates a batch of data of shape (batch_size, n_instances, n_features).

    This is optional, we just provide the function for you here to use for completeness (the code
    run below will not use "normal" distribution mode to generate the data), it'll use the same
    "unif" mode we've used so far.)
    """
    assert self.cfg.feat_mag_distn in ["unif", "normal"], f"Unknown feature distribution: {self.cfg.feat_mag_distn}"

    batch_shape = (batch_size, self.cfg.n_inst, self.cfg.n_features)
    feat_seeds = t.rand(batch_shape, device=self.W.device)
    feat_mag = (
        t.rand(batch_shape, device=self.W.device)
        if self.cfg.feat_mag_distn == "unif"
        else t.clip(0.5 + 0.2 * t.randn(batch_shape, device=self.W.device), min=0.0, max=1.0)
    )
    return t.where(feat_seeds <= self.feature_probability, feat_mag, 0.0)


ToyModel.generate_batch = generate_batch


@t.inference_mode()
def replicate_figure_15(sae_tuples: list[tuple[str, ToySAE, list[dict[str, Any]]]]) -> None:
    """
    This function should replicate figure 15 from the DeepMind paper, in a way which conforms to our
    toy model setup. It should create 2 plots:

        (1) A histogram of activation distributions projected along some chosen feature direction,
            color coded according to whether that feature is active or inactive. You should find
            that the distribution when active is almost always positive, and the distribution when
            not active has mean below zero.

        (2) A scatter plot of SAE reconstructions. In other words, the x-axis values should be the
            original feature values, and the y-axis should be the SAE's reconstructions of those
            features (i.e. the post-ReLU activations of the SAE). You should use different colors
            for different SAE architectures.
    """
    # ! (1) Histogram of activation projections

    # Generate a batch of features (with at least one feature in our instance being non-zero). Note,
    # our choice of model and instance / feature idx here is arbitrary, since we've verified all
    # models learn the uniform solution (we're only using models for this plot, not the saes)
    data = defaultdict(list)
    data = defaultdict(list)
    model = sae_tuples[0][1].model
    instance_idx = feature_idx = 0
    feature_idx = 1
    n_samples = 10_000
    feats = t.empty((0, model.cfg.n_features), device=device)
    while feats.shape[0] < n_samples:
        new_feats = model.generate_batch(n_samples)[:, instance_idx]
        new_feats = new_feats[(new_feats > 1e-4).any(dim=-1)]  # shape [batch, feats]
        feats = t.cat([feats, new_feats], dim=0)[:n_samples]

    # Map these to hidden activations, then project them back to feature directions for the 0th
    # feature, and plot them
    h = feats @ model.W[instance_idx].T
    h_proj = h @ model.W[instance_idx, :, feature_idx]
    is_active = feats[:, feature_idx] > 1e-4
    px.histogram(
        pd.DataFrame(
            {
                "x": h_proj.tolist(),
                "Feature": ["on" if active else "off" for active in is_active.tolist()],
            }
        ).sort_values(by="Feature", inplace=False),
        color="Feature",
        marginal="box",
        barmode="overlay",
        width=800,
        height=500,
        opacity=0.6,
        title="Distribution of activation projection",
    ).update_layout(bargap=0.02).show()

    # ! (2) Scatter plot of SAE reconstructions

    for mode, sae, data in sae_tuples:
        # Get repeated version of `h` to use in our fwd pass
        h = feats @ sae.model.W[instance_idx].T
        h_repeated = einops.repeat(h, "batch d_in -> batch inst d_in", inst=sae.cfg.n_inst)

        # Get the best instance, and get activations for this instance
        n_batches_for_eval = 10
        best_inst = t.concat([d["loss"] for d in data_log[-n_batches_for_eval:]]).mean(0).argmin().item()
        acts = sae.forward(h_repeated)[2][:, best_inst]  # shape [batch, d_sae]

        # Find the SAE latent that corresponds to this 0th feature (we're assuming here that there
        # actually is one!)
        latent_idx = acts[feats[:, feature_idx] > 1e-4].mean(0).argmax().item()

        # Add data for the second histogram. In this context we scale our activations by the norm of
        # model.W. This is because our activations `acts` are defined as the coefficients of unit
        # vecs whose sparse combination equals the true features, but our features `feats` weren't
        # defined this same way because model.W isn't normalized.
        data["Act"].extend(feats[:, feature_idx].tolist())
        data["Reconstructed act"].extend((acts[:, latent_idx] / sae.model.W[best_inst, :, feature_idx].norm()).tolist())
        data["SAE function"].extend([mode for _ in range(len(feats))])

    # Second histogram: comparison of activation projection & reconstructed activation projection
    px.scatter(
        pd.DataFrame(data),
        width=800,
        height=500,
        title=f"Act vs Reconstructed Act for {' & '.join(m.capitalize() for m, _, _ in sae_tuples)}",
        color="SAE function",
        x="Act",
        opacity=0.25,
        y="Reconstructed act",
        marginal_y="histogram",
        render_mode="webgl",
    ).add_shape(
        type="line",
        x0=0,
        y0=0,
        x1=1.1,
        y1=1.1,
        layer="below",
        line=dict(color="#666", width=2, dash="dash"),
    ).update_layout(xaxis=dict(range=[0, 1.1]), xaxis2=dict(range=[0, int(0.01 * n_samples)])).show()


replicate_figure_15(
    [
        ("standard", resampling_sae, resampling_data_log),
        ("gated", gated_sae, gated_data_log),
    ],
)
```
</details>

이를 올바르게 수행했다면, 두 가지 차이점을 제외하고는 논문의 그림 15b와 유사한 플롯을 관찰할 수 있을 것입니다. 그중 하나는 두 SAE 모두에서 나타나는 추가적인 노이즈(즉, 단조 증가하는 직선 위에 있지 않은 데이터 포인트들)입니다. 이는 우리의 toy model 설정이 DeepMind의 설정과 다르기 때문입니다(이 포인트들은 5개의 feature 중 두 개 이상이 동시에 활성화된 경우에 해당합니다). 하지만 또 다른 흥미로운 차이점이 있습니다. 그것이 무엇인지 찾아내고, 왜 발생하는지 설명할 수 있습니까?

참고로, 플롯을 생성하지 못했다면 솔루션 Colab이나 Streamlit 드롭다운을 확인한 후 이 질문에 답해 보시기 바랍니다.

<details>
<summary>차이점이 무엇인지</summary>

Gated 모델의 선은 논문과 동일하지만, standard 모델의 선은 더 낮게 위치합니다. 논문의 다이어그램에서처럼 Gated 선 위로 교차하지 않습니다.

</details>

<details>
<summary>차이점에 대한 설명 (힌트)</summary>

DeepMind 논문의 toy model 섹션을 살펴보십시오. 그들은 실제로 해당 플롯을 위한 데이터를 어떻게 생성했습니까? 그들의 플롯에서는 나타나지 않지만 우리의 플롯에서 경험할 수 있는 특별한 현상이 있습니까?

</details>

<details>
<summary>차이점에 대한 설명 (정답)</summary>

정답은 **shrinkage**입니다.

DeepMind 논문은 reconstruction loss와 sparsity penalty를 사용하여 실제로 SAE를 학습시켜 그림을 생성한 것이 아닙니다. 그들은 가장 작은 reconstruction loss를 유도하는 projection(및 bias / thresholding)을 찾아 문제를 분석적으로 해결했습니다. 이는 그들의 standard SAE가 shrinkage 문제를 겪지 않았음을 의미합니다. 하지만 우리는 L1 penalty로 학습시켰으며, 이는 shrinkage가 발생함을 의미합니다. 따라서 standard SAE의 선이 gated 선 아래로 떨어지게 됩니다.

gated 선(0이 아닌 부분)은 대략적으로 직선 `x=y` 을 통과하며, 즉 shrinkage를 겪지 않는다는 점에 유의하십시오. 이는 우리가 예상한 바와 일치합니다(앞서 thresholding이 모델로 하여금 shrinkage 문제를 피할 수 있게 한다는 점을 논의했습니다).

</details>

### JumpReLU SAEs

> 참고 - 이 섹션은 수학적으로 다소 밀도가 높으므로, 이 내용이 익숙하지 않으시다면 건너뛰셔도 좋습니다.

JumpReLU SAEs는 Gated SAEs와 많은 동일한 장점을 제공하지만, Gated SAEs처럼 보조 손실 함수(auxiliary loss function)를 계산하기 위해 분리된 forward pass가 필요하지 않습니다. 또한, Gated SAEs 논문의 증거(특히 ablation studies 섹션)에 따르면 Gated SAEs는 magnitude와 gating weights를 분리하는 능력으로부터 이득을 얻지 못한다는 점이 시사되었습니다. 이는 우리가 그냥 JumpReLU SAEs를 사용하는 것이 더 나을 수 있음을 의미합니다! 유일한 단점은 일부 그룹에서 학습시키기가 조금 더 어렵다는 점을 발견했다는 것이지만, 여기의 간단한 모델들의 경우에는 큰 문제 없이 학습시킬 수 있을 것입니다.

JumpReLU architecture는 일반적인 SAEs와 동일하지만, 추가 파라미터 $\theta$ (각 latent의 임계값을 나타내는 길이 `d_sae`의 벡터)가 있으며, activation function은 $\operatorname{JumpReLU}_\theta(z) = z H(z - \theta)$ 입니다. 여기서 $z$은 pre-activation SAE hidden values이고 $H$는 Heaviside step function (즉, $z > \theta$이면 1, 그렇지 않으면 0의 값)입니다. 함수 형태는 다음과 같습니다:

<img src="https://res.cloudinary.com/lesswrong-2-0/image/upload/f_auto,q_auto/v1/mirroredImages/wZqqQysfLrt2CFx4T/zzrdot3xexvcz3mqghn8" width="240">

우리는 다음과 같은 손실 함수를 사용하여 JumpReLU SAEs를 학습시킵니다:

$$
\mathcal{L}(\mathbf{x}):=\underbrace{\|\mathbf{x}-\hat{\mathbf{x}}(\mathbf{f}(\mathbf{x}))\|_2^2}_{\mathcal{L}_{\text {reconstruct }}}+\underbrace{\lambda\|\mathbf{f}(\mathbf{x})\|_0}_{\mathcal{L}_{\text {sparsity }}}
$$

이는 표준 SAE 손실 함수와 유사하지만, L1 대신 hidden activations의 L0 norm에 직접 페널티를 부여한다는 점이 다릅니다. 여기서 질문이 남습니다 - Heaviside 함수와 L0 norm 모두 불연속적인데, 어떻게 $\theta$에 대해 이 항들을 backprop 할 수 있을까요? 정답은 미분 불가능한 함수의 gradient를 근사하는 방법인 **straight-through-estimators** (STEs)에 있습니다. 구체적으로, 먼저 L0 항을 Heaviside step function $\|\mathbf{f}(\mathbf{x})\|_0 = \sum_{i=1}^{d_{\text{sae}}} H(\pi_i(\mathbf{x}) - \theta_i)$로 다시 씁니다. 여기서 $\pi_i(\mathbf{x})$은 pre-JumpReLU SAE hidden values입니다. 다음으로, 문제를 Heaviside와 JumpReLU 함수에 대한 생각으로 축소했으므로, 다음과 같은 추정치를 사용할 수 있습니다:

$$
\begin{aligned}
\frac{ð}{ð \theta} \operatorname{JumpReLU}_\theta(z) & :=-\frac{\theta}{\varepsilon} K\left(\frac{z-\theta}{\varepsilon}\right) \\
\frac{ð}{ð \theta} H(z-\theta) & :=-\frac{1}{\varepsilon} K\left(\frac{z-\theta}{\varepsilon}\right)
\end{aligned}
$$

여기서 $K$는 어떤 **valid kernel function** (즉, 중심이 잡혀 있고 유한한 분산을 가진 확률 밀도 함수의 특성을 만족해야 함)입니다. GDM 실험에서는 **rectangle function** $H(z+\frac{1}{2}) - H(z-\frac{1}{2})$을 사용합니다.

이것이 왜 작동하는지에 대해 아래에 두 가지 직관(함수적/시각적 직관과 확률 기반 직관)을 제공합니다. 이 내용에 정말 관심이 없으시다면 연습 문제 섹션으로 건너뛰셔도 됩니다 (하지만 적어도 하나는 읽어보시기를 권장합니다).

#### 기능적 / 시각적 직관

여기서 우리가 실제로 하고 있는 것은 급격한 cumulative distribution function을 사용하여 불연속 함수를 근사하는 것입니다. 예를 들어, heaviside 함수 $H(z) = \mathbf{1}(z > 0)$를 생각해 보겠습니다. 우리는 불연속점 주변에서 급격한 cdf $F$를 통해 이를 근사할 수 있습니다 (즉, 약간의 음수 $z$에 대해서는 $F(z) = 0$이고, 약간의 양수 $z$에 대해서는 $F(z) = 1$가 됩니다). 위에서 우리의 미분 근사에 probability density function $K$이 포함되는 이유는 cumulative distribution function $F$의 미분이 바로 probability density function이기 때문입니다.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/jumprelu-1.png" width="560">

관심이 있으시다면, 아래 드롭다운에서 실제 미적분학을 사용하여 이 결과를 도출하는 과정을 확인하실 수 있습니다 (즉, 충분히 큰 영역에 대해 이러한 근사 미분값을 적분한 결과가 jump discontinuity의 크기와 같음을 보여줍니다). 이는 필수적인 내용은 아니며, 특별히 궁금하지 않으시다면 굳이 권장하지 않습니다.

<details>
<summary>이 적분 결과의 도출 과정 (덜 중요함)</summary>

$F$가 $K$의 cumulative distribution function이라고 가정하면, $F'(z) = K(z)$와 $F(-\infty) = 0, F(\infty) = 1$를 얻습니다. 이제 중심이 $z$이고 반지름이 $\epsilon C$인 영역에서 근사된 Heaviside 함수의 적분을 계산해 보겠습니다. $\theta$이 $z$ 위에서 $z$ 아래로 이동할 때 출력이 0에서 1로 급격히 변하기 때문에, 음수 범위에 대해 적분을 계산한다는 점에 유의하십시오.

$$
\int\limits_{z+\epsilon C}^{z-\epsilon C} -\frac{1}{\epsilon} K\left(\frac{z-\theta}{\epsilon}\right) d\theta = \int\limits_{-C}^{C} K(\theta)\; d\theta = F(C) - F(-C) \xrightarrow[C \to \infty]{} 1 - 0 = 1
$$

이는 jump discontinuity의 크기와 같습니다. 커널 함수로 rectangle function $H(z+\frac{1}{2}) - H(z-\frac{1}{2})$를 선택한 경우, $C=\frac{1}{2}$과 같은 작은 영역에 대해 적분하더라도 $\theta \in [z - \frac{\epsilon}{2}, z + \frac{\epsilon}{2}]$와 같이 이 결과가 유지됩니다. 우리가 jump discontinuity에 가까울 때 $\theta$ 값에 미치는 영향이 가장 크고, 다른 대부분의 영역에서는 0이어야 하므로 이러한 특성을 갖는 것이 타당합니다.

JumpReLU 항의 경우, 위의 재매개변수화를 적용한 후 $\theta K(\theta)$의 적분을 pdf $K$를 가진 변수의 기댓값(우리가 선택한 $K$**에 의해 0이 됩니다)으로 인식할 수 있으며, 이는 다음과 같습니다:

$$
\int\limits_{z+\epsilon C}^{z-\epsilon C} -\frac{\theta}{\epsilon} K\left(\frac{z-\theta}{\epsilon}\right) d\theta = \int\limits_{-C}^{C} (z - \theta) K(\theta)\; d\theta = \int\limits_{-C}^{C} z K(\theta)\; d\theta \xrightarrow[C \to \infty]{} z
$$

이 결과 역시 jump discontinuity의 크기와 같으며, 선택한 커널 $K$에 대해 영역 $\theta \in [z - \frac{\epsilon}{2}, z + \frac{\epsilon}{2}]$만 취하더라도 성립하는 결과입니다.

**엄밀히 말하면 전체 도메인에 대해 적분할 때만 0이 됩니다. 하지만 우리가 선택한 $K$(및 $K$에 대한 대부분의 합리적인 선택들)는 0에 중심을 두고 있을 뿐만 아니라 0을 중심으로 대칭이며 0에서 멀어질수록 빠르게 감소하므로, 이러한 가정을 할 수 있습니다.

</details>

#### 확률 기반의 직관

이를 생각하는 또 다른 방법은 우리의 입력 $x$에 어느 정도의 무작위성 요소가 있다고 보는 것입니다. 따라서 우리의 loss 함수 값 $\mathcal{L}_\theta(x)$ 그 자체로 기대 loss $\mathbb{E}_x\left[\mathcal{L}_\theta(x)\right]$를 근사하는 random variable이 됩니다. 그리고 loss에 비연속적인 항이 포함되어 있어 loss의 gradient를 직접 계산할 수 없는 경우라도, 기대 loss의 gradient는 계산할 수 있다는 것이 밝혀졌습니다. 예를 들어, sparsity 항 $\|\mathbf{f}(\mathbf{x})\|_0 = \sum_{i=1}^{d_{\text{sae}}} H(z_i - \theta_i)$ (여기서 $z_i$는 pre-JumpReLU hidden 값들입니다)을 생각해 보십시오. 이는 0에서 미분 불가능하지만, 그 기대값은 $\mathbb{E}_x \|\mathbf{f}(\mathbf{x})\|_0 = \sum_{i=1}^{d_{\text{sae}}} \mathbb{P}(z_i > \theta_i)$이며 이는 미분 가능합니다. $\theta_i$에 대한 도함수는 $-\mathbb{E}_x\left[p_i(z_i-\theta_i)\right]$이며, 여기서 $p_i$은 $z_i$에 대한 probability density function들입니다.

좋습니다, 이제 기대값 관점에서 도함수가 어떻게 되어야 하는지는 알았습니다. 그런데 왜 우리의 선택 $\frac{ð}{ð \theta} H(z-\theta) :=-\frac{1}{\varepsilon} K\left(\frac{z-\theta}{\varepsilon}\right)$이 이를 만족하는 것일까요? 정답은 이 식이 **kernel density estimation** (KDE)의 한 형태이기 때문입니다. 즉, 이는 empirical distribution을 부드럽게 만들어 변수의 pdf를 근사합니다.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/jumprelu-2.png" width="620">

실제 실습으로 넘어가기 전에 JumpReLU SAE에 대한 몇 가지 마지막 참고 사항입니다:

- L1 대신 L0를 penalty로 사용했을 때의 장점은 특정 sparsity 값을 목표로 설정할 수 있다는 점입니다. 단순히 L0를 penalty로 사용하는 대신, L0와 특정 목표 수준 사이의 제곱 차이를 사용할 수 있습니다: $\mathcal{L}_{\text {sparsity }}(\mathbf{x})=\lambda\left(\|\mathbf{f}(\mathbf{x})\|_0 / L_0^{\text {target }}-1\right)^2$. 이번 실습에서는 이를 구현하지 않지만, 표준 버전을 성공적으로 작동시킨 후 직접 구현해 보시는 것을 추천합니다.

### 연습 문제 - 커스텀 gradient 함수 구현하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 15-30 minutes on this and the next exercise.
> ```

우리는 DeepMind가 부록에서 구현한 방식과 유사하게, 커스텀 `jumprelu` 및 `heaviside` 함수를 구현하는 것부터 시작하겠습니다. PyTorch는 forward 및 backward 패스에서 서로 다른 동작을 갖는 커스텀 함수를 생성할 수 있는 유용한 방법을 제공합니다. 예를 들어, 아래는 forward 동작 $f(x) = x^n$과 backward 동작 $f'(x) = nx^{n-1}$를 가진 함수입니다.

참고로, 우리는 실제로 chain rule을 통해 $\frac{dL}{dx} = \frac{dL}{df(x)} \times f'(x)$을 계산하고 있으므로(여기서 $x$은 `input`이고 $\frac{dL}{df(x)}$은 `grad_output`입니다), backward 함수에서 단순히 `n * (input ** (n - 1))`이 아니라 `n * (input ** (n - 1)) * grad_output`를 반환해야 합니다. 이 부분이 혼란스럽다면, 직접 backprop을 구축하는 내용이 담긴 ARENA fundamentals 챕터의 자료를 다시 살펴보시기 바랍니다.

또한 `backward` 함수는 실제로 튜플을 반환하며, 이는 `forward`에 전달된 순서대로 각 `forward` 인자에 대한 모든 gradient로 구성됩니다(여기에는 정수 `n`도 포함됩니다). 우리는 이 변수에 대한 gradient를 추적할 필요가 없으므로 `None`을 반환합니다.

In [ ]:
class CustomFunction(t.autograd.Function):
    @staticmethod
    def forward(ctx: Any, input: Tensor, n: int) -> Tensor:
        # Save any necessary information for backward pass
        ctx.save_for_backward(input)
        ctx.n = n  # Save n as it will be needed in the backward pass
        # Compute the output
        return input**n

    @staticmethod
    def backward(ctx: Any, grad_output: Tensor) -> tuple[Tensor, None]:
        # Retrieve saved tensors and n
        (input,) = ctx.saved_tensors
        n = ctx.n
        # Return gradient for input and None for n (as it's not a Tensor)
        return n * (input ** (n - 1)) * grad_output, None


# Test our function, and its gradient
input = t.tensor(3.0, requires_grad=True)
output = CustomFunction.apply(input, 2)
output.backward()

t.testing.assert_close(output, t.tensor(9.0))
t.testing.assert_close(input.grad, t.tensor(6.0))

이제 직접 `jumprelu` 및 `heaviside` 함수를 구현해야 합니다. 두 함수 모두 2개의 tensor 입력인 $z$와 $\theta$, 그리고 하나의 float $\epsilon$를 입력으로 받는다는 점에 유의하십시오. Heaviside 함수에 대해서는 다음과 같은 관례를 사용합니다:

$$
\begin{aligned}
H(z, \theta; \epsilon) & := \boldsymbol{\mathbb{1}}[z - \theta > 0] \\
\frac{ð}{ð z} H(z, \theta; \epsilon) & := 0 \\
\frac{ð}{ð \theta} H(z, \theta; \epsilon) & := -\frac{1}{\epsilon} K\left(\frac{z-\theta}{\epsilon}\right) \\
\end{aligned}
$$

그리고 JumpReLU에 대해서는 다음과 같습니다:

$$
\begin{aligned}
\operatorname{JumpReLU}(z, \theta; \epsilon) & := z \cdot \boldsymbol{\mathbb{1}}[z - \theta > 0] \\
\frac{ð}{ð z} \operatorname{JumpReLU}(z, \theta; \epsilon) & := \boldsymbol{\mathbb{1}}[z - \theta > 0] \\
\frac{ð}{ð \theta} \operatorname{JumpReLU}(z, \theta; \epsilon) & :=-\frac{\theta}{\epsilon} K\left(\frac{z-\theta}{\epsilon}\right)
\end{aligned}
$$

여기서 $K(x) = \boldsymbol{\mathbb{1}}\left[|x| < \frac{1}{2}\right]$은 rectangle kernel 함수입니다.

두 경우 모두 $\theta$에 대한 미분에는 STE estimator를 사용하지만, $z$에 대한 STE 추정치는 무시합니다. 즉, $\frac{ð}{ð z} \boldsymbol{\mathbb{1}}[z - \theta > 0] = 0$이라고 가정하고 $z$에 대해 미분합니다. 이는 파라미터 $\theta$만이 thresholding 동작을 구현하도록 하기 위함입니다. 본질적으로, 다른 파라미터들은 출력이 해당 파라미터들의 국소적으로 연속적인 함수라는 가정하에 gradient descent에 의해 업데이트된다고 생각하시면 됩니다.

시작하기 전 몇 가지 마지막 참고 사항입니다:

- 두 구현 모두에서 사용할 수 있는 `rectangle` helper 함수를 제공해 드렸습니다.
- 이번 연습 문제에서는 broadcasting 문제에 대해 걱정하실 필요가 없습니다. PyTorch의 autograd 메커니즘이 이를 처리해주기 때문입니다 (예를 들어, `backward`에서 반환하는 `theta`의 gradient에 leading batch dimension이 있어 `theta`과 모양이 다른 경우, `theta.grad`에 더해지기 전에 해당 차원에 대해 자동으로 합산됩니다). 하지만 DeepMind 논문 부록의 pseudocode와 정확히 일치시키고 싶다면, 이 합산 과정을 명시적으로 작성하셔도 좋습니다. backprop 중 차원에 대한 합산 및 broadcasting의 세부 사항에 대해 더 자세히 알고 싶다면, ARENA의 첫 번째 장을 참조하십시오!

In [ ]:
def rectangle(x: Tensor, width: float = 1.0) -> Tensor:
    """
    Returns the rectangle function value, i.e. K(x) = 1[|x| < width/2], as a float.
    """
    return (x.abs() < width / 2).float()


class Heaviside(t.autograd.Function):
    """
    Implementation of the Heaviside step function, using straight through estimators for the derivative.

        forward:
            H(z,θ,ε) = 1[z > θ]

        backward:
            dH/dz := None
            dH/dθ := -1/ε * K(z/ε)

            where K is the rectangle kernel function with width 1, centered at 0: K(u) = 1[|u| < 1/2]
    """

    @staticmethod
    def forward(ctx: Any, z: Tensor, theta: Tensor, eps: float) -> Tensor:
        raise NotImplementedError()

    @staticmethod
    def backward(ctx: Any, grad_output: Tensor) -> tuple[Tensor, Tensor, None]:
        raise NotImplementedError()


# Test our Heaviside function, and its pseudo-gradient
z = t.tensor([[1.0, 1.4, 1.6, 2.0]], requires_grad=True)
theta = t.tensor([1.5, 1.5, 1.5, 1.5], requires_grad=True)
eps = 0.5
output = Heaviside.apply(z, theta, eps)
output.backward(t.ones_like(output))  # equiv to backprop on each elem of z independently

# Test values
t.testing.assert_close(output, t.tensor([[0.0, 0.0, 1.0, 1.0]]))  # expect H(θ,z,ε) = 1[z > θ]
t.testing.assert_close(theta.grad, t.tensor([0.0, -2.0, -2.0, 0.0]))  # expect dH/dθ = -1/ε * K((z-θ)/ε)
t.testing.assert_close(z.grad, t.tensor([[0.0, 0.0, 0.0, 0.0]]))  # expect dH/dz = zero

# Test handling of batch dimension
theta.grad = None
output_stacked = Heaviside.apply(t.concat([z, z]), theta, eps)
output_stacked.backward(t.ones_like(output_stacked))
t.testing.assert_close(theta.grad, 2 * t.tensor([0.0, -2.0, -2.0, 0.0]))

print("All tests for `Heaviside` passed!")

<details>
<summary>도움이 필요합니다 - Heaviside 함수의 기대값을 이해하지 못하겠습니다.</summary>

이 다이어그램이 도움이 될 것입니다:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/jumprelu-3b.png" width="700">

</details>


<details><summary>풀이</summary>

```python
def rectangle(x: Tensor, width: float = 1.0) -> Tensor:
    """
    Returns the rectangle function value, i.e. K(x) = 1[|x| < width/2], as a float.
    """
    return (x.abs() < width / 2).float()


class Heaviside(t.autograd.Function):
    """
    Implementation of the Heaviside step function, using straight through estimators for the derivative.

        forward:
            H(z,θ,ε) = 1[z > θ]

        backward:
            dH/dz := None
            dH/dθ := -1/ε * K(z/ε)

            where K is the rectangle kernel function with width 1, centered at 0: K(u) = 1[|u| < 1/2]
    """

    @staticmethod
    def forward(ctx: Any, z: Tensor, theta: Tensor, eps: float) -> Tensor:
        # Save any necessary information for backward pass
        ctx.save_for_backward(z, theta)
        ctx.eps = eps
        # Compute the output
        return (z > theta).float()

    @staticmethod
    def backward(ctx: Any, grad_output: Tensor) -> tuple[Tensor, Tensor, None]:
        # Retrieve saved tensors & values
        (z, theta) = ctx.saved_tensors
        eps = ctx.eps
        # Compute gradient of the loss with respect to z (no STE) and theta (using STE)
        grad_z = 0.0 * grad_output
        grad_theta = -(1.0 / eps) * rectangle((z - theta) / eps) * grad_output
        grad_theta_agg = grad_theta.sum(dim=0)  # note, sum over batch dim isn't strictly necessary

        return grad_z, grad_theta_agg, None
```
</details>

In [ ]:
class JumpReLU(t.autograd.Function):
    """
    Implementation of the JumpReLU function, using straight through estimators for the derivative.

        forward:
            J(z,θ,ε) = z * 1[z > θ]

        backward:
            dJ/dθ := -θ/ε * K((z - θ)/ε)
            dJ/dz := 1[z > θ]

            where K is the rectangle kernel function with width 1, centered at 0: K(u) = 1[|u| < 1/2]
    """

    @staticmethod
    def forward(ctx: Any, z: Tensor, theta: Tensor, eps: float) -> Tensor:
        raise NotImplementedError()

    @staticmethod
    def backward(ctx: Any, grad_output: Tensor) -> tuple[Tensor, Tensor, None]:
        raise NotImplementedError()


# Test our JumpReLU function, and its pseudo-gradient
z = t.tensor([[1.0, 1.4, 1.6, 2.0]], requires_grad=True)
theta = t.tensor([1.5, 1.5, 1.5, 1.5], requires_grad=True)
eps = 0.5
output = JumpReLU.apply(z, theta, eps)
output.backward(t.ones_like(output))  # equiv to backprop on each of the 5 elements of z independently

# Test values
t.testing.assert_close(output, t.tensor([[0.0, 0.0, 1.6, 2.0]]))  # expect J(θ,z,ε) = z * 1[z > θ]
t.testing.assert_close(theta.grad, t.tensor([0.0, -3.0, -3.0, 0.0]))  # expect dJ/dθ = -θ/ε * K((z-θ)/ε)
t.testing.assert_close(z.grad, t.tensor([[0.0, 0.0, 1.0, 1.0]]))  # expect dJ/dz = 1[z > θ]

print("All tests for `JumpReLU` passed!")

<details>
<summary>도움이 필요합니다 - JumpReLU 함수의 기대값을 이해하지 못하겠습니다.</summary>

이 다이어그램이 도움이 될 것입니다. STE는 단지 JumpReLU의 불연속적인 부분에 대한 추정치(estimator)일 뿐이며, 함수 전체에 대한 연속적인 근사치가 아니라는 점을 기억하십시오.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/jumprelu-4.png" width="700">

</details>


<details><summary>솔루션</summary>

```python
class JumpReLU(t.autograd.Function):
    """
    Implementation of the JumpReLU function, using straight through estimators for the derivative.

        forward:
            J(z,θ,ε) = z * 1[z > θ]

        backward:
            dJ/dθ := -θ/ε * K((z - θ)/ε)
            dJ/dz := 1[z > θ]

            where K is the rectangle kernel function with width 1, centered at 0: K(u) = 1[|u| < 1/2]
    """

    @staticmethod
    def forward(ctx: Any, z: Tensor, theta: Tensor, eps: float) -> Tensor:
        # Save any necessary information for backward pass
        ctx.save_for_backward(z, theta)
        ctx.eps = eps
        # Compute the output
        return z * (z > theta).float()

    @staticmethod
    def backward(ctx: Any, grad_output: Tensor) -> tuple[Tensor, Tensor, None]:
        # Retrieve saved tensors & values
        (z, theta) = ctx.saved_tensors
        eps = ctx.eps
        # Compute gradient of the loss with respect to z (no STE) and theta (using STE)
        grad_z = (z > theta).float() * grad_output
        grad_theta = -(theta / eps) * rectangle((z - theta) / eps) * grad_output
        grad_theta_agg = grad_theta.sum(dim=0)  # note, sum over batch dim isn't strictly necessary
        return grad_z, grad_theta_agg, None
```
</details>

### 연습 문제 - JumpReLU SAE 구현하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 40 minutes on this exercise.
> ```

이제 이 두 함수를 모두 구현했으므로, 전체 JumpReLU SAE를 조립하는 데 필요한 요소들을 충분히 갖추게 되었습니다. 이전 연습 문제에서 Gated SAE를 구축했던 것과 동일한 방식으로, 표준 SAE 아키텍처에서 다음과 같은 차이점을 가진 별도의 클래스를 생성하여 구축하는 것을 권장합니다:

- `(n_instances, d_sae)` 형태를 가지며 JumpReLU / Heaviside 함수에 사용되는 벡터 `theta`을 생성하는 파라미터 `log_theta`을 추가합니다.
    - 임계값(threshold values)은 항상 양수여야 하므로 `theta` 대신 `log_theta`를 사용합니다.
    - 초기화와 리샘플링 시 모두, 논문의 값인 `0.001`보다는 `theta = 0.1`을 사용하는 것을 권장합니다 (이는 `log` 함수의 작은 gradient 때문에 작은 값들이 증가하는 데 오랜 시간이 걸리기 때문입니다). `log_theta`를 설정할 때 이 값들을 log-space로 변환해야 합니다.
- SAE hidden 값은 이제 표준 ReLU 대신 JumpReLU activation 함수를 사용합니다. 즉, i번째 hidden 값은 $\operatorname{JumpReLU}_\theta(\pi_i(x))$이며, 여기서 $\pi_i(x) = (W_{enc}x + b_{enc})_i$은 pre-JumpReLU activation입니다.
    - DeepMind 부록에서는 ReLU 및 JumpReLU 함수에 $\pi_i(x)$ 대신 $\operatorname{ReLU}(\pi_i(x))$를 전달할 것을 제안합니다 (이는 $\theta_i$가 충분히 작아져서 $0 > \pi_i(x) > \theta_i - \epsilon/2$이 발생할 수 있는 특수한 상황에서 $\pi_i(x)$의 음수 값이 gradient에 영향을 주지 않도록 하기 위함입니다). 저희도 이 방법을 권장합니다.
- sparsity loss 항은 더 이상 L1 norm이 아니며, 대신 $\lambda \|\mathbf{f}(\mathbf{x})\|_0 = \sum_{i=1}^{d_{\text{sae}}} H(\pi_i(x) - \theta_i)$이 됩니다. 여기서 $\lambda$은 sparsity coefficient입니다.
    - sparsity coefficient의 값으로 `0.1`부터 시작하는 것을 권장하며, 이는 아래 예제 코드에 제공되어 있습니다.
    - L1 penalty 항에 대해 `d_sae`로 합산했던 것과 동일한 이유로, 이 L0 penalty 항 역시 평균을 내지 않고 `d_sae`에 대해 합산한다는 점에 유의하십시오.
- STE의 기본값으로 DeepMind 논문의 값인 `0.001` 대신 `ste_epsilon=0.01`를 권장합니다 (이는 `ToySAEConfig`에서 사용되는 기본값입니다).

In [ ]:
THETA_INIT = 0.1


class JumpReLUToySAE(ToySAE):
    W_enc: Float[Tensor, "inst d_in d_sae"]
    _W_dec: Float[Tensor, "inst d_sae d_in"] | None
    b_enc: Float[Tensor, "inst d_sae"]
    b_dec: Float[Tensor, "inst d_in"]
    log_theta: Float[Tensor, "inst d_sae"]

    # YOUR CODE HERE - write the methods of your new SAE, which should support all 3 modes


jumprelu_sae = JumpReLUToySAE(
    cfg=ToySAEConfig(n_inst=n_inst, d_in=d_in, d_sae=d_sae, tied_weights=True, sparsity_coeff=0.1),
    model=model,
)
jumprelu_data_log = jumprelu_sae.optimize(steps=20_000, resample_method="advanced")  # batch_size=4096?

# Animate the best instances, ranked according to average loss near the end of training
n_inst_to_plot = 4
n_batches_for_eval = 10
avg_loss = t.concat([d["loss"] for d in jumprelu_data_log[-n_batches_for_eval:]]).mean(0)
best_instances = avg_loss.topk(n_inst_to_plot, largest=False).indices.tolist()

utils.animate_features_in_2d(
    jumprelu_data_log,
    rows=["W_enc", "h", "h_r"],
    instances=best_instances,
    filename=str(section_dir / "animation-training-jumprelu.html"),
    color_resampled_latents=True,
    title="JumpReLU SAE on toy model",
)

In [ ]:
# Replicate figure 15 for jumprelu SAE (should get same results as for gated)
replicate_figure_15(
    [
        ("standard", resampling_sae, resampling_data_log),
        # ("gated", gated_sae, gated_data_log), # you can uncomment this to compare all 3!
        ("jumprelu", jumprelu_sae, jumprelu_data_log),
    ]
)

<details><summary>솔루션</summary>

```python
THETA_INIT = 0.1


class JumpReLUToySAE(ToySAE):
    W_enc: Float[Tensor, "inst d_in d_sae"]
    _W_dec: Float[Tensor, "inst d_sae d_in"] | None
    b_enc: Float[Tensor, "inst d_sae"]
    b_dec: Float[Tensor, "inst d_in"]
    log_theta: Float[Tensor, "inst d_sae"]
    def __init__(self, cfg: ToySAEConfig, model: ToyModel):
        super(ToySAE, self).__init__()

        assert cfg.d_in == model.cfg.d_hidden, "ToyModel's hidden dim doesn't match SAE input dim"
        self.cfg = cfg
        self.model = model.requires_grad_(False)
        self.model.W.data[1:] = self.model.W.data[0]
        self.model.b_final.data[1:] = self.model.b_final.data[0]

        self._W_dec = (
            None
            if self.cfg.tied_weights
            else nn.Parameter(nn.init.kaiming_uniform_(t.empty((cfg.n_inst, cfg.d_sae, cfg.d_in))))
        )
        self.b_dec = nn.Parameter(t.zeros(cfg.n_inst, cfg.d_in))

        self.W_enc = nn.Parameter(nn.init.kaiming_uniform_(t.empty((cfg.n_inst, cfg.d_in, cfg.d_sae))))
        self.b_enc = nn.Parameter(t.zeros(cfg.n_inst, cfg.d_sae))
        self.log_theta = nn.Parameter(t.full((cfg.n_inst, cfg.d_sae), t.log(t.tensor(THETA_INIT))))

        self.to(device)

    @property
    def theta(self) -> Float[Tensor, "inst d_sae"]:
        return self.log_theta.exp()

    def forward(
        self, h: Float[Tensor, "batch inst d_in"]
    ) -> tuple[
        dict[str, Float[Tensor, "batch inst"]],
        Float[Tensor, ""],
        Float[Tensor, "batch inst d_sae"],
        Float[Tensor, "batch inst d_in"],
    ]:
        """
        Same as previous forward function, but allows for gated case as well (in which case we have different
        functional form, as well as a new term "L_aux" in the loss dict).
        """
        h_cent = h - self.b_dec

        acts_pre = (
            einops.einsum(h_cent, self.W_enc, "batch inst d_in, inst d_in d_sae -> batch inst d_sae") + self.b_enc
        )
        # print(self.theta.mean(), self.theta.std(), self.theta.min(), self.theta.max())
        acts_relu = F.relu(acts_pre)
        acts_post = JumpReLU.apply(acts_relu, self.theta, self.cfg.ste_epsilon)

        h_reconstructed = (
            einops.einsum(acts_post, self.W_dec, "batch inst d_sae, inst d_sae d_in -> batch inst d_in") + self.b_dec
        )

        loss_dict = {
            "L_reconstruction": (h_reconstructed - h).pow(2).mean(-1),
            "L_sparsity": Heaviside.apply(acts_relu, self.theta, self.cfg.ste_epsilon).sum(-1),
        }

        loss = loss_dict["L_reconstruction"] + self.cfg.sparsity_coeff * loss_dict["L_sparsity"]

        return loss_dict, loss, acts_post, h_reconstructed

    @t.no_grad()
    def resample_simple(
        self,
        frac_active_in_window: Float[Tensor, "window inst d_sae"],
        resample_scale: float,
    ) -> None:
        dead_latents_mask = (frac_active_in_window < 1e-8).all(dim=0)  # [instances d_sae]
        n_dead = int(dead_latents_mask.int().sum().item())

        replacement_values = t.randn((n_dead, self.cfg.d_in), device=self.W_enc.device)
        replacement_values_normed = replacement_values / (
            replacement_values.norm(dim=-1, keepdim=True) + self.cfg.weight_normalize_eps
        )

        # New names for weights & biases to resample
        self.W_enc.data.transpose(-1, -2)[dead_latents_mask] = resample_scale * replacement_values_normed
        self.W_dec.data[dead_latents_mask] = replacement_values_normed
        self.b_enc.data[dead_latents_mask] = 0.0
        self.log_theta.data[dead_latents_mask] = t.log(t.tensor(THETA_INIT))

    @t.no_grad()
    def resample_advanced(
        self,
        frac_active_in_window: Float[Tensor, "window inst d_sae"],
        resample_scale: float,
        batch_size: int,
    ) -> None:
        h = self.generate_batch(batch_size)
        l2_loss = self.forward(h)[0]["L_reconstruction"]

        for instance in range(self.cfg.n_inst):
            is_dead = (frac_active_in_window[:, instance] < 1e-8).all(dim=0)
            dead_latents = t.nonzero(is_dead).squeeze(-1)
            n_dead = dead_latents.numel()
            if n_dead == 0:
                continue

            l2_loss_instance = l2_loss[:, instance]  # [batch_size]
            if l2_loss_instance.max() < 1e-6:
                continue

            distn = Categorical(probs=l2_loss_instance.pow(2) / l2_loss_instance.pow(2).sum())
            replacement_indices = distn.sample((n_dead,))  # type: ignore

            replacement_values = (h - self.b_dec)[replacement_indices, instance]  # [n_dead d_in]
            replacement_values_normalized = replacement_values / (
                replacement_values.norm(dim=-1, keepdim=True) + self.cfg.weight_normalize_eps
            )

            W_enc_norm_alive_mean = (
                self.W_enc[instance, :, ~is_dead].norm(dim=0).mean().item() if (~is_dead).any() else 1.0
            )

            # New names for weights & biases to resample
            self.b_enc.data[instance, dead_latents] = 0.0
            self.log_theta.data[instance, dead_latents] = t.log(t.tensor(THETA_INIT))
            self.W_dec.data[instance, dead_latents, :] = replacement_values_normalized
            self.W_enc.data[instance, :, dead_latents] = (
                replacement_values_normalized.T * W_enc_norm_alive_mean * resample_scale
            )


jumprelu_sae = JumpReLUToySAE(
    cfg=ToySAEConfig(n_inst=n_inst, d_in=d_in, d_sae=d_sae, tied_weights=True, sparsity_coeff=0.1),
    model=model,
)
jumprelu_data_log = jumprelu_sae.optimize(steps=20_000, resample_method="advanced")  # batch_size=4096?

# Animate the best instances, ranked according to average loss near the end of training
n_inst_to_plot = 4
n_batches_for_eval = 10
avg_loss = t.concat([d["loss"] for d in jumprelu_data_log[-n_batches_for_eval:]]).mean(0)
best_instances = avg_loss.topk(n_inst_to_plot, largest=False).indices.tolist()

utils.animate_features_in_2d(
    jumprelu_data_log,
    rows=["W_enc", "h", "h_r"],
    instances=best_instances,
    filename=str(section_dir / "animation-training-jumprelu.html"),
    color_resampled_latents=True,
    title="JumpReLU SAE on toy model",
)
```
</details>

# ☆ 보너스

## 추천 논문 구현 과제

### [Toy Models of Superposition](https://transformer-circuits.pub/2022/toy_model/index.html#phase-change)

이 논문에는 이번 실습에서 다루지 않은 몇 가지 측면이 있습니다. 특히, **상전이로서의 superposition**은 sparsity와 상대적 feature 중요도 사이의 상호작용을 연구하며, 이러한 입력값이 변함에 따라 최적의 weight 설정에서 상전이가 발생함을 발견합니다. 몇 가지 예시는 [this notebook](https://github.com/wattenberg/superposition/blob/main/Exploring_Exact_Toy_Models.ipynb) 에서 확인하실 수 있습니다.

다음과 같은 경우라면 이 내용을 재현해 보는 것이 좋을 것입니다:

* superposition의 구체적인 수학적 세부 사항을 깊게 파고드는 것을 즐기시는 분
* 경험적 연구뿐만 아니라 이론적 연구에도 흥미를 느끼시는 분
* 이 섹션의 처음 3개 정도의 실습 세트를 즐겁게 수행하신 분

<br>

### [Polysemanticity and Capacity in Neural Networks](https://arxiv.org/pdf/2210.01892.pdf)

이 논문은 Redwood Research에서 작성한 것으로, 이 논문의 처음 세 네 부분(superposition의 toy 모델 및 feature geometry에 관한 결과)에서 논의한 아이디어들을 기반으로 합니다.

저자들은 위에서 dimensionality라고 불렀던 것과 동일한 capacity라는 척도를 깊이 있게 연구합니다. 이들의 결과는 왜 feature들이 종종 0 또는 1의 capacity에 날카롭게 "고정(pinned)"되는지(즉, 전혀 표현되지 않거나, 다른 모든 feature와 orthogonal하게 표현되는지)에 대한 설명을 제시합니다.

다음과 같은 경우에 이 논문을 재현해 보는 것이 좋을 것입니다:

* superposition의 구체적인 수학적 세부 사항을 파고드는 것을 즐기시는 분
* 선형 대수학 및 미적분학과 같은 수학적 주제에 익숙하신 분
* 실증적 연구뿐만 아니라 이론적 연구에도 흥미를 느끼시는 분
* 이 섹션의 처음 약 4개 연습 문제 세트를 즐겁게 수행하신 분

<!-- ### [Finding Neurons in a Haystack: Case Studies with Sparse Probing](https://arxiv.org/abs/2305.01610)

The authors train a set of sparse linear probes on neuron activations to predict the presence of certain input features. They manage to find **sparse combinations of neurons which represent many features in superposition**, e.g. a neuron which activates on the bigram phrase "social security" but not either word individually (see image below).

Note that this paper is slightly less relevant now that dictionary learning with SAEs has superceded its methodology - but it still represents a large step forward for the goal of extracting features from superposition in MLPs.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/socialsecurity.png" width="750"> -->

## 추가 탐구를 위해 추천하는 주제들입니다

### [Softmax Linear Units](https://transformer-circuits.pub/2022/solu/index.html)

이것은 낮은 성능 비용으로 해석 가능한 MLP의 수를 증가시키는 것으로 보이는 제안된 아키텍처 변경 사항입니다. 특히, 이는 superposition의 사례를 줄일 수 있습니다.

요약: SOLU는 softmax가 sparsity를 유도하는 방식과 동일하게 activation의 sparsity를 유도하는 activation function $\vec{x} \to \vec{x} * \operatorname{softmax}(\vec{x})$ 입니다 (종종 softmax가 적용된 확률 분포는 하나의 확률은 1에 가깝고 나머지는 0에 가깝습니다). activation sparsity를 유도하는 것은 뉴런이 polysemantic해지는 것을 더 어렵게 만들 수 있습니다.

이 논문의 결과를 재현하는 것은 마지막 주 프로젝트로 실용적이지 않을 수 있습니다. 하지만 TransformerLens 라이브러리의 여러 transformer가 SOLU로 학습되었으며 (자세한 내용은 [model page](https://neelnanda-io.github.io/TransformerLens/generated/model_properties_table.html) 참조), 이는 이 모델들을 더 자세히 연구하기 위한 좋은 후보로 만들어 줍니다. 여러분이 탐구해 볼 수 있는 몇 가지 질문은 다음과 같습니다:

- Neel의 SoLU 모델과 GELU 모델은 SoLU 논문에서 사용된 polysemanticity 지표 하에서 [neuroscope](https://neuroscope.io/) 어떻게 비교됩니까? (상위 10개 activation 데이터셋 예시를 1분 동안 살펴보았을 때, 뉴런의 어느 정도 비율이 monosemantic해 보입니까?)
- SoLU의 polysemanticity 지표는 뉴런이 강하게 activation될 때 monosemantic한지에 대한 정보만 제공하므로 다소 제한적입니다 (그리고 이는 일반적인 monosemantic 여부와 상관관계가 없을 수 있습니다 - 논문의 [this caveat](https://transformer-circuits.pub/2022/solu/index.html#:~:text=be%20reverse%2Dengineered.-,CAVEAT,-Since%20publication%2C%20we%27ve) 참조). 더 나은 지표를 찾을 수 있습니까? 더 신뢰할 수 있거나 확장 가능한 방법이 있을까요?
- 논문 [speculates](https://transformer-circuits.pub/2022/solu/index.html#section-4-3) 에서는 SoLU activation 이후의 LayerNorm이 특징들을 많은 차원에 걸쳐 뭉개고 출력을 매우 작게 만든 뒤 LayerNorm이 이를 다시 키우게 함으로써, 모델이 superposition을 "몰래 통과(smuggle through)"하게 한다고 주장합니다. solu-1l에서 이에 대한 증거를 찾을 수 있습니까?

### [Towards Monosemanticity: Decomposing Language Models With Dictionary Learning](https://transformer-circuits.pub/2023/monosemantic-features/index.html)

automated interpretability, feature motifs, finite-state automata와 같이 Anthropic의 dictionary learning 논문에서 다룬 흥미로운 주제들이 많지만, 여기서는 시간 관계상 깊게 다루지 못했습니다.

또한 마지막에 [Future Work](https://transformer-circuits.pub/2023/monosemantic-features/index.html#discussion-future-work) 섹션이 있으며, 독자분들이 프로젝트 아이디어를 얻는 데 흥미롭게 읽으실 수 있을 것입니다!

### [Exciting Open Problems In Mech Interp v2](https://docs.google.com/document/d/1lIIzMjenXh-U0j5jkuqSDTawCoMNW4TqUlxk7mmbmRg/edit)

이 문서는 Neel이 작성하였으며, mech interp의 흥미로운 오픈 문제들을 모아놓은 것입니다 (특히 SAE 관련 문제들에 중점을 두었습니다). 다시 말씀드리지만, 이 중 많은 주제들이 훌륭한 캡스톤 프로젝트가 될 수 있습니다! 다만, 이 리스트에서 더 달성 가능하고 덜 야심찬 프로젝트를 선택하시기를 권장합니다.

관심 있는 프로젝트 중 sparse autoencoder 학습이 포함되어 있다면, Arthur Conmy의 [this post](https://www.lesswrong.com/posts/fifPCos6ddsmJYahD/my-best-guess-at-the-important-tricks-for-training-1l-saes)을 *강력히* 추천합니다. 이 자료는 SAE를 잘 학습시키기 위한 다양한 기법들을 모아놓은 것입니다 (이 중 대부분은 본 실습에서 다루지 않았습니다).